# Elementary school progress and attendance

This public notebook preserves the analysis code and clears cell outputs. The [aggregate results](../README.md#key-findings) and [briefing](../briefing/KaWaiSit_Level_Director_Briefing.pptx) can be reviewed without the supplied input files. See [data availability](../data/README.md) for local reproduction.

## Project overview

This notebook records the work behind an overview for the new Level Director of four elementary schools: Cedar Ridge, Harper Creek, Lincoln Park and Meadowbrook. It uses the nine supplied SIS tables for 2023-24 to 2025-26. The data are synthetic.

The task is to select what the LD needs to know before meeting principals. I use the latest year to describe school context and learning needs, and earlier years to check patterns and support adjusted comparisons. The analysis includes both progress and concerns; it does not evaluate the effect of instruction or a particular service.

The main finding is that many students gained score points while still testing below Benchmark in Spring. Their starting level and overlapping Reading, Mathematics and attendance needs provide specific questions for principals. Adjusted school differences are small and uncertain. The adjusted absence association is statistically significant but small in score units.

The ten-step map and question table provide the project framework. Expand the QC, exploratory data analysis (EDA), model and follow-up sections to see the checks, results and reasoning. Detailed code follows this opening summary.

The QC, EDA, models, diagnostic review, dashboard, and 10-slide briefing are complete and have been reviewed for submission.


## The work in ten steps

| Step | What I did | Why I did it | Detailed section |
|---|---|---|---|
| 1 | Read the task and dictionary; define the four-school, K-5 scope | Keep the analysis useful for the LD's short introductory briefing | 1, 3.4A |
| 2 | Check all nine tables, including keys, dates, expected blanks and dictionary rules | Establish which records can be connected and counted | 2.1-2.11 |
| 3 | Build student-year context, subject-specific test pairs and participation tables | Give each measure an explicit population, denominator and attendance period | 3.4A-D |
| 4 | Explore school context, learning and attendance; review other possible priorities | Find patterns worth explaining and refine the three guiding questions | 3.1-3.9 |
| 5 | Fit one predictor at a time to Spring score | Describe raw associations before considering factors together | 4.3A |
| 6 | Fit separate Reading and Mathematics OLS models | Examine adjusted school differences and the additional absence association | 4.4-4.5 |
| 7 | Review assumptions, influential records and two sensitivity checks | Check whether the fitted relationships and conclusions need revision | 4.6-4.10 |
| 8 | Follow the same students across Benchmark levels and count overlapping needs | Clarify which students improved, remained below, or had needs in both subjects | 3.10A-C |
| 9 | Fit an exploratory logistic model for the Fall starting groups | Examine the starting-severity difference after school, grade and year adjustment | 4.11 |
| 10 | Select findings, state limits and prepare principal follow-up questions | Turn the analysis into a 10-slide briefing and a supporting dashboard | 5-7 |

The notebook groups descriptive work in Section 3 and models in Section 4. The focused follow-up in 3.10 developed after reviewing the primary OLS results; its position keeps related descriptive tables together. The broad questions guided the EDA and were refined as the findings emerged.


<details>
<summary>QC details: checks, decisions and analysis samples</summary>

## What the quality checks established

A duplicate key can multiply rows after a join. An incorrect date or denominator can change an absence rate or test comparison. I checked these relationships before interpreting differences between schools.

| Table | Checks completed | Why the check matters; where I use it |
|---|---|---|
| Schools | Unique IDs and names, required fields, school levels and grade limits | Identify the four K-5 schools and attach school names to later tables (2.1; 3.4B) |
| Students | Unique student IDs, date and text types, expected missing values, race/ethnicity consistency, SPED/504 rules and language/EL patterns | Connect student context without duplicating students or treating fixed labels as eligibility histories (2.2; 3.5; 4.8) |
| Enrollments | Unique student-year keys; student, school and homeroom references; entry/exit dates; grades and adjacent-year progression | Define the annual population and time enrolled, and provide the reference row for events (2.3; 3.4A-C) |
| Daily attendance | Unique student-date records; enrollment and calendar matches; status rules; expected enrolled-day coverage | Count absent and eligible days consistently for annual rates and intervals between tests (2.4; 3.4B-C; 3.7; 4.5) |
| Assessments | Unique result and testing-window keys; enrollment, grade and date matches; family-specific fields and documented score rules | Separate test types, pair Fall/Spring results and review participation before comparing learning (2.5; 3.4C-D; 3.6; 4) |
| Discipline | Unique incidents, reference and event-date matches, reporter assignments, action/removal-day consistency | Keep incident counts separate from students involved and use stated denominators in the supporting review (2.6; 3.8) |
| Teachers | Unique IDs/emails, employment dates, missing exit dates, FTE/experience ranges and school references | Distinguish current teacher attributes from historical staffing (2.7; 3.8) |
| Teacher assignments | Unique teacher-year keys, ID format, employment overlap, FTE and role/grade rules; employment periods without assignments | Describe recorded yearly staffing and disclose coverage gaps (2.8; 3.8) |
| School calendar | Unique dates, school-year/weekday labels, weekday coverage and instructional-day sequence | Exclude noninstructional days and calculate the correct days enrolled (2.9; 3.4B-C) |

Most checked relationships passed. The checks also established decisions that carry into the analysis:

- Fourteen employed teacher-year periods have no assignment record. I leave school and FTE unfilled and treat staffing as a question to verify.
- Two blank guardian names and several test-specific blanks are expected. They do not justify dropping students or replacing missing values with zero.
- Student program labels are fixed across the extract. Their yearly proportions describe changing student mixes, not individual eligibility changes. The perfect language/EL correspondence is a feature of this synthetic file.
- Calendars contain 188, 186 and 185 instructional days. Partial-year enrollments need their actual days, not one fixed annual denominator.

The checks support the tested relationships within the supplied files. They do not verify the extract against the original SIS or establish school performance.

### How the checked tables become analysis samples

I summarize daily attendance before joining it to one student-year context row. I then create one student-year-subject row for Fall/Spring tests and attendance strictly between the two test dates. This prevents event rows from multiplying the analysis population.

The context table contains 4,141 student-year records across three years. The test table retains 8,282 rows, including missing tests. Reading has 3,653 complete pairs from 1,780 students; Mathematics has 3,606 from 1,779. Students can recur across years and subjects, so those counts cannot be added as different people.

School profiles retain everyone enrolled. Spring status uses Spring-tested students; score change and models use complete pairs. Official testing windows are unavailable, so I estimate opportunity from recorded test dates and enrollment overlap. Paired records have longer enrollment on average than excluded records. Their results therefore apply most directly to students with both tests (3.4D).

</details>


<details>
<summary>EDA details: patterns that raised the next questions</summary>

## What the EDA made worth examining

The first six EDA tasks describe the data without adjusting for several factors at once. QC provides the definitions and connections; the patterns below come from the EDA.

| EDA task | What I examined | Finding and next question |
|---|---|---|
| 1. School context | Enrollment changes, grade mix, recorded student needs and entry/exit patterns | Cedar Ridge and Lincoln Park grew about 9%-10%; the other two schools declined. Background and mobility differ. How should this context enter a learning comparison? (3.5) |
| 2. Attendance overview | Annual school rates and student-level distributions | All four school rates fell and are close in the latest year, about 5.2%-5.7%. Some students have much higher absence. What do school averages conceal? (3.5) |
| 3. Fall starting points | Tested samples and scores within grade and subject | Some school differences already exist at Fall; no school consistently leads across all grades and both subjects. How much of a Spring difference reflects the starting point? (3.6) |
| 4. Spring status and paired change | All-Spring-tested status, same-student score changes and school-grade patterns | Median changes are positive in every latest-year school-grade-subject group, while many Spring results remain below Benchmark. Who improved and still remained below? (3.6) |
| 5. Attendance concentration | School-grade heatmaps, student groups and enrollment length | The highest-absence grade differs across schools. Which current grades and students should principals review? (3.7) |
| 6. Attendance alongside learning | Between-test absence with Fall score, Spring score and change, within grades | Absence accompanies lower Fall and Spring scores, but relates weakly to change. Fall comes before this attendance interval. What remains after allowing for starting achievement? (3.7) |

I also reviewed state tests, discipline and staffing (3.8). Recurring grade 5 literacy concerns provide a supporting point. Discipline counts and incomplete staffing coverage limit stronger conclusions. Benchmarks and attendance cover all K-5 grades and provide a broader basis for the short briefing.

</details>


## Three linked questions for the LD

These questions organize the EDA and models. The smaller questions connect school context to learning comparisons and then to practical follow-up.

| Main question | Smaller questions | How I answer it |
|---|---|---|
| Q1. Who do the four schools serve? | How have enrollment and grade mix changed? How do recorded needs, mobility and absence differ? What should be verified for the incoming year? | All-enrollment profiles and annual attendance; current-roster questions for principals (3.5; 6) |
| Q2. What do Fall starting points and Spring results show? | Who has both tests? Do school differences remain after adjustment? Who gains points but stays below? Does the lowest starting group have a higher probability of remaining below? | Participation review, paired EDA, Model 1, student paths and the exploratory starting-group model (3.4D; 3.6; 3.10; 4.4; 4.11) |
| Q3. Where is absence concentrated, and what does it add to the learning picture? | Which schools/grades have higher absence? Is between-test absence still associated with Spring score after adjustment? Which academic needs overlap with annual absence? | Heatmaps, distributions, Model 2 and the joint-needs table (3.7; 3.10B; 4.5) |

The LD needs an overview of progress and a few issues to investigate with principals. These questions describe whom schools serve, how learning changes, and where support may need coordination. Historical records cannot identify the coming fall roster or show which intervention would work.


<details>
<summary>Primary model details: results, diagnostics and sensitivity checks</summary>

## What the primary models added

I first fit five separate predictors to Spring score in each subject: Fall score, school, grade, year and between-test absence (4.3A). These preliminary results describe raw associations. The code uses student-clustered Wald F tests and intervals. It does not use a p < .2 rule to remove variables.

The primary ordinary least squares (OLS) regressions model Spring score while considering factors together. Reading and Mathematics remain separate, with the same complete-pair sample for raw and adjusted comparisons within each subject. Uncertainty estimates account for a student appearing in several years.

| Model and question | Adjustment | Result and interpretation |
|---|---|---|
| Model 1: Do Spring school differences remain? | Fall score, school, grade and year | School contrasts against Cedar Ridge are small; all pointwise 95% intervals include zero. Overall school p values are .200 in Reading and .748 in Mathematics. This is uncertain evidence about adjusted differences, not proof of equivalence or a school ranking (4.4). |
| Model 2: What does absence add? | Model 1 plus between-test absence | Per 5 percentage points more absence, Spring score is about 0.20 points lower in Reading (95% CI -0.36 to -0.04; p = .016) and 0.32 lower in Mathematics (-0.48 to -0.15; p < .001). Raw differences were about -2.72 and -2.74. The adjusted associations are statistically significant but small (4.5). |

School, grade, year and Fall score remain because they answer the comparison question or define relevant context. EL, economic-disadvantage and SPED labels enter a separate sensitivity check; they are not in the primary formulas.

I reviewed residual shape and spread, patterns within grades, student clusters, design rank and influential records (4.6). The plots and targeted checks of grade-specific Fall slopes and quadratic terms did not identify a problem requiring a different primary formula (4.7A). Student clustering was already part of the fits; it allows repeated records within a student and unequal residual variance for the reported uncertainty. Dependence between different students remains a limitation. Two sensitivity checks omit five reviewed students or add the three fixed background labels (4.8). The school comparisons remain uncertain and the absence association remains negative and small. These checks support the same interpretation across the alternatives examined. They do not show that every assumption is satisfied.

These results narrowed the claims I could make. They did not resolve which students had continuing learning needs despite progress. That question led to the focused follow-up below; the original OLS models and their results remain in the notebook.

</details>


<details>
<summary>Focused follow-up: student paths, overlap and the starting-group model</summary>

## Why I added the student-path analysis

School averages and adjusted score differences do not identify who moved across Benchmark levels. The EDA already showed positive score change alongside substantial below-Benchmark Spring status. I followed the same students to separate progress from continuing status, then examined overlapping needs (3.10).

| Latest-year paired Fall-below students, 2025-26 | Reading | Mathematics |
|---|---:|---:|
| Below at both Fall and Spring | 443 / 539 (82.2%) | 429 / 516 (83.1%) |
| Gained score points among those below in both windows | 401 / 443 (90.5%) | 399 / 429 (93.0%) |
| Median change among those below in both windows | +10 points | +11 points |

The first two percentages have different denominators. "Below in both windows" describes two observed endpoints; it does not mean no progress or below throughout the year. The dictionary provides no expected-growth target.

The one-row-per-student latest-year table identifies 618 students with this pattern in at least one subject, including 254 in both. It also retains 281 students with incomplete two-subject paths. Of those 281, 76 have a known pattern in their observed subject and already enter the 618 count. Unknown information is not assigned to "no learning needs."

Among the 618 identified students, 475 have annual absence below 10%; 143 have at least 10%. An academic-support review limited to the high-absence band would miss many identified students. The 143 with overlapping patterns provide a specific coordination question. This uses annual attendance, not the between-test interval in Model 2.

### The follow-up model: does starting severity distinguish continuing status?

The descriptive paths showed a larger difference between Fall Well Below and Fall Below students. I fit an exploratory logistic regression among Fall-below students with both tests across all three years (4.11). It models whether Spring is still below, allows for school, grade and year, and compares probabilities over a common historical mix.

| Subject | Fall Below: adjusted probability | Fall Well Below: adjusted probability | Difference, percentage points (95% CI) |
|---|---:|---:|---:|
| Reading | 70.8% | 97.0% | +26.2 (22.9 to 29.5) |
| Mathematics | 70.5% | 96.7% | +26.2 (22.8 to 29.6) |

Both focal starting-group tests have Holm-adjusted p < .001. Reading uses 1,690 student-year records from 967 students; Mathematics uses 1,637 from 938. Both groups occur in all 72 school-grade-year combinations in each subject. Convergence, separation, rank, influence and calibration within the fitted data were reviewed. The two sensitivity checks retain a substantial probability gap.

This follow-up uses the same data that raised the question. It is exploratory, not independent confirmation. The result supports closer discussion of the lowest starting group's progress and support. It does not establish the largest cause of achievement, forecast next-year outcomes or estimate a service's effect.

</details>


## What I would take to the new LD

The briefing recognizes progress: annual absence fell across all four schools and median paired scores rose in every latest-year school-grade-subject group. Continuing below-Benchmark status remains common, especially for the lowest Fall starting group. The main follow-up is to review learning support by subject and starting severity, and coordinate attendance help where needs overlap.

I would ask principals to:

1. Confirm current enrollment and which historical students have returned or advanced. Treat historical K-4 groups as counts to verify and handle former grade 5 students through middle-school transition discussions. New Kindergarten entrants are not in that historical planning group.
2. Review Reading and Mathematics support and progress monitoring, particularly for students with the lowest starting levels and needs in both subjects.
3. Identify attendance barriers and coordinate support for overlapping needs, while keeping academic follow-up available to students below the 10% absence cutoff.

These topics cover all elementary grades, use checked populations and connect to questions principals can answer. Grade 5 literacy supports the academic discussion. Staffing and discipline remain qualified context or verification questions rather than the main briefing claims. Selection follows usefulness, coverage, effect size and uncertainty; statistical significance alone does not determine slide priority.

The 10-slide briefing is in [PowerPoint briefing](../briefing/KaWaiSit_Level_Director_Briefing.pptx). Section 7 exports `Fairmont_LD_Dashboard.html`, with five views covering school context, progress, overlapping needs, attendance and completed models. The dashboard's model results have a fixed three-year scope; descriptive filters do not refit them.

Run the code from top to bottom. The CSV sources remain unchanged. A zero QC count means that check passed, not that the original SIS was independently verified. AI use and analytical responsibility are documented in 6.4. See the [project README](../README.md) and [methodology](../docs/methodology.md).


## Notebook setup

Run this setup cell first. It imports the libraries, locates the data folder and makes the local dashboard exporter available. Then run **Custom functions** before loading the nine tables.

The nine CSV files are the working data. I use the data dictionary to check what each row represents and which fields can be blank.

QC covers all seven schools so I can check their references and joins. The LD analysis focuses on the four elementary schools.

Terms used below:

- **EL:** English learner. **SPED:** special education. These labels describe the student records supplied.
- **Student-year record:** one student's enrollment in one school year. The same student may have records in several years.
- **Benchmark:** the Interim Benchmark assessment. "At/Above Benchmark" uses its performance labels. State-test proficiency is a separate measure.
- **Between-test attendance:** attendance after a student's Fall test and before their Spring test, excluding both test days.
- **Testing opportunity:** my estimate of whether enrollment overlapped the recorded testing dates. Official testing windows were not supplied.


In [ ]:
# Standard library
from pathlib import Path
import sys
import warnings

# Data, statistics and plotting
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from matplotlib.ticker import PercentFormatter
from patsy import build_design_matrices
from scipy import stats
from scipy.special import expit, logit
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import OLSInfluence
from statsmodels.tools.sm_exceptions import PerfectSeparationWarning

# Support kernels started from the repository root or the notebooks directory.
working_dir = Path.cwd().resolve()
project_dir = next(
    (folder for folder in (working_dir, *working_dir.parents)
     if (folder / "dashboard" / "build_dashboard.py").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")

DATA_DIR = project_dir / "data" / "source"
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from dashboard.build_dashboard import build_dashboard

pd.set_option("display.max_colwidth", 100)


## Custom functions

All 15 reusable functions are defined here. Run this section once after **Notebook setup**; the QC, EDA and model sections below call these functions with their own data and settings.

The imports are in **Notebook setup** above. Running the function definitions does not load the SIS tables, fit models or create figures. The small plotting helper inside `plot_residual_checks` stays inside that function because it is used only there.

| Group | Used in |
|---|---|
| Data quality | Table overviews in Section 2 |
| Attendance summaries | Student-group comparisons in Section 3.7 |
| Learning paths and overlapping needs | Section 3.10 |
| Regression samples and preliminary fits | Sections 4.3-4.3A |
| Primary OLS fits and result tables | Sections 4.4-4.5 and sensitivity checks in 4.8 |
| OLS diagnostics | Sections 4.6-4.8 |
| Exploratory logistic fits and adjusted probabilities | Section 4.11 |


### Data quality

`dataset_overview_check`

In [ ]:
def dataset_overview_check(df, max_levels=50):
    """Display column types, missingness, and compact categorical frequencies."""
    summary_rows = []
    n_rows = len(df)

    for col in df.columns:
        n_unique = df[col].nunique(dropna=True)
        missing_count = df[col].isna().sum()
        if n_unique <= max_levels:
            counts = df[col].value_counts(dropna=True)
            levels = ", ".join(
                f"{level}: {count:,} ({count / n_rows:.1%})"
                for level, count in counts.items()
            )
        else:
            levels = "More than 50 levels"
        summary_rows.append(
            {
                "Column": col,
                "Data type": str(df[col].dtype),
                "Unique nonmissing": n_unique,
                "Levels: count (%)": levels,
                "Missing count": missing_count,
                "Missing %": round(missing_count / n_rows * 100, 2),
            }
        )

    with pd.option_context("display.max_colwidth", 100):
        display(pd.DataFrame(summary_rows))


### Attendance summaries

`summarize_attendance_by_group`

In [ ]:
# Compare selected student groups


def summarize_attendance_by_group(data, group_var):

    summary = (
        data.groupby(["school_year", "school_year_label", "school_name", group_var])
        .agg(
            students=("student_id", "size"),
            total_absent_days=("absent_days", "sum"),
            total_enrolled_days=("enrolled_instructional_days", "sum"),
            median_absence_rate=("absence_rate", "median"),
            p75_absence_rate=("absence_rate", lambda x: x.quantile(0.75)),
        )
        .reset_index()
    )

    summary["aggregate_absence_rate"] = (
        summary["total_absent_days"] / summary["total_enrolled_days"]
    )

    return summary


### Student learning paths and overlapping needs

`make_learning_paths` · `summarize_learning_paths` · `make_joint_needs`

In [ ]:
def make_learning_paths(pairs):
    """Classify observed Fall-Spring status; keep missing tests visible."""
    keys = ["student_id", "school_year", "subject"]
    if pairs[keys].isna().any().any() or pairs.duplicated(keys).any():
        raise ValueError(
            "Learning paths require unique, nonmissing student-year-subject keys."
        )
    levels = [
        "Well Below Benchmark",
        "Below Benchmark",
        "At Benchmark",
        "Above Benchmark",
    ]
    for column in ["fall_performance_level", "spring_performance_level"]:
        if not pairs[column].dropna().isin(levels).all():
            raise ValueError(f"Unrecognized benchmark level in {column}.")
    out = pairs.copy()
    out["pair_complete"] = out[["fall_score", "spring_score"]].notna().all(axis=1)
    out["levels_complete"] = out.pair_complete & out[
        ["fall_performance_level", "spring_performance_level"]
    ].notna().all(axis=1)
    fall_below = out.fall_performance_level.isin(levels[:2])
    spring_below = out.spring_performance_level.isin(levels[:2])
    valid = out.levels_complete
    out["learning_path"] = "Incomplete tests"
    out.loc[out.pair_complete & ~valid, "learning_path"] = (
        "Paired scores; missing levels"
    )
    for mask, label in [
        (fall_below & spring_below, "Below in both windows"),
        (fall_below & ~spring_below, "Moved to At/Above"),
        (~fall_below & spring_below, "Moved below"),
        (~fall_below & ~spring_below, "At/Above in both windows"),
    ]:
        out.loc[valid & mask, "learning_path"] = label
    out["persistent_below"] = (fall_below & spring_below).astype("boolean").where(valid)
    order = dict(zip(levels, range(4)))
    out["level_improved"] = (
        (
            out.spring_performance_level.map(order)
            > out.fall_performance_level.map(order)
        )
        .astype("boolean")
        .where(valid)
    )
    out["score_change"] = (out.spring_score - out.fall_score).where(out.pair_complete)
    return out


def summarize_learning_paths(paths, group_columns):
    """Keep enrolled, paired, and Fall-below denominators separate."""
    rows = []
    for key, group in paths.groupby(group_columns, observed=True, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_columns, key))
        counts = group.learning_path.value_counts()
        persistent = int(counts.get("Below in both windows", 0))
        moved_up = int(counts.get("Moved to At/Above", 0))
        paired = int(group.levels_complete.sum())
        fall_below = persistent + moved_up
        row.update(
            {
                "enrolled_subject_rows": len(group),
                "classified_pairs": paired,
                "incomplete_tests": int(counts.get("Incomplete tests", 0)),
                "missing_levels": int(counts.get("Paired scores; missing levels", 0)),
                "fall_below": fall_below,
                "persistent_below": persistent,
                "moved_to_at_above": moved_up,
                "moved_below": int(counts.get("Moved below", 0)),
                "at_above_both": int(counts.get("At/Above in both windows", 0)),
                "persistent_pct_of_fall_below": (
                    100 * persistent / fall_below if fall_below else np.nan
                ),
                "persistent_pct_of_pairs": (
                    100 * persistent / paired if paired else np.nan
                ),
                "pair_coverage_pct": 100 * paired / len(group),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


def make_joint_needs(paths, context, year):
    """Join subjects once per student; incomplete information stays unknown."""
    keys = ["student_id", "school_year"]
    out = context.loc[context.school_year.eq(year)].copy()
    source = paths.loc[paths.school_year.eq(year)].copy()
    if out[keys].isna().any().any() or out.duplicated(keys).any():
        raise ValueError("The context must have one complete student-year key per row.")
    for subject, short in [("Reading", "reading"), ("Mathematics", "math")]:
        part = source.loc[source.subject.eq(subject)]
        if part.duplicated(keys).any():
            raise ValueError("Duplicate subject rows cannot be joined to context.")
        check = part[keys + ["school_id", "grade_level"]].merge(
            out[keys + ["school_id", "grade_level"]],
            on=keys,
            how="left",
            validate="one_to_one",
            suffixes=("_test", "_context"),
            indicator=True,
        )
        if (
            check["_merge"].ne("both").any()
            or check.school_id_test.ne(check.school_id_context).any()
            or check.grade_level_test.ne(check.grade_level_context).any()
        ):
            raise ValueError(
                "Subject school or grade does not match the enrollment context."
            )
        fields = part[keys + ["levels_complete", "persistent_below"]].rename(
            columns={
                "levels_complete": f"{short}_paired",
                "persistent_below": f"persistent_{short}",
            }
        )
        out = out.merge(fields, on=keys, how="left", validate="one_to_one")
        out[f"{short}_paired"] = out[f"{short}_paired"].astype("boolean").fillna(False)
        out[f"persistent_{short}"] = out[f"persistent_{short}"].astype("boolean")
    reading = out.persistent_reading.fillna(False)
    math = out.persistent_math.fillna(False)
    both_known = out.reading_paired & out.math_paired
    out["two_subject_pairs"] = both_known
    out["academic_need"] = "Incomplete paired information"
    for mask, label in [
        (reading & math, "Persistent below in both subjects"),
        (reading & ~math, "Persistent below in Reading only"),
        (~reading & math, "Persistent below in Mathematics only"),
        (~reading & ~math, "No persistent-below pattern in either subject"),
    ]:
        out.loc[both_known & mask, "academic_need"] = label
    out["any_persistent_identified"] = reading | math
    if ((out.absence_rate.dropna() < 0) | (out.absence_rate.dropna() > 1)).any():
        raise ValueError("Annual absence rates must be between zero and one.")
    out["attendance_band"] = "Attendance unavailable"
    out.loc[out.absence_rate.lt(0.10), "attendance_band"] = "Below 10% absence"
    out.loc[out.absence_rate.ge(0.10), "attendance_band"] = "10% or more absence"
    out["overlapping_identified_needs"] = (
        out.any_persistent_identified & out.absence_rate.ge(0.10)
    )
    # Planning labels only: neither return to this school nor promotion is observed.
    out["potential_next_grade"] = (
        (out.grade_level + 1).where(out.grade_level.le(4)).astype("Int64")
    )
    return out


### Regression samples and preliminary fits

`prepare_regression_samples` · `fit_univariable_models`

In [ ]:
def prepare_regression_samples(pairs):
    """Use one common paired sample per subject for preliminary and primary fits."""
    required = [
        "student_id",
        "school_year",
        "school_id",
        "grade_level",
        "fall_score",
        "spring_score",
        "between_test_absence_rate",
        "between_test_eligible_days",
    ]
    missing = [
        name
        for name in required + ["subject", "has_fall_spring_pair"]
        if name not in pairs
    ]
    if missing:
        raise ValueError(f"Missing model fields: {missing}")
    if not pairs.index.is_unique:
        raise ValueError("Use a unique index before preparing or fitting the models.")
    if pairs.duplicated(["student_id", "school_year", "subject"]).any():
        raise ValueError("Found duplicate student-year-subject keys.")
    samples, audit = {}, []
    for subject in ["Reading", "Mathematics"]:
        subject_rows = pairs.loc[pairs.subject.eq(subject)]
        paired = subject_rows.loc[subject_rows.has_fall_spring_pair.eq(True)]
        incomplete = paired[required].isna().any(axis=1)
        sample = paired.loc[~incomplete].copy()
        if sample.empty:
            raise ValueError(f"{subject}: no complete model rows remain.")
        numeric = sample[required].to_numpy(dtype=float)
        if not np.isfinite(numeric).all():
            raise ValueError(f"{subject}: non-finite score, ID, or absence value.")
        if not sample.between_test_absence_rate.between(0, 1).all():
            raise ValueError(f"{subject}: absence must be a rate from 0 to 1.")
        if not sample.between_test_eligible_days.gt(0).all():
            raise ValueError(
                f"{subject}: attendance intervals must contain enrolled instructional days."
            )
        for name in ["school_id", "school_year", "grade_level"]:
            if not sample[name].eq(np.floor(sample[name])).all():
                raise ValueError(f"{subject}: {name} must contain whole numbers.")
            sample[name] = sample[name].astype("int64")
        sample["fall_score"] = sample.fall_score.astype("float64")
        sample["spring_score"] = sample.spring_score.astype("float64")
        sample["absence_5pp"] = (
            sample.between_test_absence_rate.astype("float64") / 0.05
        )
        samples[subject] = sample
        audit.append(
            {
                "subject": subject,
                "enrolled_records": len(subject_rows),
                "paired_rows": len(paired),
                "model_rows": len(sample),
                "unique_students": sample.student_id.nunique(),
                "extra_exclusions": int(incomplete.sum()),
            }
        )
    return samples, pd.DataFrame(audit)


def fit_univariable_models(data, subject):
    """Fit five prespecified single-predictor models without selecting variables."""
    specifications = {
        "Fall score": "fall_score",
        "School": "C(school_id, Treatment(reference=101))",
        "Grade": "C(grade_level, Treatment(reference=0))",
        "School year": "C(school_year, Treatment(reference=2024))",
        "Between-test absence": "absence_5pp",
    }
    fields = [
        "student_id",
        "spring_score",
        "fall_score",
        "school_id",
        "grade_level",
        "school_year",
        "absence_5pp",
    ]
    if data[fields].isna().any().any() or not data.index.is_unique:
        raise ValueError("Supply a complete common sample with a unique index.")
    if data.student_id.nunique() < 2:
        raise ValueError("At least two student clusters are needed.")
    fits, overall, coefficients = {}, [], []
    for variable, term in specifications.items():
        model = smf.ols(f"spring_score ~ {term}", data=data, missing="raise")
        if (
            np.linalg.matrix_rank(model.exog) != model.exog.shape[1]
            or model.exog.shape[1] < 2
        ):
            raise ValueError(
                f"{variable}: insufficient variation or a redundant design."
            )
        result = model.fit(
            cov_type="cluster",
            cov_kwds={
                "groups": data.student_id.to_numpy(),
                "use_correction": True,
                "df_correction": True,
            },
            use_t=True,
        )
        fits[variable] = result
        test = result.wald_test_terms(skip_single=False, scalar=True).table.loc[term]
        overall.append(
            {
                "subject": subject,
                "variable": variable,
                "model_rows": len(data),
                "unique_students": data.student_id.nunique(),
                "wald_F": np.asarray(test["statistic"]).item(),
                "df_numerator": np.asarray(test["df_constraint"]).item(),
                "df_denominator": np.asarray(test["df_denom"]).item(),
                "p_value": np.asarray(test["pvalue"]).item(),
            }
        )
        interval = result.conf_int(alpha=0.05)
        for name in result.params.index:
            if name == "Intercept":
                continue
            coefficients.append(
                {
                    "subject": subject,
                    "variable": variable,
                    "term": name,
                    "estimate": result.params[name],
                    "clustered_se": result.bse[name],
                    "ci_low": interval.loc[name, 0],
                    "ci_high": interval.loc[name, 1],
                    "p_value": result.pvalues[name],
                }
            )
    return fits, pd.DataFrame(overall), pd.DataFrame(coefficients)


### Primary OLS fits and result tables

`fit_primary_model` · `primary_result_tables`

In [ ]:
def fit_primary_model(data, formula):
    """Fit on the full supplied common sample, with student-clustered uncertainty."""
    if not data.index.is_unique or data.student_id.isna().any():
        raise ValueError(
            "Supply a complete sample with nonmissing student IDs and a unique index."
        )
    if data.student_id.nunique() < 2:
        raise ValueError("At least two student clusters are needed.")
    try:
        model = smf.ols(formula, data=data, missing="raise")
    except Exception as error:
        raise ValueError(
            "The formula requires complete valid fields on the common sample."
        ) from error
    if model.data.row_labels.tolist() != data.index.tolist():
        raise ValueError("Fitting dropped rows from the common sample.")
    if not np.isfinite(model.exog).all() or not np.isfinite(model.endog).all():
        raise ValueError("The outcome and model predictors must be finite.")
    if np.linalg.matrix_rank(model.exog) != model.exog.shape[1]:
        raise ValueError(
            "The design is rank deficient. Review the formula before fitting."
        )
    return model.fit(
        cov_type="cluster",
        cov_kwds={
            "groups": data.student_id.to_numpy(),
            "use_correction": True,
            "df_correction": True,
        },
        use_t=True,
    )


def primary_result_tables(result, subject, model_name):
    interval = result.conf_int(alpha=0.05)
    coefficients = pd.DataFrame(
        {
            "term": result.params.index,
            "estimate": result.params.to_numpy(),
            "clustered_se": result.bse.to_numpy(),
            "ci_low": interval.iloc[:, 0].to_numpy(),
            "ci_high": interval.iloc[:, 1].to_numpy(),
            "p_value": result.pvalues.to_numpy(),
        }
    )
    coefficients = coefficients.loc[coefficients.term.ne("Intercept")].copy()
    coefficients.insert(0, "model", model_name)
    coefficients.insert(0, "subject", subject)
    raw_tests = result.wald_test_terms(skip_single=False, scalar=True).table
    terms = []
    for term, test in raw_tests.iterrows():
        if term == "Intercept":
            continue
        terms.append(
            {
                "subject": subject,
                "model": model_name,
                "term": term,
                "wald_F": np.asarray(test["statistic"]).item(),
                "df_numerator": np.asarray(test["df_constraint"]).item(),
                "df_denominator": np.asarray(test["df_denom"]).item(),
                "p_value": np.asarray(test["pvalue"]).item(),
            }
        )
    return coefficients.reset_index(drop=True), pd.DataFrame(terms)


### OLS diagnostics

`check_ols_model` · `plot_residual_checks` · `plot_influence_checks`

In [ ]:
def check_ols_model(result, data, label='OLS model'):
    """Review an existing statsmodels formula OLS fit; never fit or change it."""
    if not isinstance(result.model, OLS):
        raise TypeError('These checks support formula OLS results, including clustered OLS. GEE and WLS need their own diagnostics.')
    if not data.index.is_unique:
        raise ValueError('The supplied data need a unique index. Set it before fitting, then keep it unchanged.')

    row_labels = result.model.data.row_labels
    design_info = getattr(result.model.data, 'design_info', None)
    if row_labels is None or design_info is None:
        raise ValueError('Use a statsmodels formula OLS fit so the fitted rows and predictors can be verified.')
    if not pd.Index(row_labels).is_unique or not pd.Index(row_labels).isin(data.index).all():
        raise ValueError('Fitted row labels do not match a unique set of rows in the supplied data.')
    rows = data.loc[row_labels].copy()
    required = ['student_id', 'school_year', 'school_id', 'grade_level', 'fall_score', 'spring_score']
    missing = [column for column in required if column not in rows]
    if missing:
        raise ValueError(f'Missing diagnostic columns: {missing}')
    if rows[required].isna().any().any():
        raise ValueError('The fitted rows contain missing IDs, categories, or scores in the supplied data.')
    if 'subject' in rows and rows.subject.nunique() != 1:
        raise ValueError('Check one subject at a time.')
    if rows.duplicated(['student_id', 'school_year']).any():
        raise ValueError('Repeated student-year keys found within the subject model.')

    observed = np.asarray(result.model.endog, dtype=float)
    design = np.asarray(result.model.exog, dtype=float)
    rebuilt = np.asarray(build_design_matrices([design_info], rows, NA_action='raise')[0])
    if rebuilt.shape != design.shape or not np.allclose(rebuilt, design, rtol=1e-8, atol=1e-8):
        raise ValueError('Predictors do not match the fitted model. The supplied data may have changed after fitting.')
    if not np.allclose(rows.spring_score.to_numpy(dtype=float), observed, rtol=1e-8, atol=1e-8):
        raise ValueError('Spring scores do not match the fitted outcome. The supplied data may have changed after fitting.')
    if not np.isfinite(design).all() or not np.isfinite(observed).all():
        raise ValueError('The fitted outcome or design contains non-finite values.')
    if 'between_test_absence_rate' in rows:
        rate = rows.between_test_absence_rate.to_numpy(dtype=float)
        if not np.isfinite(rate).all() or ((rate < 0) | (rate > 1)).any():
            raise ValueError('Between-test absence must be a finite rate from 0 to 1 on the fitted rows.')

    counts = rows.groupby('student_id').size()
    covariance = getattr(result, 'cov_type', 'nonrobust')
    notes = []
    if covariance == 'cluster':
        groups = np.asarray(result.cov_kwds.get('groups'))
        if groups.ndim != 1 or len(groups) != len(rows) or pd.isna(groups).any():
            raise ValueError('Expected one nonmissing student cluster label per fitted row.')
        supplied_codes = pd.factorize(groups)[0]
        student_codes = pd.factorize(rows.student_id)[0]
        if not np.array_equal(supplied_codes, student_codes):
            raise ValueError('The covariance groups do not match student clusters in the fitted-row order.')
    else:
        notes.append('This fit is not clustered by student. Review uncertainty before using its confidence intervals.')
    if len(counts) < 30:
        notes.append('There are fewer than 30 student clusters. Review small-sample uncertainty rather than assuming clustered intervals are reliable.')
    notes.append('Student clustering does not allow dependence between different students in the same classroom or school.')

    n, columns = design.shape
    rank = int(np.linalg.matrix_rank(design))
    if rank < columns:
        notes.append('The design is rank deficient. Review redundant predictors or unsupported categories before interpreting coefficients.')
    if result.df_resid <= 0 or not np.isfinite(result.mse_resid) or result.mse_resid <= 0:
        raise ValueError('There is no usable residual variance or residual degrees of freedom for these diagnostics.')

    rows['fitted'] = np.asarray(result.fittedvalues)
    rows['residual'] = np.asarray(result.resid)
    influence = OLSInfluence(result)
    with np.errstate(divide='ignore', invalid='ignore'):
        rows['leverage'] = influence.hat_matrix_diag
        rows['studentized_residual'] = influence.resid_studentized_internal
        rows['cooks_distance'] = influence.cooks_distance[0]
    cook_cutoff = 4 / n
    leverage_cutoff = 2 * rank / n
    rows['influence_flag'] = (rows.cooks_distance > cook_cutoff) | (rows.leverage > leverage_cutoff)
    notes.append('Cook distance > 4/n and leverage > 2*rank/n are screening rules. They do not justify automatic deletion.')
    if not np.isfinite(rows[['leverage', 'studentized_residual', 'cooks_distance']].to_numpy()).all():
        notes.append('Some influence values are undefined. Inspect near-perfect leverage or a nearly exact fit.')
    names = result.model.exog_names
    for category in ['school_id', 'grade_level', 'school_year']:
        if category in names:
            notes.append(f'{category} appears as a numeric term. Confirm whether the intended specification treats it as a category.')

    ranges = rows.groupby(['grade_level', 'school_year', 'school_id']).agg(
        records=('student_id', 'size'), fall_min=('fall_score', 'min'),
        fall_median=('fall_score', 'median'), fall_max=('fall_score', 'max'),
    ).reset_index()
    if 'between_test_absence_rate' in rows:
        absence_ranges = rows.groupby(['grade_level', 'school_year', 'school_id']).agg(
            absence_min=('between_test_absence_rate', 'min'),
            absence_max=('between_test_absence_rate', 'max'),
        ).reset_index()
        ranges = ranges.merge(absence_ranges, on=['grade_level', 'school_year', 'school_id'], validate='one_to_one')
    overlap = ranges.groupby(['grade_level', 'school_year']).agg(
        schools_observed=('school_id', 'nunique'),
        common_fall_low=('fall_min', 'max'), common_fall_high=('fall_max', 'min'),
    ).reset_index()
    overlap['fall_ranges_overlap'] = overlap.common_fall_low <= overlap.common_fall_high
    notes.append('Range overlap is a first screen. Inspect the distributions too; overlapping endpoints do not establish adequate common support.')
    student_influence = rows.groupby('student_id').agg(
        records=('student_id', 'size'), flagged_records=('influence_flag', 'sum'),
        max_cooks_distance=('cooks_distance', 'max'),
        sum_cooks_distance=('cooks_distance', 'sum'),
    ).sort_values('sum_cooks_distance', ascending=False)
    notes.append('Student influence totals help locate records to review. They are not a formal leave-one-student-out analysis.')
    summary = pd.Series({
        'Fitted rows': n, 'Unique students': len(counts),
        'Students with repeated records': int((counts > 1).sum()),
        'Unused supplied rows': len(data) - n, 'Covariance type': covariance,
        'Design columns': columns, 'Design rank': rank,
        'Residual degrees of freedom': result.df_resid,
        'Flagged records (screen only)': int(rows.influence_flag.sum()),
    }, name=label)
    return {
        'label': label, 'summary': summary, 'rows': rows, 'notes': notes,
        'predictor_ranges': ranges, 'fall_overlap': overlap,
        'student_influence': student_influence,
        'cook_cutoff': cook_cutoff, 'leverage_cutoff': leverage_cutoff,
    }


In [ ]:
def plot_residual_checks(report):
    rows = report['rows']
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)

    def scatter(ax, x, y, xlabel, ylabel='Residual', zero=True):
        valid = np.isfinite(x) & np.isfinite(y)
        x, y = np.asarray(x)[valid], np.asarray(y)[valid]
        ax.scatter(x, y, s=12, alpha=.25, color='#227c77')
        if len(x) >= 8 and len(np.unique(x)) >= 4:
            smooth = lowess(y, x, frac=.6, return_sorted=True)
            ax.plot(smooth[:, 0], smooth[:, 1], color='#c26733', linewidth=2)
        if zero:
            ax.axhline(0, color='gray', linewidth=.8)
        ax.set(xlabel=xlabel, ylabel=ylabel)

    scatter(axes[0, 0], rows.fitted, rows.residual, 'Fitted Spring score')
    axes[0, 0].set_title('Mean relationship: look for curved patterns')
    scatter(axes[0, 1], rows.fitted, np.sqrt(np.abs(rows.studentized_residual)),
            'Fitted Spring score', 'Sqrt(|studentized residual|)', zero=False)
    axes[0, 1].set_title('Spread: look for a widening or narrowing band')
    ax = axes[0, 2]
    for grade, part in rows.groupby('grade_level'):
        name = 'K' if grade == 0 else str(grade)
        x, y = part.fall_score.to_numpy(), part.residual.to_numpy()
        points = ax.scatter(x, y, s=10, alpha=.18)
        color = points.get_facecolor()[0][:3]
        if len(part) >= 8 and part.fall_score.nunique() >= 4:
            smooth = lowess(y, x, frac=.7, return_sorted=True)
            ax.plot(smooth[:, 0], smooth[:, 1], color=color, linewidth=2, label=f'Grade {name}')
        else:
            ax.plot([], [], color=color, label=f'Grade {name} (few values)')
    ax.axhline(0, color='gray', linewidth=.8)
    ax.set(xlabel='Fall score', ylabel='Residual', title='Fall relationship within grades')
    ax.legend(fontsize=8)
    if 'between_test_absence_rate' in rows:
        scatter(axes[1, 0], rows.between_test_absence_rate * 100, rows.residual,
                'Between-test absence (%)')
        axes[1, 0].set_title('Absence relationship: check the shape')
    else:
        axes[1, 0].text(.5, .5, 'No between-test absence column supplied', ha='center', va='center', transform=axes[1, 0].transAxes)
        axes[1, 0].set_axis_off()
    for ax, column, title in [(axes[1, 1], 'grade_level', 'Residuals by grade'),
                               (axes[1, 2], 'school_id', 'Residuals by school')]:
        groups = list(rows.groupby(column))
        names = [('K' if key == 0 else str(key)) if column == 'grade_level' else str(key) for key, _ in groups]
        ax.boxplot([part.residual.to_numpy() for _, part in groups], tick_labels=names, showfliers=False)
        ax.axhline(0, color='gray', linewidth=.8)
        ax.set(title=title, xlabel=column, ylabel='Residual')
    fig.suptitle(f"{report['label']}: residual checks", fontsize=15)
    return fig


In [ ]:
def plot_influence_checks(report):
    rows = report['rows']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    stats.probplot(rows.residual, dist='norm', plot=axes[0])
    axes[0].set(title='Q-Q plot: inspect residual tails', ylabel='Ordered residual')
    positions = np.arange(len(rows))
    axes[1].scatter(positions, rows.cooks_distance, s=12, alpha=.5, color='#227c77')
    axes[1].axhline(report['cook_cutoff'], color='#c26733', linestyle='--', label='4/n screen')
    axes[1].set(title="Cook's distance: records to review", xlabel='Fitted row position', ylabel="Cook's distance")
    axes[1].legend(fontsize=8)
    valid = np.isfinite(rows.leverage) & np.isfinite(rows.studentized_residual)
    axes[2].scatter(rows.loc[valid, 'leverage'], rows.loc[valid, 'studentized_residual'],
                    s=14, alpha=.4, color='#227c77')
    axes[2].axvline(report['leverage_cutoff'], color='#c26733', linestyle='--', label='2*rank/n screen')
    axes[2].axhline(0, color='gray', linewidth=.8)
    axes[2].set(title='Leverage and residual size', xlabel='Leverage', ylabel='Studentized residual')
    axes[2].legend(fontsize=8)
    fig.suptitle(f"{report['label']}: tails and influence", fontsize=15)
    return fig


### Exploratory logistic fits and adjusted probabilities

`prepare_persistence_samples` · `fit_persistence_model` · `standardized_persistence`

In [ ]:
def prepare_persistence_samples(paired_samples):
    """Restrict the paired samples to the two recorded Fall-below levels."""
    levels = [
        "Well Below Benchmark",
        "Below Benchmark",
        "At Benchmark",
        "Above Benchmark",
    ]
    samples, audit = {}, []
    for subject, data in paired_samples.items():
        for column in ["fall_performance_level", "spring_performance_level"]:
            if data[column].isna().any() or not data[column].isin(levels).all():
                raise ValueError("Missing or unrecognized levels in the paired sample.")
        if (
            data[["student_id", "school_year"]].isna().any().any()
            or data.duplicated(["student_id", "school_year"]).any()
        ):
            raise ValueError("Use one complete key per student-year and subject.")
        sample = data.loc[data.fall_performance_level.isin(levels[:2])].copy()
        sample["fall_well_below"] = sample.fall_performance_level.eq(levels[0]).astype(
            int
        )
        sample["spring_below"] = sample.spring_performance_level.isin(
            levels[:2]
        ).astype(int)
        samples[subject] = sample
        audit.append(
            {
                "subject": subject,
                "paired_rows": len(data),
                "fall_below_rows": len(sample),
                "students": sample.student_id.nunique(),
                "spring_below_rows": int(sample.spring_below.sum()),
                "reached_at_above_rows": int(sample.spring_below.eq(0).sum()),
            }
        )
    return samples, pd.DataFrame(audit)


def fit_persistence_model(data, formula):
    """Fit a binary logit model without dropping rows; cluster by student."""
    required = ["student_id", "school_year", "fall_well_below", "spring_below"]
    if (
        not data.index.is_unique
        or data[required].isna().any().any()
        or data.duplicated(["student_id", "school_year"]).any()
    ):
        raise ValueError("Supply complete, uniquely indexed student-year records.")
    for column in ["fall_well_below", "spring_below"]:
        if not data[column].isin([0, 1]).all() or data[column].nunique() != 2:
            raise ValueError(
                "The binary predictor and outcome must each contain zero and one."
            )
    if data.student_id.nunique() < 2:
        raise ValueError("At least two student clusters are required.")
    model = smf.glm(formula, data=data, family=sm.families.Binomial(), missing="raise")
    if model.data.row_labels.tolist() != data.index.tolist():
        raise ValueError("The formula dropped rows.")
    if (
        not np.isfinite(model.exog).all()
        or np.linalg.matrix_rank(model.exog) != model.exog.shape[1]
    ):
        raise ValueError("The model design must be finite and full rank.")
    with warnings.catch_warnings():
        warnings.simplefilter("error", PerfectSeparationWarning)
        result = model.fit(
            maxiter=200,
            tol=1e-9,
            cov_type="cluster",
            cov_kwds={
                "groups": data.student_id.to_numpy(),
                "use_correction": True,
                "df_correction": True,
            },
            use_t=True,
        )
    if (
        not result.converged
        or not np.isfinite(result.params).all()
        or not np.isfinite(result.cov_params()).all().all()
    ):
        raise ValueError("The logistic fit did not produce stable finite estimates.")
    return result


def standardized_persistence(result, data):
    """Average over the same observed mix; delta intervals use clustered covariance."""
    design_info = result.model.data.design_info
    covariance = np.asarray(result.cov_params())
    beta = np.asarray(result.params)
    degrees = result.df_resid_inference
    critical = stats.t.ppf(0.975, degrees)
    gradients, risks = [], []
    for status, label in [(0, "Fall Below"), (1, "Fall Well Below")]:
        target = data.copy()
        for column in target.select_dtypes(include="boolean").columns:
            target[column] = target[column].astype(int)
        target["fall_well_below"] = status
        matrix = np.asarray(
            build_design_matrices([design_info], target, NA_action="raise")[0]
        )
        predictions = expit(matrix @ beta)
        probability = float(predictions.mean())
        if not 0 < probability < 1:
            raise ValueError("A boundary probability prevents a usable delta interval.")
        gradient = (matrix * (predictions * (1 - predictions))[:, None]).mean(axis=0)
        gradients.append(gradient)
        logit_gradient = gradient / (probability * (1 - probability))
        logit_se = np.sqrt(logit_gradient @ covariance @ logit_gradient)
        limits = expit(logit(probability) + np.array([-1, 1]) * critical * logit_se)
        risks.append(
            {
                "fall_group": label,
                "probability": probability,
                "ci_low": limits[0],
                "ci_high": limits[1],
            }
        )
    difference = risks[1]["probability"] - risks[0]["probability"]
    gradient = gradients[1] - gradients[0]
    se = np.sqrt(gradient @ covariance @ gradient)
    contrast = pd.Series(
        {
            "difference_pp": 100 * difference,
            "clustered_se_pp": 100 * se,
            "ci_low_pp": 100 * (difference - critical * se),
            "ci_high_pp": 100 * (difference + critical * se),
            "target_rows": len(data),
        }
    )
    return pd.DataFrame(risks), contrast


## 1. Load the nine tables

I keep ZIP codes and state student IDs as text to preserve their formatting. School and student IDs are the join keys.

Calendar dates are converted here because several checks use them. Dates in the other tables are converted in their own sections.


In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        "The original synthetic CSVs are required for a full run. See data/README.md."
    )

assessment = pd.read_csv(DATA_DIR / "assessment_results.csv")
attendance = pd.read_csv(DATA_DIR / "attendance_daily.csv")
discipline = pd.read_csv(DATA_DIR / "discipline_incidents.csv")
enrollments = pd.read_csv(DATA_DIR / "enrollments.csv")
calendar = pd.read_csv(DATA_DIR / "school_calendar.csv")
calendar["calendar_date"] = pd.to_datetime(calendar["calendar_date"], errors="raise")
schools = pd.read_csv(DATA_DIR / "schools.csv", dtype={"zip": "string"})
students = pd.read_csv(
    DATA_DIR / "students.csv", dtype={"zip": "string", "state_student_id": "string"}
)
teacher_assignments = pd.read_csv(DATA_DIR / "teacher_assignments.csv")
teachers = pd.read_csv(DATA_DIR / "teachers.csv")


In [ ]:
tables = {
    "schools": schools,
    "students": students,
    "enrollments": enrollments,
    "attendance_daily": attendance,
    "assessment_results": assessment,
    "discipline_incidents": discipline,
    "teachers": teachers,
    "teacher_assignments": teacher_assignments,
    "school_calendar": calendar,
}
source_inventory = pd.DataFrame(
    [
        {"table": name, "rows": len(df), "columns": len(df.columns)}
        for name, df in tables.items()
    ]
)
display(source_inventory)


## 2. Data quality checks

For each table, I check its IDs, missing values, dates, and relevant dictionary rules. The notes after each set of checks explain the results and how I use that table in the analysis.


### 2.1 Schools

One row represents one school. I check unique school IDs and names, missing values, and the grades each school serves.


In [ ]:
# Column types
schools.dtypes


In [ ]:
# Check school_id uniqueness
schools["school_id"].duplicated().sum()


In [ ]:
# Check missing values
schools.isna().sum()


In [ ]:
# Check low_grade <= high_grade
(schools["low_grade"] > schools["high_grade"]).sum()


In [ ]:
schools.groupby(["school_level", "low_grade", "high_grade"]).size()


In [ ]:
# Duplicate school names
schools["school_name"].duplicated().sum()


School IDs link the tables. ZIP codes stay as text to preserve leading zeros; grade limits stay numeric so I can check enrollment grades.


In [ ]:
dataset_overview_check(schools)


#### What the checks show

- The seven schools have distinct IDs and names: four elementary, two middle, and one high school.
- No missing values or reversed grade limits were found. The elementary schools serve K-5.
- This table provides the school names and characteristics for later joins. Names, addresses, and Title I labels have not been checked against an outside source.


#### How I use this table

- ZIP codes are text and `title_i` is boolean (True/False). I join on `school_id` and display `school_name`.
- The main analysis selects elementary schools 101-104. Other schools provide context only when it helps answer an LD question.


### 2.2 Students

One row represents one student across the three years. Student program labels are fixed in this file; it does not record their status separately for each year.


In [ ]:
# Column overview
dataset_overview_check(students)


In [ ]:
# Convert birth_date to datetime
students["birth_date"] = pd.to_datetime(students["birth_date"], errors="coerce")

# Check conversion
students["birth_date"].dtype

# Check whether any dates failed to convert
students["birth_date"].isna().sum()


In [ ]:
pd.Series(
    {
        "duplicate student_id": students["student_id"].duplicated().sum(),
        "duplicate state_student_id": students["state_student_id"].duplicated().sum(),
    }
)


In [ ]:
# Dictionary rule: a student cannot have both an IEP and a Section 504 plan.
display(
    pd.Series(
        {
            "special education and 504 overlap": int(
                (
                    students["special_education_flag"] & students["section_504_flag"]
                ).sum()
            )
        }
    )
)


In [ ]:
# Check all race/ethnicity levels against Hispanic/Latino flag
pd.crosstab(students["race_ethnicity"], students["hispanic_latino_flag"], margins=True)


In [ ]:
# Descriptive comparison only: home language does not uniquely determine EL status.
pd.crosstab(students["home_language"], students["english_learner_flag"], margins=True)


In [ ]:
# Count missing guardian names without displaying student contact details.
students["guardian_name"].isna().sum()


#### What the checks show

- Both student identifiers are unique. Birth dates converted successfully, and ZIP codes are text.
- Two guardian names are blank, as the dictionary notes. These students remain in the analysis.
- The Hispanic/Latino indicator matches the race/ethnicity categories. No student has both SPED and Section 504 flags, consistent with the dictionary.
- The language/EL table describes this synthetic dataset; it is not a rule for real students. Because demographic and program labels are fixed, changes in yearly group proportions do not show individuals entering or leaving a program.


#### How I use this table

- `birth_date` is a date. ZIP and state student ID remain text; `student_id` is the numeric join key.
- Blank guardian names do not affect the planned measures. Analysis outputs avoid student names and contact details.
- Program labels describe the students in the file. They cannot show when a student became eligible for, or left, a program.


### 2.3 Enrollments

One row represents one student in one school year. It gives the school, grade, and entry and exit dates used to check attendance, tests, and incidents.


In [ ]:
dataset_overview_check(enrollments)


In [ ]:
# Parse entry dates
enrollments["entry_date"] = pd.to_datetime(enrollments["entry_date"], errors="raise")
enrollments["entry_date"].isna().sum()


In [ ]:
# Parse exit dates
enrollments["exit_date"] = pd.to_datetime(enrollments["exit_date"], errors="raise")
enrollments["exit_date"].isna().sum()


In [ ]:
# A student can appear in multiple years, but only once within a year.
enrollments.duplicated(subset=["student_id", "school_year"]).sum()


In [ ]:
# Exit dates before entry dates
(enrollments["exit_date"] < enrollments["entry_date"]).sum()


In [ ]:
# Enrollment dates must belong to the school year recorded on each row.
enrollment_year_bounds = (
    calendar.groupby("school_year")["calendar_date"]
    .agg(year_start="min", year_end="max")
    .reset_index()
)
enrollment_year_check = enrollments.merge(
    enrollment_year_bounds, on="school_year", how="left", validate="many_to_one"
)
enrollment_year_issues = pd.Series(
    {
        "school years absent from calendar": int(
            enrollment_year_check["year_start"].isna().sum()
        ),
        "entries before school-year calendar": int(
            enrollment_year_check["entry_date"]
            .lt(enrollment_year_check["year_start"])
            .sum()
        ),
        "exits after school-year calendar": int(
            enrollment_year_check["exit_date"]
            .gt(enrollment_year_check["year_end"])
            .sum()
        ),
    }
)
display(enrollment_year_issues)


In [ ]:
# Check school_year and school_year_label consistency
enrollments.groupby(["school_year", "school_year_label"]).size()


In [ ]:
# Grades outside the school range
enrollment_school_check = enrollments.merge(
    schools[["school_id", "school_level", "low_grade", "high_grade"]],
    on="school_id",
    how="left",
    validate="many_to_one",
)

invalid_grade_school = enrollment_school_check[
    (enrollment_school_check["grade_level"] < enrollment_school_check["low_grade"])
    | (enrollment_school_check["grade_level"] > enrollment_school_check["high_grade"])
]

invalid_grade_school
len(invalid_grade_school)


In [ ]:
# Check grade_level and grade_code consistency
enrollments.groupby(["grade_level", "grade_code"]).size()


In [ ]:
# Check reference IDs
# Student_id should exist in students
(~enrollments["student_id"].isin(students["student_id"])).sum()


In [ ]:
# School_id should exist in schools
(~enrollments["school_id"].isin(schools["school_id"])).sum()


In [ ]:
# Homeroom_teacher_id should exist in teachers
(~enrollments["homeroom_teacher_id"].isin(teachers["teacher_id"])).sum()


In [ ]:
# Check homeroom assignments in the same school and year.
homeroom_assignment_check = enrollments.merge(
    teacher_assignments[["teacher_id", "school_year", "school_id"]],
    left_on=["homeroom_teacher_id", "school_year"],
    right_on=["teacher_id", "school_year"],
    how="left",
    suffixes=("_enrollment", "_assignment"),
    validate="many_to_one",
)
matched_homeroom = homeroom_assignment_check["teacher_id"].notna()
display(
    pd.Series(
        {
            "homeroom missing teacher-year assignment": int((~matched_homeroom).sum()),
            "homeroom school mismatch": int(
                (
                    matched_homeroom
                    & homeroom_assignment_check["school_id_enrollment"].ne(
                        homeroom_assignment_check["school_id_assignment"]
                    )
                ).sum()
            ),
        }
    )
)


In [ ]:
# Duplicate enrollment IDs
enrollments["enrollment_id"].duplicated().sum()


In [ ]:
# Grade changes between observed enrollment records
ordered_enrollments = enrollments.sort_values(["student_id", "school_year"])
grade_change = ordered_enrollments.groupby("student_id")["grade_level"].diff()
grade_change.value_counts(dropna=False).sort_index()


In [ ]:
# Check graduation and exit type consistency
enrollments[enrollments["exit_type"] == "Graduated"]["grade_level"].value_counts()


In [ ]:
enrollments.groupby("entry_type")["entry_date"].agg(["min", "max", "nunique"])


In [ ]:
enrollments.groupby("exit_type")["exit_date"].agg(["min", "max", "nunique"])


#### What the checks show

- Enrollment IDs and student-year combinations are unique. Student, school, and homeroom teacher IDs match their reference tables.
- Homeroom teachers have an assignment at the student's school in the same year.
- Entry dates are on or before exit dates, and both fall within the recorded school-year calendar. Grades fit the schools' grade ranges; year and grade labels are consistent.
- Students with records in adjacent years advance one grade. A gap of several years would need a separate review.
- Attendance joins through `enrollment_id`. Tests and incidents join through student and school year, with school, grade, and dates checked afterward. `validate="many_to_one"` checks that each event matches at most one reference row.


#### How I use this table

- Entry and exit fields are dates. `school_year` names the spring year: 2024 means 2023-24. I use `school_year_label` in displays.
- School profiles count anyone enrolled during the year. A count on a particular date would need a different definition.
- I summarize daily attendance and other event records before joining them. Joining event tables directly on student ID could multiply the rows.


### 2.4 Daily attendance

One row represents an enrolled instructional day for a student. I check that every expected day has a record before calculating absence rates. A missing record would not tell me whether the student was absent.


In [ ]:
# Column overview
dataset_overview_check(attendance)


In [ ]:
# Check attendance code/status consistency
pd.crosstab(
    attendance["attendance_code"], attendance["attendance_status"], margins=True
)


In [ ]:
# Check is_absent
pd.crosstab(attendance["attendance_status"], attendance["is_absent"], margins=True)


In [ ]:
# Check days_attended
pd.crosstab(attendance["attendance_status"], attendance["days_attended"], margins=True)


In [ ]:
# Check absence_reason
pd.crosstab(
    attendance["absence_reason"].fillna("Missing"),
    attendance["attendance_status"],
    margins=True,
)


In [ ]:
# Check foreign keys
# Enrollment_id
(~attendance["enrollment_id"].isin(enrollments["enrollment_id"])).sum()


In [ ]:
# Student_id
(~attendance["student_id"].isin(students["student_id"])).sum()


In [ ]:
# School_id
(~attendance["school_id"].isin(schools["school_id"])).sum()


In [ ]:
# Convert attendance_date to datetime
attendance["attendance_date"] = pd.to_datetime(
    attendance["attendance_date"], errors="coerce"
)

# Check conversion failures
attendance["attendance_date"].isna().sum()


In [ ]:
# Check primary key: one row per student per attendance date
attendance.duplicated(subset=["student_id", "attendance_date"]).sum()


In [ ]:
# Check attendance date with enrollment period
attendance_date_check = attendance.merge(
    enrollments[["enrollment_id", "entry_date", "exit_date"]],
    on="enrollment_id",
    how="left",
    validate="many_to_one",
)

invalid_attendance_dates = attendance_date_check[
    (attendance_date_check["attendance_date"] < attendance_date_check["entry_date"])
    | (attendance_date_check["attendance_date"] > attendance_date_check["exit_date"])
]

len(invalid_attendance_dates)


In [ ]:
# Row-level consistency check
attendance_enrollment_check = attendance.merge(
    enrollments[["enrollment_id", "student_id", "school_id", "school_year"]],
    on="enrollment_id",
    how="left",
    suffixes=("_attendance", "_enrollment"),
    validate="many_to_one",
)


In [ ]:
# Student consistency
(
    attendance_enrollment_check["student_id_attendance"]
    != attendance_enrollment_check["student_id_enrollment"]
).sum()


In [ ]:
# School consistency
(
    attendance_enrollment_check["school_id_attendance"]
    != attendance_enrollment_check["school_id_enrollment"]
).sum()


In [ ]:
# School year consistency
(
    attendance_enrollment_check["school_year_attendance"]
    != attendance_enrollment_check["school_year_enrollment"]
).sum()


In [ ]:
calendar_check = attendance.merge(
    calendar[["calendar_date", "school_year", "is_instructional_day"]],
    left_on=["attendance_date", "school_year"],
    right_on=["calendar_date", "school_year"],
    how="left",
    validate="many_to_one",
)

calendar_check["is_instructional_day"].value_counts(dropna=False)


In [ ]:
# Count expected attendance days within each enrollment.
# The calendar join expands rows; date and uniqueness checks also apply.

expected_days = enrollments[
    ["enrollment_id", "school_year", "entry_date", "exit_date"]
].merge(
    calendar.loc[calendar["is_instructional_day"], ["school_year", "calendar_date"]],
    on="school_year",
    how="left",
    validate="many_to_many",
)
expected_days = expected_days.loc[
    expected_days["calendar_date"].between(
        expected_days["entry_date"], expected_days["exit_date"]
    )
]
expected_by_enrollment = expected_days.groupby("enrollment_id").size()
observed_by_enrollment = attendance.groupby("enrollment_id").size()
coverage = (
    enrollments[["enrollment_id"]]
    .set_index("enrollment_id")
    .join(expected_by_enrollment.rename("expected_days"))
    .join(observed_by_enrollment.rename("observed_rows"))
    .fillna(0)
    .astype(int)
)
display(
    pd.Series(
        {
            "expected enrolled instructional days": int(
                coverage["expected_days"].sum()
            ),
            "observed attendance rows": int(coverage["observed_rows"].sum()),
            "enrollments with a coverage gap": int(
                coverage["expected_days"].ne(coverage["observed_rows"]).sum()
            ),
            "duplicate enrollment-date rows": int(
                attendance.duplicated(["enrollment_id", "attendance_date"]).sum()
            ),
        }
    )
)
del expected_days


#### What the checks show

- Student-date combinations are unique. Records match the student's enrollment, school, year, enrollment dates, and an instructional calendar day.
- `P` (present) and `T` (tardy) count as attended. `AE` and `AU` count as absent. An absence reason is recorded only for `AE`.
- Observed day counts match the expected instructional days in each enrollment. Together with the date and uniqueness checks, this supports complete daily coverage.
- Absence rate is absent days divided by enrolled instructional days. I use the same definition for school and subgroup comparisons.


#### How I use this table

- `attendance_date` is a date. `days_attended` is 0 or 1, and `is_absent` is True/False. Tardy counts as present under the supplied definition.
- Rates use the days a student was enrolled and school was in session. I summarize daily rows before joining them to the student-year table.
- Blank `absence_reason` values outside excused absences are expected. The daily file is larger than a standard Excel worksheet can hold.


### 2.5 Assessment results

One row represents one test administration for one student. Benchmark and state-test scores use different scales, so I analyze them separately.


In [ ]:
# Column overview
dataset_overview_check(assessment)


In [ ]:
# Convert test_date to datetime
assessment["test_date"] = pd.to_datetime(assessment["test_date"], errors="coerce")

# Check conversion failures
assessment["test_date"].isna().sum()


In [ ]:
# Check primary key
assessment["assessment_result_id"].duplicated().sum()


In [ ]:
# Screen for potential repeated test records within the same
# Student-year-assessment-subject-window; this is not the declared primary key.
assessment.duplicated(
    subset=["student_id", "school_year", "assessment_name", "subject", "test_window"]
).sum()


In [ ]:
pd.Series(
    {
        "student IDs absent from students": (
            ~assessment["student_id"].isin(students["student_id"])
        ).sum(),
        "school IDs absent from schools": (
            ~assessment["school_id"].isin(schools["school_id"])
        ).sum(),
    }
)


In [ ]:
assessment_enrollment_check = assessment.merge(
    enrollments[["student_id", "school_year", "school_id", "grade_level"]],
    on=["student_id", "school_year"],
    how="left",
    suffixes=("_assessment", "_enrollment"),
    validate="many_to_one",
)


In [ ]:
# Check school
(
    assessment_enrollment_check["school_id_assessment"]
    != assessment_enrollment_check["school_id_enrollment"]
).sum()


In [ ]:
# Check grade
(
    assessment_enrollment_check["grade_level_tested"]
    != assessment_enrollment_check["grade_level"]
).sum()


In [ ]:
# Check assessment × subject
pd.crosstab(assessment["assessment_name"], assessment["subject"], margins=True)


In [ ]:
# Check assessment × test window
pd.crosstab(assessment["assessment_name"], assessment["test_window"], margins=True)


In [ ]:
# Check assessment × grade
pd.crosstab(assessment["assessment_name"], assessment["grade_level_tested"])


In [ ]:
assessment.groupby(["assessment_name", "subject"])["grade_level_tested"].unique()


In [ ]:
# Plausibility check only: these assessment families use different score scales.
assessment.groupby("assessment_name")["scale_score"].agg(["count", "min", "max"])


In [ ]:
# Check score ≥ 400 → meets standard
state_tests = assessment[
    assessment["assessment_name"].isin(
        ["State Summative Assessment", "State End-of-Course"]
    )
]

(state_tests["met_standard"] != (state_tests["scale_score"] >= 400)).sum()


In [ ]:
# Check performance level × met_standard
pd.crosstab(state_tests["performance_level"], state_tests["met_standard"], margins=True)


In [ ]:
# Check national_percentile
benchmark = assessment[assessment["assessment_name"] == "Interim Benchmark"]

benchmark["national_percentile"].agg(["min", "max", "count"])


In [ ]:
benchmark["met_standard"].notna().sum()


In [ ]:
# Check met_standard structural missingness
pd.crosstab(assessment["assessment_name"], assessment["met_standard"].fillna("Missing"))


In [ ]:
# Benchmark should have national percentile
benchmark["national_percentile"].isna().sum()


In [ ]:
# State tests should NOT have national percentile
state_tests["national_percentile"].notna().sum()


In [ ]:
# Check performance_level and assessment family
# Check performance level by assessment type
pd.crosstab(
    assessment["assessment_name"], assessment["performance_level"], margins=True
)


In [ ]:
# Check test_date within enrollment period
# Check assessment date within enrollment period
assessment_date_check = assessment.merge(
    enrollments[["student_id", "school_year", "entry_date", "exit_date"]],
    on=["student_id", "school_year"],
    how="left",
    validate="many_to_one",
)

invalid_assessment_dates = assessment_date_check[
    (assessment_date_check["test_date"] < assessment_date_check["entry_date"])
    | (assessment_date_check["test_date"] > assessment_date_check["exit_date"])
]

len(invalid_assessment_dates)


In [ ]:
# Assessment dates should appear as instructional days in the district calendar.
assessment_calendar_check = assessment.merge(
    calendar[["calendar_date", "school_year", "is_instructional_day"]],
    left_on=["test_date", "school_year"],
    right_on=["calendar_date", "school_year"],
    how="left",
    validate="many_to_one",
)
matched_assessment_day = assessment_calendar_check["calendar_date"].notna()
display(
    pd.Series(
        {
            "assessment dates missing from calendar": int(
                (~matched_assessment_day).sum()
            ),
            "assessment dates on noninstructional days": int(
                (
                    matched_assessment_day
                    & assessment_calendar_check["is_instructional_day"].eq(False)
                ).sum()
            ),
        }
    )
)


#### What the checks show

- Result IDs are unique. Each student-year-assessment-subject-window combination is also unique in this file. Test records match enrollment school, grade, and dates.
- Test dates are instructional days in the calendar.
- State tests use a score of 400 as the standard. `met_standard` is blank for benchmarks; `national_percentile` is blank for state tests, as expected.
- Tables by subject, grade, and test window show whether the combinations look reasonable. They do not establish whether every eligible student tested.
- Before comparing results, I check participation and state who is counted. Benchmark and state-test scale scores are kept separate.


#### How I use this table

- `test_date` is a date. The final type conversions below store `met_standard` as True/False with blanks allowed, and `national_percentile` as a whole number with blanks allowed.
- Assessment family, subject, and window stay in the groupings. Participation checks account for the different grades and tests covered.


### 2.6 Discipline incidents

One row represents one recorded incident. A student can have several incidents, so I report incident counts and students involved separately.


In [ ]:
# Column overview
dataset_overview_check(discipline)


In [ ]:
# Incident_date convert datetime
# Convert incident_date to datetime
discipline["incident_date"] = pd.to_datetime(
    discipline["incident_date"], errors="coerce"
)

# Check conversion failures
discipline["incident_date"].isna().sum()


In [ ]:
# Check primary key
discipline["incident_id"].duplicated().sum()


In [ ]:
pd.Series(
    {
        "student IDs absent from students": (
            ~discipline["student_id"].isin(students["student_id"])
        ).sum(),
        "school IDs absent from schools": (
            ~discipline["school_id"].isin(schools["school_id"])
        ).sum(),
        "reporter IDs absent from teachers": (
            ~discipline["reported_by_teacher_id"].isin(teachers["teacher_id"])
        ).sum(),
    }
)


In [ ]:
# The reporting teacher should be assigned to the incident school that year.
reporter_assignment_check = discipline.merge(
    teacher_assignments[["teacher_id", "school_year", "school_id"]],
    left_on=["reported_by_teacher_id", "school_year"],
    right_on=["teacher_id", "school_year"],
    how="left",
    suffixes=("_incident", "_assignment"),
    validate="many_to_one",
)
matched_reporter = reporter_assignment_check["teacher_id"].notna()
display(
    pd.Series(
        {
            "reporter missing teacher-year assignment": int((~matched_reporter).sum()),
            "reporter school mismatch": int(
                (
                    matched_reporter
                    & reporter_assignment_check["school_id_incident"].ne(
                        reporter_assignment_check["school_id_assignment"]
                    )
                ).sum()
            ),
        }
    )
)


In [ ]:
# Check enrollment and school grade
discipline_enrollment_check = discipline.merge(
    enrollments[
        [
            "student_id",
            "school_year",
            "school_id",
            "grade_level",
            "entry_date",
            "exit_date",
        ]
    ],
    on=["student_id", "school_year"],
    how="left",
    suffixes=("_discipline", "_enrollment"),
    validate="many_to_one",
)


In [ ]:
# School
(
    discipline_enrollment_check["school_id_discipline"]
    != discipline_enrollment_check["school_id_enrollment"]
).sum()


In [ ]:
# Grade
(
    discipline_enrollment_check["grade_level_discipline"]
    != discipline_enrollment_check["grade_level_enrollment"]
).sum()


In [ ]:
# Check Incident date within enrollment date
invalid_incident_dates = discipline_enrollment_check[
    (
        discipline_enrollment_check["incident_date"]
        < discipline_enrollment_check["entry_date"]
    )
    | (
        discipline_enrollment_check["incident_date"]
        > discipline_enrollment_check["exit_date"]
    )
]

len(invalid_incident_dates)


In [ ]:
# Check incident date in instructional date
discipline_calendar_check = discipline.merge(
    calendar[["calendar_date", "school_year", "is_instructional_day"]],
    left_on=["incident_date", "school_year"],
    right_on=["calendar_date", "school_year"],
    how="left",
    validate="many_to_one",
)

discipline_calendar_check["is_instructional_day"].value_counts(dropna=False)


In [ ]:
# Check action_taken × days_removed
pd.crosstab(discipline["action_taken"], discipline["days_removed"], margins=True)


In [ ]:
# Check incident_time format (HH:MM)
(~discipline["incident_time"].str.match(r"^(?:[01]\d|2[0-3]):[0-5]\d$")).sum()


#### What the checks show

- Incident IDs and reference IDs pass the checks. School, grade, incident dates, and instructional-day status match enrollment and calendar records.
- Reporting teachers have an assignment at the incident school in that year.
- The action/days-removed table provides a reasonableness check before I interpret removal days.
- School comparisons need a stated denominator, such as enrolled students or enrolled days. Incident counts and distinct students involved answer different questions.


#### How I use this table

- `incident_date` is a date. `incident_time` remains checked `HH:MM` text; no time-of-day analysis is planned. Severity and removal days are numeric.
- I use a stated denominator for school comparisons and show counts so small numbers are visible.


### 2.7 Teachers

One row represents a teacher employed during these three years. Current status and home school describe the teacher record, not a separate staffing record for each year.


In [ ]:
# Column overview
dataset_overview_check(teachers)


In [ ]:
# Convert date columns
for col in ["birth_date", "hire_date", "exit_date"]:
    teachers[col] = pd.to_datetime(teachers[col], errors="coerce")


In [ ]:
# Chronology checks after date conversion; a missing exit date means current staff.
display(
    pd.Series(
        {
            "teacher exits before hire": int(
                (
                    teachers["exit_date"].notna()
                    & teachers["exit_date"].lt(teachers["hire_date"])
                ).sum()
            ),
            "teacher birth dates on or after hire": int(
                teachers["birth_date"].ge(teachers["hire_date"]).sum()
            ),
        }
    )
)


In [ ]:
teachers[["birth_date", "hire_date", "exit_date"]].isna().sum()


In [ ]:
pd.Series(
    {
        "duplicate teacher_id": teachers["teacher_id"].duplicated().sum(),
        "duplicate email": teachers["email"].duplicated().sum(),
    }
)


In [ ]:
# Employment status × exit date
pd.crosstab(teachers["employment_status"], teachers["exit_date"].notna(), margins=True)


In [ ]:
# Home_school_id should exist in schools
(~teachers["home_school_id"].isin(schools["school_id"])).sum()


In [ ]:
# Check fte
teachers["fte"].value_counts()


In [ ]:
# Years_experience_range
teachers["years_experience"].agg(["min", "max", "mean", "median"])


#### What the checks show

- Teacher IDs and emails are unique. Hire and birth dates converted successfully. Missing exit dates match active status.
- No exit date precedes hire date, and no birth date is on or after hire date.
- FTE (full-time equivalent) and experience values look reasonable. Home-school IDs match the school table.
- Yearly staffing comes from `teacher_assignments`. Current home school, status, and experience as of June 2026 cannot be used as historical values for all three years.


#### How I use this table

- Birth, hire, and exit fields are dates. Active teachers are expected to have no exit date.
- I use the assignment table for yearly staffing. Current teacher attributes do not show past assignments or experience separately by year.


### 2.8 Teacher assignments

One row represents one teacher's assignment in one school year. This is the source for yearly school staffing summaries.


In [ ]:
# Column overview
dataset_overview_check(teacher_assignments)


In [ ]:
pd.Series(
    {
        "duplicate assignment ID": teacher_assignments["teacher_assignment_id"]
        .duplicated()
        .sum(),
        "duplicate teacher-year": teacher_assignments.duplicated(
            subset=["teacher_id", "school_year"]
        ).sum(),
    }
)


In [ ]:
pd.Series(
    {
        "teacher IDs absent from teachers": (
            ~teacher_assignments["teacher_id"].isin(teachers["teacher_id"])
        ).sum(),
        "school IDs absent from schools": (
            ~teacher_assignments["school_id"].isin(schools["school_id"])
        ).sum(),
    }
)


In [ ]:
# Check assignment grade level
pd.crosstab(
    teacher_assignments["assignment"],
    teacher_assignments["grade_level"].fillna("Missing"),
    margins=True,
)


In [ ]:
grade_map = {
    "Kindergarten": 0,
    "Grade 1": 1,
    "Grade 2": 2,
    "Grade 3": 3,
    "Grade 4": 4,
    "Grade 5": 5,
}

grade_teacher = teacher_assignments[
    teacher_assignments["assignment"].isin(grade_map)
].copy()

(grade_teacher["grade_level"] != grade_teacher["assignment"].map(grade_map)).sum()


In [ ]:
non_grade_teacher = teacher_assignments[
    ~teacher_assignments["assignment"].isin(grade_map)
]

non_grade_teacher["grade_level"].notna().sum()


In [ ]:
teacher_assignments["fte"].value_counts()


In [ ]:
# The assignment ID is defined as teacher_id followed by school_year.
expected_assignment_id = (
    teacher_assignments["teacher_id"].astype(str)
    + teacher_assignments["school_year"].astype(str)
).astype("int64")
display(
    pd.Series(
        {
            "assignment ID mismatches": int(
                teacher_assignments["teacher_assignment_id"]
                .ne(expected_assignment_id)
                .sum()
            )
        }
    )
)


In [ ]:
# Each teacher-year assignment must overlap the teacher's employment period.
school_year_bounds = (
    calendar.groupby("school_year")["calendar_date"]
    .agg(school_year_start="min", school_year_end="max")
    .reset_index()
)
teacher_employment_check = teacher_assignments.merge(
    teachers[["teacher_id", "hire_date", "exit_date"]],
    on="teacher_id",
    how="left",
    validate="many_to_one",
).merge(
    school_year_bounds,
    on="school_year",
    how="left",
    validate="many_to_one",
)
display(
    pd.Series(
        {
            "assignments outside employment period": int(
                (
                    teacher_employment_check["hire_date"].gt(
                        teacher_employment_check["school_year_end"]
                    )
                    | teacher_employment_check["exit_date"].lt(
                        teacher_employment_check["school_year_start"]
                    )
                ).sum()
            ),
            "assignments missing school-year bounds": int(
                teacher_employment_check["school_year_start"].isna().sum()
            ),
        }
    )
)


In [ ]:
# Find employed teacher-years without an assignment.
teacher_years = (
    teachers.assign(_join_key=1)
    .merge(school_year_bounds.assign(_join_key=1), on="_join_key")
    .drop(columns="_join_key")
)
expected_teacher_years = teacher_years.loc[
    teacher_years["hire_date"].le(teacher_years["school_year_end"])
    & (
        teacher_years["exit_date"].isna()
        | teacher_years["exit_date"].ge(teacher_years["school_year_start"])
    ),
    ["teacher_id", "school_year"],
]
teacher_year_coverage = expected_teacher_years.merge(
    teacher_assignments[["teacher_id", "school_year"]],
    on=["teacher_id", "school_year"],
    how="left",
    indicator=True,
    validate="one_to_one",
)
teacher_year_missing = teacher_year_coverage.loc[
    teacher_year_coverage["_merge"].eq("left_only")
]
display(
    pd.Series(
        {
            "employment-overlapping teacher-years without assignment": len(
                teacher_year_missing
            )
        }
    )
)


#### What the checks show

- Assignment IDs and teacher-year combinations are unique. IDs follow the dictionary's teacher-ID/year format, and reference IDs match.
- Every recorded assignment overlaps the teacher's employment dates. Recorded FTE values are 0.5 or 1.0.
- Grade is populated for elementary grade-specific roles and blank for other roles, as expected.
- Looking in the other direction, 14 teacher-year periods overlap employment but have no assignment record. The source or assignment-year definition needs clarification; I do not invent assignments.
- School and FTE come from each year's assignment. Current `home_school_id` cannot fill a historical assignment gap.


#### How I use this table

- Grade is stored as a whole number with blanks allowed. I sum FTE rather than treating every assignment as a full-time teacher.
- The 14 missing teacher-year assignments remain an open source question. I do not fill their school or FTE from current teacher records. Staffing trends need this qualification until coverage is clarified.


### 2.9 School calendar

One row represents a weekday in a school year. Instructional-day flags identify which dates belong in attendance denominators.


In [ ]:
# Column overview
dataset_overview_check(calendar)


In [ ]:
# Check primary key
calendar["calendar_date"].duplicated().sum()


In [ ]:
# Check school_year × school_year_label
calendar.groupby(["school_year", "school_year_label"]).size()


In [ ]:
# Check day_of_week against calendar_date
(calendar["calendar_date"].dt.day_name() != calendar["day_of_week"]).sum()


In [ ]:
# Check is_instructional_day × day_type
pd.crosstab(calendar["day_type"], calendar["is_instructional_day"], margins=True)


In [ ]:
# Check instructional_day_number
# Non-instructional days should have missing instructional_day_number
calendar.loc[
    ~calendar["is_instructional_day"], "instructional_day_number"
].notna().sum()


In [ ]:
# Instructional days should have instructional_day_number
calendar.loc[calendar["is_instructional_day"], "instructional_day_number"].isna().sum()


In [ ]:
calendar[calendar["is_instructional_day"]].groupby("school_year")[
    "instructional_day_number"
].agg(["min", "max", "nunique"])


In [ ]:
# A weekday calendar should contain no Saturday or Sunday dates.
display(
    pd.Series(
        {
            "calendar weekend dates": int(
                calendar["calendar_date"].dt.dayofweek.ge(5).sum()
            )
        }
    )
)


In [ ]:
# Strictly require day 1, 2, ... in date order within each school year.
day_sequence_ok = (
    calendar.loc[calendar["is_instructional_day"]]
    .sort_values(["school_year", "calendar_date"])
    .groupby("school_year")["instructional_day_number"]
    .apply(
        lambda numbers: pd.to_numeric(numbers, errors="coerce").tolist()
        == list(range(1, len(numbers) + 1))
    )
)
display(pd.Series({"calendar numbering failures": int((~day_sequence_ok).sum())}))
display(day_sequence_ok.rename("numbering_is_continuous"))


In [ ]:
# The attendance denominator depends on a complete weekday calendar.
calendar_weekday_coverage = []
for year, group in calendar.groupby("school_year"):
    expected_weekdays = pd.date_range(
        start=group["calendar_date"].min(),
        end=group["calendar_date"].max(),
        freq="B",  # Monday-Friday, including noninstructional holidays.
    )
    actual_dates = pd.DatetimeIndex(group["calendar_date"])
    calendar_weekday_coverage.append(
        {
            "school_year": year,
            "expected_weekdays": len(expected_weekdays),
            "actual_rows": len(group),
            "missing_weekdays": len(expected_weekdays.difference(actual_dates)),
            "unexpected_dates": len(actual_dates.difference(expected_weekdays)),
        }
    )
display(pd.DataFrame(calendar_weekday_coverage))


#### What the checks show

- Calendar dates are unique. Year labels and weekday names match the dates; no weekend dates were found.
- Every weekday between each year's first and last calendar date is present.
- Instructional-day numbers run in date order from 1 to 188, 186, and 185 in the three years. Noninstructional dates have no instructional-day number.
- Noninstructional weekdays belong in the calendar but should have no attendance records.


#### How I use this table

- Calendar dates are dates and `is_instructional_day` is True/False. Instructional-day numbers are whole numbers with blanks allowed.
- I use instructional dates to count enrolled school days and check event dates. Noninstructional days are excluded from absence-rate denominators.


### 2.10 Decisions from the quality checks

| Finding | Decision | What it means for analysis |
|---|---|---|
| Two blank guardian names | Keep the students | Planned school measures are unaffected |
| Expected blanks in attendance and tests | Follow the dictionary | Do not fill these blanks with zero |
| Fixed student program labels | Describe them as labels in the supplied file | They cannot show individuals changing eligibility by year |
| Test participation is not guaranteed | Review opportunities and participation in Section 3.4D | State who is counted in each test comparison |
| 14 teacher-year periods without assignments | Leave school and FTE unfilled | Staffing trends require a coverage qualification |
| Yearly staffing | Use `teacher_assignments` | Current teacher status does not establish past staffing |

The checks support the tested relationships within these files. They do not verify the extract against the original SIS. Later sections state any further exclusions and measure definitions.


### 2.11 Final data types

These conversions let whole-number and True/False fields retain the blanks allowed by the dictionary. They leave missing values and source CSVs unchanged. Attendance indicators, FTE, removal days, and scores already have suitable numeric types.


In [ ]:
# Keep expected blanks when converting integer and boolean fields.
assessment["met_standard"] = assessment["met_standard"].astype("boolean")
assessment["national_percentile"] = assessment["national_percentile"].astype("Int64")
teacher_assignments["grade_level"] = teacher_assignments["grade_level"].astype("Int64")
calendar["instructional_day_number"] = calendar["instructional_day_number"].astype(
    "Int64"
)

display(
    pd.DataFrame(
        [
            {
                "table": "assessment_results",
                "column": "met_standard",
                "dtype": str(assessment["met_standard"].dtype),
                "missing": assessment["met_standard"].isna().sum(),
            },
            {
                "table": "assessment_results",
                "column": "national_percentile",
                "dtype": str(assessment["national_percentile"].dtype),
                "missing": assessment["national_percentile"].isna().sum(),
            },
            {
                "table": "teacher_assignments",
                "column": "grade_level",
                "dtype": str(teacher_assignments["grade_level"].dtype),
                "missing": teacher_assignments["grade_level"].isna().sum(),
            },
            {
                "table": "school_calendar",
                "column": "instructional_day_number",
                "dtype": str(calendar["instructional_day_number"].dtype),
                "missing": calendar["instructional_day_number"].isna().sum(),
            },
        ]
    )
)


## 3. Exploratory data analysis

The six EDA tasks, the review of other possible priorities, and the focused student-path analysis in 3.10 are complete. This section describes the results; Section 4 now reports the primary adjusted models and their diagnostic review. Section 4.3A contains the univariable preliminary analysis.

The briefing should help the new LD understand the four schools and decide what to ask principals first this fall. I start with school context, then compare learning starting points and Spring results, and finally examine attendance alongside learning.

School profiles and headline comparisons use 2025-26 unless noted. Yearly comparisons cover 2023-24 to 2025-26. Attendance-score correlations and absence-band summaries combine all three years, with subjects and grades kept separate. Historical grades are not the upcoming fall roster.


### 3.1 What QC made me ask

QC identified features that affect fair comparisons: students enter and leave during the year, tests occur in several windows, and calendars have different numbers of school days. These observations helped shape the questions. The school differences themselves come from the EDA that follows.

| What I noticed | Question it raised | Why the LD needs to know |
|---|---|---|
| Schools and enrollments (2.1, 2.3): four schools serve K-5. Each student-year has one enrollment, with some midyear entries and exits. | Do school size, grade mix, student needs, and time enrolled differ? | The LD needs to know whom each school serves before comparing averages. Time enrolled also affects attendance and testing opportunities. |
| Students (2.2): EL, economic disadvantage, and SPED labels are available but fixed across years. Non-English home language and EL status correspond perfectly in this synthetic file. | Could student background help explain school differences? | These labels provide context, but cannot show yearly eligibility changes. The language pattern should not become a rule for real students. |
| Assessments (2.5): benchmarks cover Reading and Mathematics in K-8, with Fall, Winter, and Spring results. State ELA/Math cover grades 3-8 on a different scale. | What do Fall and Spring benchmarks show across all elementary grades? | Starting points help the LD interpret Spring results. Reading and Mathematics need separate comparisons. |
| Assessments (2.5): districtwide Fall and Spring benchmark counts differ: 11,571 and 11,427. Each student-year-subject-window combination is unique. | Who has both tests, and who is left out of paired comparisons? | These are K-8 test records, not unique elementary students. Participation needs its own review; counts alone do not explain missing results. |
| Attendance and calendar (2.4, 2.9): expected enrolled-day coverage is complete. Calendars have 188, 186, and 185 instructional days; tardy counts as present. Enrollment and test dates are valid. | Where is absence concentrated? How does absence between tests relate to Spring results? | Rates account for calendar length and time enrolled. Valid dates let me measure attendance before the Spring test. |
| Staffing (2.7-2.8): current teacher records do not give historical assignments. Fourteen employed teacher-year periods lack assignments. | Is there a staffing question worth checking, and is the record complete enough? | Differences in recorded FTE or role labels need verification before conclusions about class size, services, or shortages. |

These observations led to the three questions below. QC checks alone do not measure school performance.


### 3.2 Three questions for the briefing

| Question | What the EDA covers | What remains to answer | Use for the LD |
|---|---|---|---|
| Q1. Who do the four schools serve? | Enrollment, grade mix, student needs, starting scores, and attendance | Which background differences matter when interpreting results? | Prepare for discussions with each principal |
| Q2. How do Fall starting points and Spring Reading/Math results differ? | Participation, paired scores and changes, and benchmark status by school, grade, subject, and year | Do Spring school differences remain when Fall scores and relevant context are taken into account? | Interpret Spring results alongside students' starting points |
| Q3. Where is absence concentrated, and how does it relate to learning? | Absence distributions and yearly patterns; between-test absence alongside Fall, Spring, and score change | Is absence still related to Spring scores after accounting for starting points and context? | Consider whether attendance and academic support should be discussed together |

Q1 establishes the school context. Q2 examines learning within that context. Q3 asks what attendance adds. Section 4 addresses these questions about adjusted differences and associations.

I also reviewed state tests, discipline, and staffing to check whether another finding deserves space in the short briefing. Final slide choices will follow the evidence.

Section 3.10 deepens Q2 by following the same students across reported levels, and Q3 by counting overlapping needs once per student. These provide the main support-planning findings; the models help interpret school and attendance comparisons.


### 3.3 Work completed

| Work | Output | Location |
|---|---|---|
| Define the population and measures | Elementary scope, denominators, and estimated testing opportunities | 3.4A |
| Prepare analysis tables | One row per student-year for context; one per student-year-subject for paired tests | 3.4B-C |
| Check sample coverage | Participation, paired tests, and included/excluded comparisons | 3.4D |
| Describe school context | Enrollment, needs, movement during the year, and annual attendance | EDA 1-2 |
| Describe learning | Fall and Spring results, observed changes, and sample counts | EDA 3-4 |
| Examine attendance with learning | School-grade patterns, days enrolled, correlations, and absence bands | EDA 5-6 |
| Review other priorities | State tests, discipline, and recorded staffing | 3.8 |
| Follow student paths and overlapping needs | Status movement, score change, and historical K-4 groups to verify | 3.10 |
| Summarize findings | Results, limits, and questions for models | 3.9 |

Section 4 compares results after accounting for starting points and relevant background. The current comparisons describe patterns; they do not estimate the effect of a school or intervention.


### 3.4 Prepare the analysis tables

### 3.4A Population and measure definitions

Data quality checks cover the full seven-school district. The primary briefing analysis focuses on schools 101–104 (K–5), because these are the four schools the incoming Level Director will supervise.

The analysis covers elementary schools 101-104, K-5, in 2023-24 to 2025-26 (`school_year` 2024-2026). Latest-year results lead the briefing; earlier years provide context.

School profiles count anyone enrolled during the year. Attendance rates use the instructional days within each student's enrollment dates. A school absence rate is total absent days divided by total enrolled instructional days; it is different from the average of individual student rates.

Official testing windows were not supplied. I estimate opportunity from overlap between enrollment dates and the recorded test-date range for the same school, year, grade, subject, and window. Students entering after Fall testing or leaving before Spring testing may not have had both opportunities.

Benchmark scores, benchmark status, state proficiency, and national percentiles remain separate. Benchmarks use `performance_level` for status because `met_standard` is intentionally blank. The recorded grades are historical; former grade 5 students may have moved to middle school by the coming fall.

| Measure | Who is counted | Calculation | Dates and exclusions |
|---|---|---|---|
| Enrollment and student needs | All elementary student-year enrollments | Enrollment count; group count / all enrolled records for each school-year | All three years; no exclusions for school profiles |
| Annual absence | All elementary enrollments with daily coverage | Absent days / enrolled instructional days | All three years; partial-year students stay included, using their actual days enrolled |
| Benchmark participation | Students with estimated opportunity for each subject and window | Observed results / estimated opportunities | Fall and Spring, all three years; students without estimated opportunity are reported separately. The estimate is not official eligibility. |
| Fall-Spring pairs | Student-year-subject records with both tests | Complete pairs / records with both estimated opportunities for the pairing rate | Reading and Mathematics separately; incomplete records remain in participation summaries but not score-change analysis |
| Between-test absence | Complete pairs with valid dates and attendance coverage | Absent days / enrolled instructional days strictly between tests | Exclude both test days. Missing dates or zero-day intervals cannot produce a valid rate; undefined rates stay blank. |


In [ ]:
# Define the analysis population

# Main scope for the Level Director briefing
ELEMENTARY_SCHOOL_IDS = [101, 102, 103, 104]
ANALYSIS_YEARS = [2024, 2025, 2026]

# Elementary student-year enrollment population
elem_enrollments = enrollments.loc[
    enrollments["school_id"].isin(ELEMENTARY_SCHOOL_IDS)
    & enrollments["school_year"].isin(ANALYSIS_YEARS)
    & enrollments["grade_level"].between(0, 5)
].copy()

# Verify the intended scope
pd.Series(
    {
        "student-year records": len(elem_enrollments),
        "unique students": elem_enrollments["student_id"].nunique(),
        "schools": elem_enrollments["school_id"].nunique(),
        "school years": elem_enrollments["school_year"].nunique(),
        "duplicate student-year records": elem_enrollments.duplicated(
            ["student_id", "school_year"]
        ).sum(),
    }
)


In [ ]:
# Student-year counts by school and year
population_check = (
    elem_enrollments.groupby(["school_year", "school_id"])
    .agg(
        student_years=("student_id", "size"), unique_students=("student_id", "nunique")
    )
    .reset_index()
)

display(population_check)


In [ ]:
# Scope validation
assert elem_enrollments["school_id"].isin(ELEMENTARY_SCHOOL_IDS).all()

assert elem_enrollments["school_year"].isin(ANALYSIS_YEARS).all()

assert elem_enrollments["grade_level"].between(0, 5).all()

assert elem_enrollments.duplicated(["student_id", "school_year"]).sum() == 0


#### Population check

- There are 4,141 elementary student-year records across the four schools and three years, representing 1,897 unique students.
- Each student-year combination is unique. Within each school-year, record counts equal distinct student counts.
- Total elementary enrollment is 1,364, 1,376, and 1,401 in the three years. Q1 examines these changes by school, alongside grade mix and student needs.


#### 3.4B Student-year context table

I start with one row per elementary student-year. The table keeps school, grade, entry and exit dates, and selected student characteristics. Student and school reference joins are checked for unique matches.

Daily attendance is summarized before the join. This table is used for school profiles and annual attendance, including students who do not have both tests.


In [ ]:
# Add student characteristics and school names; keep one row per student-year.

elem_context = elem_enrollments.merge(
    students[
        [
            "student_id",
            "gender",
            "race_ethnicity",
            "english_learner_flag",
            "economically_disadvantaged_flag",
            "special_education_flag",
        ]
    ],
    on="student_id",
    how="left",
    validate="many_to_one",
).merge(
    schools[["school_id", "school_name", "title_i"]],
    on="school_id",
    how="left",
    validate="many_to_one",
)


In [ ]:
# Keep daily attendance for the elementary enrollment population.

elem_attendance = attendance[
    attendance["enrollment_id"].isin(elem_enrollments["enrollment_id"])
].copy()


In [ ]:
# Summarize daily attendance to one row per enrollment before joining.

annual_attendance = (
    elem_attendance.groupby("enrollment_id")
    .agg(
        # Total enrolled instructional days
        enrolled_instructional_days=("attendance_date", "size"),
        # Present + Tardy days
        attended_days=("days_attended", "sum"),
        # Total absent days
        absent_days=("is_absent", "sum"),
        # Tardy days
        tardy_days=("attendance_code", lambda x: (x == "T").sum()),
        # Excused absence days
        excused_absent_days=("attendance_code", lambda x: (x == "AE").sum()),
        # Unexcused absence days
        unexcused_absent_days=("attendance_code", lambda x: (x == "AU").sum()),
    )
    .reset_index()
)


In [ ]:
# Calculate student-year attendance rates

# Overall absence rate
annual_attendance["absence_rate"] = (
    annual_attendance["absent_days"] / annual_attendance["enrolled_instructional_days"]
)

# Overall attendance rate
annual_attendance["attendance_rate"] = (
    annual_attendance["attended_days"]
    / annual_attendance["enrolled_instructional_days"]
)

# Tardy rate
annual_attendance["tardy_rate"] = (
    annual_attendance["tardy_days"] / annual_attendance["enrolled_instructional_days"]
)

# Unexcused absence rate
annual_attendance["unexcused_absence_rate"] = (
    annual_attendance["unexcused_absent_days"]
    / annual_attendance["enrolled_instructional_days"]
)


In [ ]:
# Join the annual summary once per enrollment.

elem_context = elem_context.merge(
    annual_attendance, on="enrollment_id", how="left", validate="one_to_one"
)


In [ ]:
# Check row counts, unique student-years, and complete reference joins.

pd.Series(
    {
        "final rows": len(elem_context),
        "duplicate student-year rows": elem_context.duplicated(
            ["student_id", "school_year"]
        ).sum(),
        "missing attendance summaries": elem_context["enrolled_instructional_days"]
        .isna()
        .sum(),
        "missing school names": elem_context["school_name"].isna().sum(),
        "missing EL flags": elem_context["english_learner_flag"].isna().sum(),
    }
)


In [ ]:
# Preview the final student-year context table
display(
    elem_context[
        [
            "student_id",
            "school_year",
            "school_name",
            "grade_level",
            "entry_date",
            "exit_date",
            "english_learner_flag",
            "economically_disadvantaged_flag",
            "special_education_flag",
            "enrolled_instructional_days",
            "absent_days",
            "absence_rate",
        ]
    ].head()
)


In [ ]:
# Review the amount of attendance exposure per student-year
elem_context["enrolled_instructional_days"].describe()


#### Context table check

- `elem_context` retains all 4,141 student-year records, with no duplicate student-year combinations.
- Selected student characteristics and school information join completely.
- Attendance is summarized by student-year before joining. Every record has an annual attendance summary.
- Partial-year students remain included. Their rates use their own enrolled instructional days, rather than a full-year denominator.


#### 3.4C Fall-Spring tests and attendance

I create one row per student-year-subject for Reading and Mathematics, keeping scores, test dates, performance labels, school, and grade. Incomplete pairs remain visible for the participation review. Winter results are available as context; the main analysis requires Fall and Spring only.

For complete pairs, I count attendance on enrolled instructional dates after the Fall test and before the Spring test:

`Fall test date < attendance date < Spring test date`

The table stores absent days, eligible days, and their ratio. A zero-day interval has no defined absence rate, so it stays blank. Reading and Mathematics may have different test dates and attendance intervals. Uniqueness and row-count checks protect the one-row-per-student-year-subject structure.


In [ ]:
# Give each student-year a Reading row and a Mathematics row.

benchmark_subjects = pd.DataFrame({"subject": ["Reading", "Mathematics"]})

benchmark_pairs = elem_context[
    [
        "enrollment_id",
        "student_id",
        "school_id",
        "school_name",
        "school_year",
        "school_year_label",
        "grade_level",
        "entry_date",
        "exit_date",
        "english_learner_flag",
        "economically_disadvantaged_flag",
        "special_education_flag",
    ]
].merge(benchmark_subjects, how="cross")


In [ ]:
# Select elementary Fall and Spring benchmarks in both subjects.

elem_benchmark = assessment.loc[
    assessment["assessment_name"].eq("Interim Benchmark")
    & assessment["school_id"].isin(ELEMENTARY_SCHOOL_IDS)
    & assessment["school_year"].isin(ANALYSIS_YEARS)
    & assessment["subject"].isin(["Reading", "Mathematics"])
    & assessment["test_window"].isin(["Fall", "Spring"])
].copy()


In [ ]:
# Prepare Fall benchmark results

fall_benchmark = elem_benchmark.loc[
    elem_benchmark["test_window"].eq("Fall"),
    [
        "student_id",
        "school_year",
        "subject",
        "test_date",
        "scale_score",
        "performance_level",
    ],
].rename(
    columns={
        "test_date": "fall_test_date",
        "scale_score": "fall_score",
        "performance_level": "fall_performance_level",
    }
)

# Confirm one Fall result per student-year-subject
assert fall_benchmark.duplicated(["student_id", "school_year", "subject"]).sum() == 0


In [ ]:
# Prepare Spring benchmark results

spring_benchmark = elem_benchmark.loc[
    elem_benchmark["test_window"].eq("Spring"),
    [
        "student_id",
        "school_year",
        "subject",
        "test_date",
        "scale_score",
        "performance_level",
    ],
].rename(
    columns={
        "test_date": "spring_test_date",
        "scale_score": "spring_score",
        "performance_level": "spring_performance_level",
    }
)

# Confirm one Spring result per student-year-subject
assert spring_benchmark.duplicated(["student_id", "school_year", "subject"]).sum() == 0


In [ ]:
# Join Fall and Spring results

benchmark_pairs = benchmark_pairs.merge(
    fall_benchmark,
    on=["student_id", "school_year", "subject"],
    how="left",
    validate="one_to_one",
).merge(
    spring_benchmark,
    on=["student_id", "school_year", "subject"],
    how="left",
    validate="one_to_one",
)


In [ ]:
# Identify complete Fall-Spring pairs

benchmark_pairs["has_fall"] = benchmark_pairs["fall_score"].notna()
benchmark_pairs["has_spring"] = benchmark_pairs["spring_score"].notna()
benchmark_pairs["has_fall_spring_pair"] = (
    benchmark_pairs["has_fall"] & benchmark_pairs["has_spring"]
)


In [ ]:
# Review Fall-Spring pair counts

benchmark_pairs.groupby("subject").agg(
    total_student_years=("student_id", "size"),
    fall_results=("has_fall", "sum"),
    spring_results=("has_spring", "sum"),
    complete_pairs=("has_fall_spring_pair", "sum"),
)


In [ ]:
# Keep pairs with both tests; these supply dates for the attendance interval.

complete_pairs = benchmark_pairs.loc[
    benchmark_pairs["has_fall_spring_pair"],
    [
        "enrollment_id",
        "student_id",
        "school_year",
        "subject",
        "fall_test_date",
        "spring_test_date",
    ],
].copy()

# Spring test should always occur after Fall test.
pd.Series(
    {
        "complete pairs": len(complete_pairs),
        "Spring on or before Fall": (
            complete_pairs["spring_test_date"] <= complete_pairs["fall_test_date"]
        ).sum(),
    }
)


In [ ]:
# Attach daily attendance to each subject pair.
# Multiple subjects and days intentionally create a temporary many-to-many join.

between_test_daily = complete_pairs.merge(
    elem_attendance[["enrollment_id", "attendance_date", "is_absent"]],
    on="enrollment_id",
    how="left",
    validate="many_to_many",
)


In [ ]:
# Keep enrolled instructional days between tests, excluding both test days.

between_test_daily = between_test_daily.loc[
    (between_test_daily["attendance_date"] > between_test_daily["fall_test_date"])
    & (between_test_daily["attendance_date"] < between_test_daily["spring_test_date"])
].copy()


In [ ]:
# Confirm one row per enrollment-year-subject pair
pd.Series(
    {
        "complete pair rows": len(complete_pairs),
        "duplicate enrollment-subject rows": complete_pairs.duplicated(
            ["enrollment_id", "subject"]
        ).sum(),
    }
)
pd.Series(
    {
        "duplicate enrollment-date rows": elem_attendance.duplicated(
            ["enrollment_id", "attendance_date"]
        ).sum()
    }
)


In [ ]:
# Summarize the daily expansion back to one row per student-year-subject.

between_test_attendance = (
    between_test_daily.groupby(
        ["enrollment_id", "student_id", "school_year", "subject"]
    )
    .agg(
        # Number of enrolled instructional days
        # Strictly between Fall and Spring tests
        between_test_eligible_days=("attendance_date", "size"),
        # Number of absent instructional days
        # Strictly between Fall and Spring tests
        between_test_absent_days=("is_absent", "sum"),
    )
    .reset_index()
)


In [ ]:
# Divide absent days by eligible days. Undefined rates stay blank.

between_test_attendance["between_test_absence_rate"] = (
    between_test_attendance["between_test_absent_days"]
    / between_test_attendance["between_test_eligible_days"]
)


In [ ]:
# Join one attendance interval summary to each subject pair.

benchmark_pairs = benchmark_pairs.merge(
    between_test_attendance,
    on=["enrollment_id", "student_id", "school_year", "subject"],
    how="left",
    validate="one_to_one",
)


In [ ]:
# Validate the final Fall-Spring table

pd.Series(
    {
        "final student-year-subject rows": len(benchmark_pairs),
        "expected student-year-subject rows": len(elem_context) * 2,
        "duplicate student-year-subject rows": benchmark_pairs.duplicated(
            ["student_id", "school_year", "subject"]
        ).sum(),
        "complete Fall-Spring pairs": benchmark_pairs["has_fall_spring_pair"].sum(),
        "complete pairs missing between-test attendance": (
            benchmark_pairs["has_fall_spring_pair"]
            & benchmark_pairs["between_test_eligible_days"].isna()
        ).sum(),
        "complete pairs with zero eligible days": (
            benchmark_pairs["has_fall_spring_pair"]
            & benchmark_pairs["between_test_eligible_days"].eq(0)
        ).sum(),
    }
)


#### What the paired table contains

- The table retains 4,141 student-year records for each of two subjects: 8,282 rows with unique keys. Missing tests remain visible.
- Reading has 3,653 complete pairs and Mathematics 3,606, totaling 7,259. They represent 1,780 and 1,779 unique students respectively. The subject groups overlap, so those unique-student counts cannot be added.
- Attendance uses each student's subject-specific test dates and excludes both test days. Median intervals are 140 instructional days for Reading and 139 for Mathematics across the three years.
- All complete pairs have valid date order, positive intervals, and attendance summaries. No duplicate keys, missing attendance summaries, or zero-day intervals were found.
- These pairs cover students with both recorded tests. The next section checks how they differ from the full enrolled population.


#### 3.4D Participation and sample coverage

For each subject, I report enrollment, estimated Fall and Spring opportunities, results in each window, complete pairs, and usable attendance intervals. Fall and Spring are parallel checks; having a Fall result is not a condition for reporting a Spring result.

I separate records without both estimated opportunities from those with an expected result missing. Unknown opportunities would be labeled separately. I compare included and excluded records by school, grade, student characteristics, days enrolled, and annual attendance. Counts and proportions are sufficient for this sample description.


In [ ]:
# Estimate testing windows from recorded dates in each school-year-grade-subject.
# Official windows were not supplied.

testing_window_proxy = (
    elem_benchmark.merge(
        elem_enrollments[["student_id", "school_year", "school_id", "grade_level"]],
        on=["student_id", "school_year", "school_id"],
        how="left",
        validate="many_to_one",
    )
    .groupby(["school_year", "school_id", "grade_level", "subject", "test_window"])
    .agg(
        proxy_window_start=("test_date", "min"),
        proxy_window_end=("test_date", "max"),
        observed_results=("student_id", "size"),
    )
    .reset_index()
)

display(testing_window_proxy.head(20))


In [ ]:
# Prepare Fall and Spring opportunity windows

fall_window_proxy = testing_window_proxy.loc[
    testing_window_proxy["test_window"].eq("Fall"),
    [
        "school_year",
        "school_id",
        "grade_level",
        "subject",
        "proxy_window_start",
        "proxy_window_end",
    ],
].rename(
    columns={
        "proxy_window_start": "fall_proxy_start",
        "proxy_window_end": "fall_proxy_end",
    }
)

spring_window_proxy = testing_window_proxy.loc[
    testing_window_proxy["test_window"].eq("Spring"),
    [
        "school_year",
        "school_id",
        "grade_level",
        "subject",
        "proxy_window_start",
        "proxy_window_end",
    ],
].rename(
    columns={
        "proxy_window_start": "spring_proxy_start",
        "proxy_window_end": "spring_proxy_end",
    }
)


In [ ]:
# Build the participation table

participation = benchmark_pairs.merge(
    elem_context[["enrollment_id", "enrolled_instructional_days", "absence_rate"]],
    on="enrollment_id",
    how="left",
    validate="many_to_one",
)


In [ ]:
# Attach Fall and Spring proxy windows

participation = participation.merge(
    fall_window_proxy,
    on=["school_year", "school_id", "grade_level", "subject"],
    how="left",
    validate="many_to_one",
).merge(
    spring_window_proxy,
    on=["school_year", "school_id", "grade_level", "subject"],
    how="left",
    validate="many_to_one",
)


In [ ]:
# Define Fall testing opportunity

participation["fall_opportunity"] = pd.Series(
    pd.NA, index=participation.index, dtype="boolean"
)

fall_proxy_known = (
    participation["fall_proxy_start"].notna() & participation["fall_proxy_end"].notna()
)

participation.loc[fall_proxy_known, "fall_opportunity"] = (
    participation.loc[fall_proxy_known, "entry_date"]
    <= participation.loc[fall_proxy_known, "fall_proxy_end"]
) & (
    participation.loc[fall_proxy_known, "exit_date"]
    >= participation.loc[fall_proxy_known, "fall_proxy_start"]
)


In [ ]:
# Define Spring testing opportunity

participation["spring_opportunity"] = pd.Series(
    pd.NA, index=participation.index, dtype="boolean"
)

spring_proxy_known = (
    participation["spring_proxy_start"].notna()
    & participation["spring_proxy_end"].notna()
)

participation.loc[spring_proxy_known, "spring_opportunity"] = (
    participation.loc[spring_proxy_known, "entry_date"]
    <= participation.loc[spring_proxy_known, "spring_proxy_end"]
) & (
    participation.loc[spring_proxy_known, "exit_date"]
    >= participation.loc[spring_proxy_known, "spring_proxy_start"]
)


In [ ]:
# Define opportunity for both windows

participation["both_opportunities"] = (
    participation["fall_opportunity"] & participation["spring_opportunity"]
)


In [ ]:
# Validate the opportunity proxy

pd.Series(
    {
        "Fall results classified as no Fall opportunity": (
            participation["has_fall"] & participation["fall_opportunity"].eq(False)
        ).sum(),
        "Spring results classified as no Spring opportunity": (
            participation["has_spring"] & participation["spring_opportunity"].eq(False)
        ).sum(),
        "Fall opportunity uncertain": participation["fall_opportunity"].isna().sum(),
        "Spring opportunity uncertain": participation["spring_opportunity"]
        .isna()
        .sum(),
    }
)


In [ ]:
# Define usable paired-analysis records

participation["usable_between_test_interval"] = participation[
    "has_fall_spring_pair"
] & participation["between_test_eligible_days"].gt(0)

participation["included_in_paired_analysis"] = participation[
    "usable_between_test_interval"
]


In [ ]:
# Build the sample flow by subject

sample_flow = participation.groupby("subject").agg(
    enrolled_population=("student_id", "size"),
    fall_opportunity=("fall_opportunity", lambda x: x.eq(True).sum()),
    spring_opportunity=("spring_opportunity", lambda x: x.eq(True).sum()),
    both_opportunities=("both_opportunities", lambda x: x.eq(True).sum()),
    fall_results=("has_fall", "sum"),
    spring_results=("has_spring", "sum"),
    both_results=("has_fall_spring_pair", "sum"),
    usable_between_test_interval=("usable_between_test_interval", "sum"),
    fall_opportunity_uncertain=("fall_opportunity", lambda x: x.isna().sum()),
    spring_opportunity_uncertain=("spring_opportunity", lambda x: x.isna().sum()),
)


In [ ]:
# Add participation rates

sample_flow["fall_participation_rate"] = (
    sample_flow["fall_results"] / sample_flow["fall_opportunity"]
)

sample_flow["spring_participation_rate"] = (
    sample_flow["spring_results"] / sample_flow["spring_opportunity"]
)

sample_flow["paired_rate"] = (
    sample_flow["both_results"] / sample_flow["both_opportunities"]
)

sample_flow["usable_interval_rate"] = (
    sample_flow["usable_between_test_interval"] / sample_flow["both_results"]
)

display(sample_flow)


In [ ]:
# Classify why records do or do not enter the paired analysis

participation["paired_analysis_status"] = "Other / review"

# Opportunity cannot be established
uncertain_opportunity = (
    participation["fall_opportunity"].isna()
    | participation["spring_opportunity"].isna()
)

participation.loc[uncertain_opportunity, "paired_analysis_status"] = (
    "Opportunity uncertain"
)

# Student did not have both testing opportunities
no_both_opportunity = ~uncertain_opportunity & ~participation["both_opportunities"].eq(
    True
)

participation.loc[no_both_opportunity, "paired_analysis_status"] = (
    "Did not have both opportunities"
)

# Student had both opportunities but is missing Fall and/or Spring result
expected_result_missing = (
    participation["both_opportunities"].eq(True)
    & ~participation["has_fall_spring_pair"]
)

participation.loc[expected_result_missing, "paired_analysis_status"] = (
    "Expected result missing"
)

# Complete tests but unusable attendance interval
unusable_interval = (
    participation["has_fall_spring_pair"]
    & ~participation["usable_between_test_interval"]
)

participation.loc[unusable_interval, "paired_analysis_status"] = (
    "Complete pair but unusable interval"
)

# Final paired-analysis sample
participation.loc[
    participation["included_in_paired_analysis"], "paired_analysis_status"
] = "Included in paired analysis"


In [ ]:
# Status counts and proportions by subject

pair_status_summary = (
    participation.groupby(["subject", "paired_analysis_status"])
    .size()
    .rename("n")
    .reset_index()
)

pair_status_summary["proportion"] = pair_status_summary[
    "n"
] / pair_status_summary.groupby("subject")["n"].transform("sum")

display(pair_status_summary)


In [ ]:
# Compare included and excluded records

included_excluded_overall = (
    participation.groupby(["subject", "included_in_paired_analysis"])
    .agg(
        n=("student_id", "size"),
        mean_enrolled_instructional_days=("enrolled_instructional_days", "mean"),
        median_enrolled_instructional_days=("enrolled_instructional_days", "median"),
        mean_annual_absence_rate=("absence_rate", "mean"),
        median_annual_absence_rate=("absence_rate", "median"),
    )
    .reset_index()
)

display(included_excluded_overall)


In [ ]:
# Inclusion by school

school_inclusion = (
    participation.groupby(["subject", "school_name"])
    .agg(n=("student_id", "size"), included=("included_in_paired_analysis", "sum"))
    .reset_index()
)

school_inclusion["included_rate"] = school_inclusion["included"] / school_inclusion["n"]

display(school_inclusion)


In [ ]:
# Inclusion by grade

grade_inclusion = (
    participation.groupby(["subject", "grade_level"])
    .agg(n=("student_id", "size"), included=("included_in_paired_analysis", "sum"))
    .reset_index()
)

grade_inclusion["included_rate"] = grade_inclusion["included"] / grade_inclusion["n"]

display(grade_inclusion)


In [ ]:
# Inclusion by student characteristics

for variable in [
    "english_learner_flag",
    "economically_disadvantaged_flag",
    "special_education_flag",
]:

    summary = (
        participation.groupby(["subject", variable])
        .agg(n=("student_id", "size"), included=("included_in_paired_analysis", "sum"))
        .reset_index()
    )

    summary["included_rate"] = summary["included"] / summary["n"]

    print(f"\n{variable}")
    display(summary)


#### What participation shows

A student has an estimated opportunity when enrollment overlaps the recorded testing dates for their school, year, grade, subject, and window. Every observed test passes this consistency check. Official dates are unavailable, so the estimate remains an approximation of opportunity.

| Coverage, 2023-24 to 2025-26 | Reading | Mathematics |
|---|---:|---:|
| Enrolled student-year records per subject | 4,141 | 4,141 |
| Fall results / estimated Fall opportunities | 3,905 / 4,069 (96.0%) | 3,878 / 4,071 (95.3%) |
| Spring results / estimated Spring opportunities | 3,871 / 4,036 (95.9%) | 3,854 / 4,036 (95.5%) |
| Complete pairs / estimated opportunities for both tests | 3,653 / 3,964 (92.2%) | 3,606 / 3,966 (90.9%) |
| Complete pairs / all enrolled records | 3,653 / 4,141 (88.2%) | 3,606 / 4,141 (87.1%) |

- Reading excludes 177 records without both estimated opportunities and 311 with an expected result missing. Mathematics excludes 175 and 360 respectively. Missing results do not tell us why a test was missed.
- Paired records average about 186 enrolled instructional days. Excluded records average 154 days in Reading and 157 in Mathematics. Their median is still 185 days, so not all exclusions involve short enrollment.
- Included and excluded records have mean annual student absence rates of 5.84% and 6.17% in Reading, and 5.90% and 5.77% in Mathematics. Excluded students do not consistently have higher absence.
- School inclusion rates range from 85.5% to 90.3% for Reading and 85.4% to 88.4% for Mathematics across three years. School and subgroup counts help interpret coverage differences.
- Paired findings apply most directly to students with both tests, who had longer enrollment on average. School profiles retain everyone enrolled; Spring summaries can use everyone with a Spring test.


### 3.5 Q1: Who do the schools serve?

QC confirmed a unique student-year enrollment table and showed that time enrolled varies. The student file supplies background labels. I use these to describe each school before comparing learning results.

For the LD, school size, grade mix, and student needs help frame discussions about support and make school averages easier to interpret.

#### EDA 1: Enrollment and student context

1. Count students by school and year, and calculate the three-year change.
2. Compare grade proportions, including Kindergarten.
3. Compare EL, economic disadvantage, and SPED using the same enrolled population.
4. Summarize days enrolled and midyear entries and exits.

The outputs are a latest-year school profile and three-year enrollment comparison. Displays use K for Kindergarten. Student labels describe the supplied file, not changes in individual program eligibility.


In [ ]:
# Count students by school and year

enrollment_trend = (
    elem_context.groupby(["school_year", "school_year_label", "school_name"])
    .agg(enrolled_students=("student_id", "size"))
    .reset_index()
)

display(enrollment_trend)


In [ ]:
# Show the three-year enrollment trend

enrollment_trend_wide = enrollment_trend.pivot(
    index="school_name", columns="school_year_label", values="enrolled_students"
)

display(enrollment_trend_wide)


In [ ]:
# Calculate enrollment change

enrollment_change = enrollment_trend_wide.assign(
    three_year_change=lambda x: x["2025-26"] - x["2023-24"],
    three_year_change_pct=lambda x: (x["2025-26"] - x["2023-24"]) / x["2023-24"],
)

display(enrollment_change)


In [ ]:
# Keep the latest school year

latest_year = elem_context.loc[elem_context["school_year"].eq(2026)].copy()

latest_year["grade_display"] = latest_year["grade_level"].replace(
    {0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
)


In [ ]:
# Compare grade composition

grade_composition = (
    latest_year.groupby(["school_name", "grade_display"])
    .agg(students=("student_id", "size"))
    .reset_index()
)

grade_composition["school_total"] = grade_composition.groupby("school_name")[
    "students"
].transform("sum")

grade_composition["grade_proportion"] = (
    grade_composition["students"] / grade_composition["school_total"]
)

display(grade_composition)


In [ ]:
# Compare recorded student needs

student_needs_profile = (
    latest_year.groupby("school_name")
    .agg(
        enrolled_students=("student_id", "size"),
        english_learners=("english_learner_flag", "sum"),
        economically_disadvantaged=("economically_disadvantaged_flag", "sum"),
        special_education=("special_education_flag", "sum"),
        english_learner_pct=("english_learner_flag", "mean"),
        economically_disadvantaged_pct=("economically_disadvantaged_flag", "mean"),
        special_education_pct=("special_education_flag", "mean"),
    )
    .reset_index()
)

display(student_needs_profile)


In [ ]:
# Summarize enrollment exposure

enrollment_exposure = (
    latest_year.groupby("school_name")
    .agg(
        students=("student_id", "size"),
        mean_enrolled_days=("enrolled_instructional_days", "mean"),
        median_enrolled_days=("enrolled_instructional_days", "median"),
        min_enrolled_days=("enrolled_instructional_days", "min"),
    )
    .reset_index()
)

display(enrollment_exposure)


In [ ]:
# Define instructional-year boundaries

instructional_year_bounds = (
    calendar.loc[calendar["is_instructional_day"]]
    .groupby("school_year")
    .agg(
        first_instructional_day=("calendar_date", "min"),
        last_instructional_day=("calendar_date", "max"),
    )
    .reset_index()
)


In [ ]:
# Identify midyear entries and exits

latest_year_flow = latest_year.merge(
    instructional_year_bounds, on="school_year", how="left", validate="many_to_one"
)

latest_year_flow["midyear_entry"] = (
    latest_year_flow["entry_date"] > latest_year_flow["first_instructional_day"]
)

latest_year_flow["midyear_exit"] = (
    latest_year_flow["exit_date"] < latest_year_flow["last_instructional_day"]
)


In [ ]:
# Summarize student mobility

student_mobility = (
    latest_year_flow.groupby("school_name")
    .agg(
        students=("student_id", "size"),
        midyear_entries=("midyear_entry", "sum"),
        midyear_exits=("midyear_exit", "sum"),
        midyear_entry_pct=("midyear_entry", "mean"),
        midyear_exit_pct=("midyear_exit", "mean"),
    )
    .reset_index()
)

display(student_mobility)


In [ ]:
# Build the latest-year school profile

latest_school_profile = student_needs_profile.merge(
    enrollment_exposure,
    on="school_name",
    how="left",
    suffixes=("", "_exposure"),
    validate="one_to_one",
).merge(
    student_mobility,
    on="school_name",
    how="left",
    suffixes=("", "_mobility"),
    validate="one_to_one",
)

latest_school_profile = latest_school_profile[
    [
        "school_name",
        "enrolled_students",
        "english_learner_pct",
        "economically_disadvantaged_pct",
        "special_education_pct",
        "mean_enrolled_days",
        "midyear_entry_pct",
        "midyear_exit_pct",
    ]
]

display(latest_school_profile)


In [ ]:
# Plot three-year enrollment trends


fig, ax = plt.subplots(figsize=(9, 5))

for school, group in enrollment_trend.groupby("school_name"):
    group = group.sort_values("school_year")

    ax.plot(
        group["school_year_label"], group["enrolled_students"], marker="o", label=school
    )

ax.set_title("Elementary Enrollment Trends")
ax.set_xlabel("School Year")
ax.set_ylabel("Students Enrolled During the Year")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


#### Findings from EDA 1

- Enrollment moved in different directions. Cedar Ridge grew from 322 to 351 (+9.0%) and Lincoln Park from 342 to 376 (+9.9%). Harper Creek fell from 358 to 349 (-2.5%) and Meadowbrook from 342 to 325 (-5.0%). Lincoln Park is the largest school in 2025-26; Meadowbrook is the smallest.
- Grade mix and recorded needs differ. Lincoln Park has the largest EL proportion, Cedar Ridge the largest economic-disadvantage proportion, and Meadowbrook the largest SPED proportion.
- Median days enrolled are similar, but Cedar Ridge has fewer days enrolled on average and the highest midyear entry and exit rates. Participation and attendance comparisons need to account for partial-year enrollment.
- I compare academic results within grade and subject. Schoolwide averages also reflect grade mix, student background, and time enrolled.


#### EDA 2: Attendance overview

1. Compare school absence rates across three years.
2. Examine individual student rates and the days enrolled behind them.
3. Identify broad grade patterns for closer review in Q3.

The outputs show yearly rates and the spread of student rates. I also describe students missing at least 10% of enrolled days. This is an exploratory cutoff because the task supplies no official chronic-absence definition; it supplements the full rate distribution.


In [ ]:
# Calculate aggregate absence rates
# By school and year

school_attendance_trend = (
    elem_context.groupby(["school_year", "school_year_label", "school_name"])
    .agg(
        students=("student_id", "size"),
        total_absent_days=("absent_days", "sum"),
        total_enrolled_days=("enrolled_instructional_days", "sum"),
    )
    .reset_index()
)

school_attendance_trend["aggregate_absence_rate"] = (
    school_attendance_trend["total_absent_days"]
    / school_attendance_trend["total_enrolled_days"]
)

display(school_attendance_trend)


In [ ]:
# Plot three-year absence-rate trends


fig, ax = plt.subplots(figsize=(9, 5))

for school, group in school_attendance_trend.groupby("school_name"):
    group = group.sort_values("school_year")

    ax.plot(
        group["school_year_label"],
        group["aggregate_absence_rate"] * 100,
        marker="o",
        label=school,
    )

ax.set_title("Elementary Absence Rate Trends")
ax.set_xlabel("School Year")
ax.set_ylabel("Aggregate Absence Rate (%)")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# Summarize student absence-rate distributions

student_absence_distribution = (
    elem_context.groupby(["school_year", "school_year_label", "school_name"])
    .agg(
        students=("student_id", "size"),
        mean_absence_rate=("absence_rate", "mean"),
        median_absence_rate=("absence_rate", "median"),
        p25_absence_rate=("absence_rate", lambda x: x.quantile(0.25)),
        p75_absence_rate=("absence_rate", lambda x: x.quantile(0.75)),
        p90_absence_rate=("absence_rate", lambda x: x.quantile(0.90)),
        median_enrolled_days=("enrolled_instructional_days", "median"),
        min_enrolled_days=("enrolled_instructional_days", "min"),
    )
    .reset_index()
)

display(student_absence_distribution)


In [ ]:
# Build latest-year attendance summary

latest_attendance_summary = (
    elem_context.loc[elem_context["school_year"].eq(2026)]
    .groupby("school_name")
    .agg(
        students=("student_id", "size"),
        total_absent_days=("absent_days", "sum"),
        total_enrolled_days=("enrolled_instructional_days", "sum"),
        median_student_absence_rate=("absence_rate", "median"),
        p75_student_absence_rate=("absence_rate", lambda x: x.quantile(0.75)),
        p90_student_absence_rate=("absence_rate", lambda x: x.quantile(0.90)),
        median_enrolled_days=("enrolled_instructional_days", "median"),
        min_enrolled_days=("enrolled_instructional_days", "min"),
    )
    .reset_index()
)

latest_attendance_summary["aggregate_absence_rate"] = (
    latest_attendance_summary["total_absent_days"]
    / latest_attendance_summary["total_enrolled_days"]
)

display(latest_attendance_summary)


In [ ]:
# Explore students with >=10% absence

elem_context["absence_10pct_plus"] = elem_context["absence_rate"] >= 0.10

absence_threshold_summary = (
    elem_context.groupby(["school_year", "school_year_label", "school_name"])
    .agg(
        students=("student_id", "size"),
        students_10pct_plus=("absence_10pct_plus", "sum"),
    )
    .reset_index()
)

absence_threshold_summary["pct_students_10pct_plus"] = (
    absence_threshold_summary["students_10pct_plus"]
    / absence_threshold_summary["students"]
)

display(absence_threshold_summary)


In [ ]:
# Screen broad grade-level absence patterns

grade_attendance = (
    elem_context.groupby(
        ["school_year", "school_year_label", "school_name", "grade_level"]
    )
    .agg(
        students=("student_id", "size"),
        total_absent_days=("absent_days", "sum"),
        total_enrolled_days=("enrolled_instructional_days", "sum"),
        median_student_absence_rate=("absence_rate", "median"),
    )
    .reset_index()
)

grade_attendance["aggregate_absence_rate"] = (
    grade_attendance["total_absent_days"] / grade_attendance["total_enrolled_days"]
)

grade_attendance["grade_display"] = grade_attendance["grade_level"].replace(
    {0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
)

display(grade_attendance)


In [ ]:
# Latest-year grade pattern

latest_grade_attendance = grade_attendance.loc[
    grade_attendance["school_year"].eq(2026)
].pivot(index="school_name", columns="grade_display", values="aggregate_absence_rate")

display(latest_grade_attendance)


In [ ]:
# Check absence rates by enrollment exposure

elem_context["enrollment_exposure_group"] = pd.cut(
    elem_context["enrolled_instructional_days"],
    bins=[0, 90, 150, float("inf")],
    labels=["<=90 days", "91-150 days", ">150 days"],
)

exposure_attendance_check = (
    elem_context.groupby("enrollment_exposure_group", observed=True)
    .agg(
        students=("student_id", "size"),
        median_enrolled_days=("enrolled_instructional_days", "median"),
        mean_absence_rate=("absence_rate", "mean"),
        median_absence_rate=("absence_rate", "median"),
    )
    .reset_index()
)

display(exposure_attendance_check)


#### Findings from EDA 2

- Absence rates fell at all four schools. In 2025-26, school rates are close, at about 5.2% to 5.7%; Cedar Ridge is slightly higher.
- The median student rate is below the school rate, while the 90th percentile is around 12% to 14%. In other words, the highest-absence tenth of records starts around that range. Higher absence is concentrated among a smaller group.
- The grades with higher absence differ by school. Q3 examines these school-grade patterns more closely.
- Most records cover nearly a full year. A few students have much shorter enrollment, where a small number of missed days can produce a large percentage.
- The share missing at least 10% of enrolled days also fell at every school. This remains an exploratory measure, not an official chronic-absence rate.


### 3.6 Q2: How do starting points and Spring results differ?

QC found unique Fall and Spring benchmark results with valid dates, so I can pair them. Different window counts also prompted the participation check.

The LD needs both starting points and later results: a Spring score alone does not show how far a student has moved during the year.

#### EDA 3: Fall starting points

1. Summarize Reading and Mathematics separately by school, grade, and year.
2. Compare distributions and counts within grade. Benchmark scale scores rise with grade, so a schoolwide average can hide differences in grade mix.
3. Compare Fall performance categories alongside the score summaries.

The outputs show within-grade Fall results and their participation context.


In [ ]:
# Keep observed Fall results, with subjects analyzed separately.

fall_analysis = benchmark_pairs.loc[benchmark_pairs["has_fall"]].copy()

pd.Series(
    {
        "Fall benchmark records": len(fall_analysis),
        "Reading records": (fall_analysis["subject"] == "Reading").sum(),
        "Mathematics records": (fall_analysis["subject"] == "Mathematics").sum(),
    }
)


In [ ]:
# Review Fall sample sizes

fall_sample_size = (
    fall_analysis.groupby(
        ["school_year", "school_year_label", "subject", "school_name"]
    )
    .agg(tested_students=("student_id", "size"))
    .reset_index()
)

display(fall_sample_size)


In [ ]:
# Summarize Fall scores within grade

fall_score_summary = (
    fall_analysis.groupby(
        ["school_year", "school_year_label", "subject", "school_name", "grade_level"]
    )
    .agg(
        tested_students=("student_id", "size"),
        mean_fall_score=("fall_score", "mean"),
        median_fall_score=("fall_score", "median"),
        p25_fall_score=("fall_score", lambda x: x.quantile(0.25)),
        p75_fall_score=("fall_score", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

fall_score_summary["grade_display"] = fall_score_summary["grade_level"].replace(
    {0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
)

display(fall_score_summary)


In [ ]:
# Keep latest-year Fall summaries

latest_fall_summary = fall_score_summary.loc[
    fall_score_summary["school_year"].eq(2026)
].copy()

display(
    latest_fall_summary[
        [
            "subject",
            "school_name",
            "grade_display",
            "tested_students",
            "mean_fall_score",
            "median_fall_score",
            "p25_fall_score",
            "p75_fall_score",
        ]
    ]
)


In [ ]:
# Latest-year Reading starting points

latest_reading_fall = latest_fall_summary.loc[
    latest_fall_summary["subject"].eq("Reading")
].pivot(index="school_name", columns="grade_display", values="median_fall_score")

display(latest_reading_fall)


In [ ]:
# Latest-year Mathematics starting points

latest_math_fall = latest_fall_summary.loc[
    latest_fall_summary["subject"].eq("Mathematics")
].pivot(index="school_name", columns="grade_display", values="median_fall_score")

display(latest_math_fall)


In [ ]:
# Compare pooled Fall school summaries

pooled_fall_summary = (
    fall_analysis.groupby(
        ["school_year", "school_year_label", "subject", "school_name"]
    )
    .agg(
        tested_students=("student_id", "size"),
        mean_fall_score=("fall_score", "mean"),
        median_fall_score=("fall_score", "median"),
        p25_fall_score=("fall_score", lambda x: x.quantile(0.25)),
        p75_fall_score=("fall_score", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

display(pooled_fall_summary.loc[pooled_fall_summary["school_year"].eq(2026)])


In [ ]:
# Count Fall performance levels

fall_performance_distribution = (
    fall_analysis.groupby(
        [
            "school_year",
            "school_year_label",
            "subject",
            "school_name",
            "grade_level",
            "fall_performance_level",
        ]
    )
    .size()
    .rename("students")
    .reset_index()
)


In [ ]:
# Calculate performance-level proportions

fall_performance_distribution["group_total"] = fall_performance_distribution.groupby(
    ["school_year", "subject", "school_name", "grade_level"]
)["students"].transform("sum")

fall_performance_distribution["performance_level_pct"] = (
    fall_performance_distribution["students"]
    / fall_performance_distribution["group_total"]
)

fall_performance_distribution["grade_display"] = fall_performance_distribution[
    "grade_level"
].replace({0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"})

display(
    fall_performance_distribution.loc[
        fall_performance_distribution["school_year"].eq(2026)
    ]
)


In [ ]:
# Plot latest-year Fall Reading scores by grade

latest_reading = fall_analysis.loc[
    (fall_analysis["school_year"].eq(2026)) & (fall_analysis["subject"].eq("Reading"))
].copy()

fig, ax = plt.subplots(figsize=(11, 5))

latest_reading.boxplot(
    column="fall_score", by=["grade_level", "school_name"], ax=ax, grid=False, rot=90
)

ax.set_title("2025–26 Fall Reading Scores by Grade and School")
ax.set_xlabel("Grade and School")
ax.set_ylabel("Fall Benchmark Score")

plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
# Add Fall participation context

fall_participation_note = sample_flow[
    [
        "enrolled_population",
        "fall_opportunity",
        "fall_results",
        "fall_participation_rate",
    ]
].copy()

display(fall_participation_note)


In [ ]:
# Create a compact At/Above Benchmark summary

latest_fall_performance = fall_analysis.loc[
    fall_analysis["school_year"].eq(2026)
].copy()

latest_fall_performance["at_or_above_benchmark"] = latest_fall_performance[
    "fall_performance_level"
].isin(["At Benchmark", "Above Benchmark"])

fall_status_summary = (
    latest_fall_performance.groupby(["subject", "school_name", "grade_level"])
    .agg(
        tested_students=("student_id", "size"),
        at_or_above_benchmark=("at_or_above_benchmark", "sum"),
        at_or_above_benchmark_pct=("at_or_above_benchmark", "mean"),
    )
    .reset_index()
)

fall_status_summary["grade_display"] = fall_status_summary["grade_level"].replace(
    {0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
)

display(fall_status_summary)


#### Findings from EDA 3

- Within a grade, Fall starting points are generally similar across the four schools, with differences in selected grades. No school is consistently highest or lowest across all K-5 grades and both subjects.
- Overall 2025-26 school averages are especially close in Mathematics. Looking within grade reveals differences that the overall averages hide.
- Fall participation is about 95% of estimated opportunities in both subjects. Results describe tested students, and the counts remain important when comparing grade groups.


#### EDA 4: Spring results and Fall-Spring changes

1. Summarize Spring scores and At/Above Benchmark percentages with tested counts.
2. For students with both tests, calculate Spring minus Fall and examine the spread of changes.
3. Plot Fall against Spring within subject and grade, identifying schools.
4. Check whether lower Spring results were already visible at Fall.
5. Label all Spring-tested results separately from paired results because the samples differ.

These outputs describe observed changes. The task supplies no expected-growth target, so I cannot judge whether a change is sufficient. Benchmark status and state proficiency are separate measures, and raw changes alone cannot establish school effectiveness.


In [ ]:
# Keep all observed Spring results, including students without a Fall result.

spring_analysis = benchmark_pairs.loc[benchmark_pairs["has_spring"]].copy()

pd.Series(
    {
        "Spring benchmark records": len(spring_analysis),
        "Reading records": (spring_analysis["subject"] == "Reading").sum(),
        "Mathematics records": (spring_analysis["subject"] == "Mathematics").sum(),
    }
)


In [ ]:
# Define Spring benchmark status

spring_analysis["at_or_above_benchmark"] = spring_analysis[
    "spring_performance_level"
].isin(["At Benchmark", "Above Benchmark"])


In [ ]:
# Summarize Spring outcomes
# By subject, school, grade, and year

spring_summary = (
    spring_analysis.groupby(
        ["school_year", "school_year_label", "subject", "school_name", "grade_level"]
    )
    .agg(
        tested_students=("student_id", "size"),
        mean_spring_score=("spring_score", "mean"),
        median_spring_score=("spring_score", "median"),
        p25_spring_score=("spring_score", lambda x: x.quantile(0.25)),
        p75_spring_score=("spring_score", lambda x: x.quantile(0.75)),
        at_or_above_benchmark=("at_or_above_benchmark", "sum"),
        at_or_above_benchmark_pct=("at_or_above_benchmark", "mean"),
    )
    .reset_index()
)

spring_summary["grade_display"] = spring_summary["grade_level"].replace(
    {0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
)

display(spring_summary)


In [ ]:
# Keep latest-year Spring outcomes

latest_spring_summary = spring_summary.loc[
    spring_summary["school_year"].eq(2026)
].copy()

display(
    latest_spring_summary[
        [
            "subject",
            "school_name",
            "grade_display",
            "tested_students",
            "mean_spring_score",
            "median_spring_score",
            "p25_spring_score",
            "p75_spring_score",
            "at_or_above_benchmark_pct",
        ]
    ]
)


In [ ]:
# Keep both-tested students for score change; attendance eligibility is separate.

paired_analysis = benchmark_pairs.loc[benchmark_pairs["has_fall_spring_pair"]].copy()

# Observed score change
paired_analysis["score_change"] = (
    paired_analysis["spring_score"] - paired_analysis["fall_score"]
)

pd.Series(
    {
        "paired records": len(paired_analysis),
        "Reading pairs": (paired_analysis["subject"] == "Reading").sum(),
        "Mathematics pairs": (paired_analysis["subject"] == "Mathematics").sum(),
    }
)


In [ ]:
# Summarize observed Fall-Spring score changes

paired_change_summary = (
    paired_analysis.groupby(
        ["school_year", "school_year_label", "subject", "school_name", "grade_level"]
    )
    .agg(
        paired_students=("student_id", "size"),
        mean_fall_score=("fall_score", "mean"),
        median_fall_score=("fall_score", "median"),
        mean_spring_score=("spring_score", "mean"),
        median_spring_score=("spring_score", "median"),
        mean_score_change=("score_change", "mean"),
        median_score_change=("score_change", "median"),
        p25_score_change=("score_change", lambda x: x.quantile(0.25)),
        p75_score_change=("score_change", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

paired_change_summary["grade_display"] = paired_change_summary["grade_level"].replace(
    {0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
)

display(paired_change_summary)


In [ ]:
# Review latest-year paired changes

latest_paired_change = paired_change_summary.loc[
    paired_change_summary["school_year"].eq(2026)
].copy()

display(
    latest_paired_change[
        [
            "subject",
            "school_name",
            "grade_display",
            "paired_students",
            "median_fall_score",
            "median_spring_score",
            "median_score_change",
            "p25_score_change",
            "p75_score_change",
        ]
    ]
)


In [ ]:
# Plot Fall vs Spring Reading scores

latest_reading_pairs = paired_analysis.loc[
    (paired_analysis["school_year"].eq(2026))
    & (paired_analysis["subject"].eq("Reading"))
].copy()

for grade in sorted(latest_reading_pairs["grade_level"].unique()):

    plot_data = latest_reading_pairs.loc[latest_reading_pairs["grade_level"].eq(grade)]

    fig, ax = plt.subplots(figsize=(7, 5))

    for school, group in plot_data.groupby("school_name"):
        ax.scatter(group["fall_score"], group["spring_score"], alpha=0.6, label=school)

    grade_label = "K" if grade == 0 else str(grade)

    ax.set_title(f"2025–26 Grade {grade_label} Reading: Fall vs Spring")
    ax.set_xlabel("Fall Benchmark Score")
    ax.set_ylabel("Spring Benchmark Score")
    ax.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# Compare Fall starting points,
# Spring outcomes, and observed changes

fall_spring_comparison = latest_paired_change[
    [
        "subject",
        "school_name",
        "grade_display",
        "paired_students",
        "median_fall_score",
        "median_spring_score",
        "median_score_change",
    ]
].sort_values(["subject", "grade_display", "school_name"])

display(fall_spring_comparison)


In [ ]:
# Compare full Spring-tested and paired samples

spring_sample_counts = (
    spring_analysis.loc[spring_analysis["school_year"].eq(2026)]
    .groupby(["subject", "school_name", "grade_level"])
    .agg(spring_tested_students=("student_id", "size"))
    .reset_index()
)

paired_sample_counts = (
    paired_analysis.loc[paired_analysis["school_year"].eq(2026)]
    .groupby(["subject", "school_name", "grade_level"])
    .agg(paired_students=("student_id", "size"))
    .reset_index()
)

spring_vs_paired_sample = spring_sample_counts.merge(
    paired_sample_counts,
    on=["subject", "school_name", "grade_level"],
    how="left",
    validate="one_to_one",
)

spring_vs_paired_sample["paired_share_of_spring_tested"] = (
    spring_vs_paired_sample["paired_students"]
    / spring_vs_paired_sample["spring_tested_students"]
)

display(spring_vs_paired_sample)


#### Findings from EDA 4

In 2025-26, Reading At/Above Benchmark ranges from 53.6% to 61.3% and Mathematics from 52.8% to 62.2%. These percentages include all Spring-tested students and combine K-5. Grade mix and student background may contribute to the differences.

| School | Reading At/Above / Spring tested | Mathematics At/Above / Spring tested |
|---|---:|---:|
| Cedar Ridge | 198 / 323 (61.3%) | 198 / 323 (61.3%) |
| Harper Creek | 187 / 333 (56.2%) | 206 / 331 (62.2%) |
| Lincoln Park | 188 / 351 (53.6%) | 198 / 354 (55.9%) |
| Meadowbrook | 164 / 303 (54.1%) | 160 / 303 (52.8%) |

- Median paired change is positive in every 2025-26 school-grade-subject group. Reading medians range from 4 to 22 points; Mathematics from 3.5 to 21.5. Early grades show larger changes. Without grade-specific growth targets, this does not tell us that teaching is stronger in those grades or growth insufficient in older grades.
- Starting points help explain Spring comparisons. In Kindergarten Mathematics, Lincoln Park's paired Fall and Spring medians are 132 and 154.5 (52 pairs), compared with 140 and 159.5 at Cedar Ridge (56 pairs). Median individual changes are 19 and 20.5. The schools already differ at Fall. Subtracting group medians is not the same calculation as taking the median of individual changes.
- School patterns vary by grade and subject. The within-grade tables provide context for later adjusted comparisons; an overall statistic cannot capture every pattern.
- Paired records cover 86.2% to 100% of Spring Reading records and 84.6% to 100% of Spring Mathematics records within latest-year school-grade groups. The table above uses everyone Spring-tested; change summaries use both-tested students. These counts show coverage, not identical score distributions between the samples.
- The remaining question is whether Spring school differences persist after accounting for Fall score, grade, year, and selected student background. The EDA does not establish a causal school effect.


### 3.7 Q3: Where is absence concentrated, and how does it relate to learning?

QC confirmed daily coverage and valid enrollment and test dates. This lets me calculate rates using actual days enrolled, including the period between Fall and Spring tests. Calendar length and time enrolled make rates more useful than raw missed-day counts.

For the LD, overlapping attendance and academic needs would be a reason to ask how support is coordinated. These data show associations, not how many points an attendance intervention would add.

#### EDA 5: Locate attendance concerns

1. Compare student absence-rate distributions by school, grade, and year.
2. Examine selected student groups with counts, keeping small groups visible.
3. Review whether grade patterns recur and describe short enrollment periods. The effect of excluding short records remains untested.
4. Summarize days and absence rates between tests separately by subject.

The outputs show school-grade attendance patterns and between-test coverage. A recurring grade pattern can involve different students each year; it does not identify the incoming fall cohort.


In [ ]:
# Summarize annual absence-rate distributions
# By school, grade, and year

annual_attendance_by_grade = (
    elem_context.groupby(
        ["school_year", "school_year_label", "school_name", "grade_level"]
    )
    .agg(
        students=("student_id", "size"),
        total_absent_days=("absent_days", "sum"),
        total_enrolled_days=("enrolled_instructional_days", "sum"),
        mean_student_absence_rate=("absence_rate", "mean"),
        median_student_absence_rate=("absence_rate", "median"),
        p75_student_absence_rate=("absence_rate", lambda x: x.quantile(0.75)),
        p90_student_absence_rate=("absence_rate", lambda x: x.quantile(0.90)),
    )
    .reset_index()
)

annual_attendance_by_grade["aggregate_absence_rate"] = (
    annual_attendance_by_grade["total_absent_days"]
    / annual_attendance_by_grade["total_enrolled_days"]
)

annual_attendance_by_grade["grade_display"] = annual_attendance_by_grade[
    "grade_level"
].replace({0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"})

display(annual_attendance_by_grade)


In [ ]:
# Latest-year school-by-grade attendance view

latest_grade_attendance_view = annual_attendance_by_grade.loc[
    annual_attendance_by_grade["school_year"].eq(2026)
].pivot(index="school_name", columns="grade_display", values="aggregate_absence_rate")

display(latest_grade_attendance_view)


In [ ]:
# Latest-year sample sizes by school and grade

latest_grade_sample_size = annual_attendance_by_grade.loc[
    annual_attendance_by_grade["school_year"].eq(2026)
].pivot(index="school_name", columns="grade_display", values="students")

display(latest_grade_sample_size)


In [ ]:
# Check whether grade patterns recur across years

recurring_grade_patterns = annual_attendance_by_grade[
    [
        "school_year_label",
        "school_name",
        "grade_display",
        "students",
        "aggregate_absence_rate",
        "median_student_absence_rate",
    ]
].sort_values(["school_name", "grade_display", "school_year_label"])

display(recurring_grade_patterns)


In [ ]:
# Three-year absence pattern by school and grade

grade_trend_pivot = annual_attendance_by_grade.pivot_table(
    index=["school_name", "grade_display"],
    columns="school_year_label",
    values="aggregate_absence_rate",
)

display(grade_trend_pivot)


In [ ]:
el_attendance = summarize_attendance_by_group(elem_context, "english_learner_flag")

econ_attendance = summarize_attendance_by_group(
    elem_context, "economically_disadvantaged_flag"
)

sped_attendance = summarize_attendance_by_group(elem_context, "special_education_flag")

display(el_attendance)
display(econ_attendance)
display(sped_attendance)


In [ ]:
# Flag small subgroup cells

MIN_CELL_SIZE = 20

for summary in [el_attendance, econ_attendance, sped_attendance]:
    summary["small_cell"] = summary["students"] < MIN_CELL_SIZE


In [ ]:
# Compare attendance by enrollment exposure

exposure_attendance_by_school = (
    elem_context.groupby(
        ["school_year", "school_name", "enrollment_exposure_group"], observed=True
    )
    .agg(
        students=("student_id", "size"),
        median_enrolled_days=("enrolled_instructional_days", "median"),
        median_absence_rate=("absence_rate", "median"),
        mean_absence_rate=("absence_rate", "mean"),
    )
    .reset_index()
)

display(exposure_attendance_by_school)


In [ ]:
# Build the between-test attendance sample

between_test_analysis = benchmark_pairs.loc[
    benchmark_pairs["has_fall_spring_pair"]
].copy()

pd.Series(
    {
        "between-test records": len(between_test_analysis),
        "Reading records": (between_test_analysis["subject"] == "Reading").sum(),
        "Mathematics records": (
            between_test_analysis["subject"] == "Mathematics"
        ).sum(),
        "missing eligible-day counts": between_test_analysis[
            "between_test_eligible_days"
        ]
        .isna()
        .sum(),
        "zero eligible-day intervals": between_test_analysis[
            "between_test_eligible_days"
        ]
        .eq(0)
        .sum(),
    }
)


In [ ]:
# Summarize between-test exposure by subject

between_test_exposure_summary = (
    between_test_analysis.groupby("subject")
    .agg(
        student_year_subjects=("student_id", "size"),
        mean_eligible_days=("between_test_eligible_days", "mean"),
        median_eligible_days=("between_test_eligible_days", "median"),
        p25_eligible_days=("between_test_eligible_days", lambda x: x.quantile(0.25)),
        p75_eligible_days=("between_test_eligible_days", lambda x: x.quantile(0.75)),
        total_absent_days=("between_test_absent_days", "sum"),
        total_eligible_days=("between_test_eligible_days", "sum"),
        median_student_absence_rate=("between_test_absence_rate", "median"),
        p75_student_absence_rate=(
            "between_test_absence_rate",
            lambda x: x.quantile(0.75),
        ),
        p90_student_absence_rate=(
            "between_test_absence_rate",
            lambda x: x.quantile(0.90),
        ),
    )
    .reset_index()
)

between_test_exposure_summary["aggregate_between_test_absence_rate"] = (
    between_test_exposure_summary["total_absent_days"]
    / between_test_exposure_summary["total_eligible_days"]
)

display(between_test_exposure_summary)


In [ ]:
# Summarize between-test absence
# By subject, school, and grade

between_test_by_grade = (
    between_test_analysis.groupby(
        ["school_year", "school_year_label", "subject", "school_name", "grade_level"]
    )
    .agg(
        students=("student_id", "size"),
        total_between_test_absent_days=("between_test_absent_days", "sum"),
        total_between_test_eligible_days=("between_test_eligible_days", "sum"),
        median_between_test_absence_rate=("between_test_absence_rate", "median"),
        p75_between_test_absence_rate=(
            "between_test_absence_rate",
            lambda x: x.quantile(0.75),
        ),
    )
    .reset_index()
)

between_test_by_grade["aggregate_between_test_absence_rate"] = (
    between_test_by_grade["total_between_test_absent_days"]
    / between_test_by_grade["total_between_test_eligible_days"]
)

between_test_by_grade["grade_display"] = between_test_by_grade["grade_level"].replace(
    {0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
)

display(between_test_by_grade)


In [ ]:
# Latest-year Reading between-test absence

latest_reading_between_test = between_test_by_grade.loc[
    (between_test_by_grade["school_year"].eq(2026))
    & (between_test_by_grade["subject"].eq("Reading"))
].pivot(
    index="school_name",
    columns="grade_display",
    values="aggregate_between_test_absence_rate",
)

display(latest_reading_between_test)


In [ ]:
# Latest-year Mathematics between-test absence

latest_math_between_test = between_test_by_grade.loc[
    (between_test_by_grade["school_year"].eq(2026))
    & (between_test_by_grade["subject"].eq("Mathematics"))
].pivot(
    index="school_name",
    columns="grade_display",
    values="aggregate_between_test_absence_rate",
)

display(latest_math_between_test)


#### Findings from EDA 5

- Attendance concerns vary by school and grade. In 2025-26, each school's highest grade-level absence is Grade 1 at Cedar Ridge and Harper Creek, Kindergarten at Lincoln Park, and Grade 5 at Meadowbrook. There is no single highest-absence grade across the four schools.
- Some patterns recur; others shift. Lincoln Park Kindergarten has relatively high absence across several years. These historical patterns suggest follow-up questions, but do not identify the same students across years or describe the coming fall roster.
- Economically disadvantaged students have higher descriptive absence rates in several schools in 2025-26, especially Cedar Ridge, Lincoln Park, and Meadowbrook. The pattern is not uniform. EL and SPED comparisons are less consistent across schools and years.
- Most student-year records have more than 150 enrolled instructional days. Short-enrollment groups are small and their rates are less stable. I have not tested how excluding them changes the results.
- Between-test periods are similar by subject: medians of 140 instructional days in Reading and 139 in Mathematics. Aggregate absence is 5.86% and 5.94% across three years. All complete pairs have positive intervals and attendance summaries, so the rates can be used in the learning comparisons.


#### EDA 6: Attendance alongside learning

1. Compare between-test absence with Fall scores to check whether students already differ at the start.
2. Compare it with Spring scores and Fall-Spring changes, separately by subject.
3. Examine relationships within grade and, where useful, within school. Look for curves, unusual points, and sparsely represented ranges.
4. Use clearly labeled absence bands for descriptive summaries, with counts and days enrolled. The bands are display categories, not official policy thresholds.

The plots and summaries describe raw relationships. A weak relationship with score change is still a finding; the question about Spring scores after adjusting for Fall remains open.


In [ ]:
# Build the attendance-learning analysis sample

attendance_learning = between_test_analysis.copy()

attendance_learning["score_change"] = (
    attendance_learning["spring_score"] - attendance_learning["fall_score"]
)

pd.Series(
    {
        "analysis records": len(attendance_learning),
        "Reading records": (attendance_learning["subject"] == "Reading").sum(),
        "Mathematics records": (attendance_learning["subject"] == "Mathematics").sum(),
        "missing between-test absence rate": attendance_learning[
            "between_test_absence_rate"
        ]
        .isna()
        .sum(),
        "missing Fall score": attendance_learning["fall_score"].isna().sum(),
        "missing Spring score": attendance_learning["spring_score"].isna().sum(),
    }
)


In [ ]:
# Review the between-test absence-rate distribution

absence_distribution = attendance_learning.groupby("subject")[
    "between_test_absence_rate"
].agg(
    n="size",
    mean="mean",
    median="median",
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75),
    p90=lambda x: x.quantile(0.90),
    p95=lambda x: x.quantile(0.95),
    max="max",
)

display(absence_distribution)


In [ ]:
# Between-test absence vs Fall Reading score

reading_data = attendance_learning.loc[
    attendance_learning["subject"].eq("Reading")
].copy()

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    reading_data["between_test_absence_rate"] * 100,
    reading_data["fall_score"],
    alpha=0.25,
)

ax.set_title("2023–24 to 2025–26 Reading: Absence vs Fall Score")
ax.set_xlabel("Between-Test Absence Rate (%)")
ax.set_ylabel("Fall Benchmark Score")

plt.tight_layout()
plt.show()


In [ ]:
# Between-test absence vs Fall Mathematics score

math_data = attendance_learning.loc[
    attendance_learning["subject"].eq("Mathematics")
].copy()

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    math_data["between_test_absence_rate"] * 100, math_data["fall_score"], alpha=0.25
)

ax.set_title("2023–24 to 2025–26 Mathematics: Absence vs Fall Score")
ax.set_xlabel("Between-Test Absence Rate (%)")
ax.set_ylabel("Fall Benchmark Score")

plt.tight_layout()
plt.show()


In [ ]:
# Reading absence vs Fall score within grade

for grade in sorted(reading_data["grade_level"].unique()):

    plot_data = reading_data.loc[reading_data["grade_level"].eq(grade)]

    grade_label = "K" if grade == 0 else str(grade)

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.scatter(
        plot_data["between_test_absence_rate"] * 100,
        plot_data["fall_score"],
        alpha=0.35,
    )

    ax.set_title(
        f"2023–24 to 2025–26 Grade {grade_label} Reading: Absence vs Fall Score"
    )
    ax.set_xlabel("Between-Test Absence Rate (%)")
    ax.set_ylabel("Fall Benchmark Score")

    plt.tight_layout()
    plt.show()


In [ ]:
# Mathematics absence vs Fall score within grade

for grade in sorted(math_data["grade_level"].unique()):

    plot_data = math_data.loc[math_data["grade_level"].eq(grade)]

    grade_label = "K" if grade == 0 else str(grade)

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.scatter(
        plot_data["between_test_absence_rate"] * 100,
        plot_data["fall_score"],
        alpha=0.35,
    )

    ax.set_title(
        f"2023–24 to 2025–26 Grade {grade_label} Mathematics: Absence vs Fall Score"
    )
    ax.set_xlabel("Between-Test Absence Rate (%)")
    ax.set_ylabel("Fall Benchmark Score")

    plt.tight_layout()
    plt.show()


In [ ]:
# Reading absence vs Spring score

for grade in sorted(reading_data["grade_level"].unique()):

    plot_data = reading_data.loc[reading_data["grade_level"].eq(grade)]

    grade_label = "K" if grade == 0 else str(grade)

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.scatter(
        plot_data["between_test_absence_rate"] * 100,
        plot_data["spring_score"],
        alpha=0.35,
    )

    ax.set_title(
        f"2023–24 to 2025–26 Grade {grade_label} Reading: Absence vs Spring Score"
    )
    ax.set_xlabel("Between-Test Absence Rate (%)")
    ax.set_ylabel("Spring Benchmark Score")

    plt.tight_layout()
    plt.show()


In [ ]:
# Mathematics absence vs Spring score

for grade in sorted(math_data["grade_level"].unique()):

    plot_data = math_data.loc[math_data["grade_level"].eq(grade)]

    grade_label = "K" if grade == 0 else str(grade)

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.scatter(
        plot_data["between_test_absence_rate"] * 100,
        plot_data["spring_score"],
        alpha=0.35,
    )

    ax.set_title(
        f"2023–24 to 2025–26 Grade {grade_label} Mathematics: Absence vs Spring Score"
    )
    ax.set_xlabel("Between-Test Absence Rate (%)")
    ax.set_ylabel("Spring Benchmark Score")

    plt.tight_layout()
    plt.show()


In [ ]:
# Reading absence vs observed score change

for grade in sorted(reading_data["grade_level"].unique()):

    plot_data = reading_data.loc[reading_data["grade_level"].eq(grade)]

    grade_label = "K" if grade == 0 else str(grade)

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.scatter(
        plot_data["between_test_absence_rate"] * 100,
        plot_data["score_change"],
        alpha=0.35,
    )

    ax.axhline(0, linewidth=1)

    ax.set_title(
        f"2023–24 to 2025–26 Grade {grade_label} Reading: Absence vs Score Change"
    )
    ax.set_xlabel("Between-Test Absence Rate (%)")
    ax.set_ylabel("Spring Score - Fall Score")

    plt.tight_layout()
    plt.show()


In [ ]:
# Mathematics absence vs observed score change

for grade in sorted(math_data["grade_level"].unique()):

    plot_data = math_data.loc[math_data["grade_level"].eq(grade)]

    grade_label = "K" if grade == 0 else str(grade)

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.scatter(
        plot_data["between_test_absence_rate"] * 100,
        plot_data["score_change"],
        alpha=0.35,
    )

    ax.axhline(0, linewidth=1)

    ax.set_title(
        f"2023–24 to 2025–26 Grade {grade_label} Mathematics: Absence vs Score Change"
    )
    ax.set_xlabel("Between-Test Absence Rate (%)")
    ax.set_ylabel("Spring Score - Fall Score")

    plt.tight_layout()
    plt.show()


In [ ]:
# Summarize raw relationships within grade

relationship_summary = (
    attendance_learning.groupby(["subject", "grade_level"])
    .apply(
        lambda g: pd.Series(
            {
                "student_year_records": len(g),
                "absence_fall_spearman": g["between_test_absence_rate"].corr(
                    g["fall_score"], method="spearman"
                ),
                "absence_spring_spearman": g["between_test_absence_rate"].corr(
                    g["spring_score"], method="spearman"
                ),
                "absence_change_spearman": g["between_test_absence_rate"].corr(
                    g["score_change"], method="spearman"
                ),
            }
        )
    )
    .reset_index()
)

relationship_summary["grade_display"] = relationship_summary["grade_level"].replace(
    {0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
)

display(relationship_summary)


In [ ]:
# Summarize relationships within school and grade

school_grade_relationship = (
    attendance_learning.groupby(["subject", "school_name", "grade_level"])
    .apply(
        lambda g: pd.Series(
            {
                "student_year_records": len(g),
                "absence_fall_spearman": g["between_test_absence_rate"].corr(
                    g["fall_score"], method="spearman"
                ),
                "absence_spring_spearman": g["between_test_absence_rate"].corr(
                    g["spring_score"], method="spearman"
                ),
                "absence_change_spearman": g["between_test_absence_rate"].corr(
                    g["score_change"], method="spearman"
                ),
            }
        )
    )
    .reset_index()
)

school_grade_relationship["grade_display"] = school_grade_relationship[
    "grade_level"
].replace({0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"})

display(school_grade_relationship)


In [ ]:
# Check sample sizes across absence ranges

absence_range_counts = pd.DataFrame(
    {
        "range": ["<5%", "5% to <10%", "10% to <15%", "15% to <20%", "20%+"],
        "Reading": [
            (reading_data["between_test_absence_rate"] < 0.05).sum(),
            (
                (reading_data["between_test_absence_rate"] >= 0.05)
                & (reading_data["between_test_absence_rate"] < 0.10)
            ).sum(),
            (
                (reading_data["between_test_absence_rate"] >= 0.10)
                & (reading_data["between_test_absence_rate"] < 0.15)
            ).sum(),
            (
                (reading_data["between_test_absence_rate"] >= 0.15)
                & (reading_data["between_test_absence_rate"] < 0.20)
            ).sum(),
            (reading_data["between_test_absence_rate"] >= 0.20).sum(),
        ],
        "Mathematics": [
            (math_data["between_test_absence_rate"] < 0.05).sum(),
            (
                (math_data["between_test_absence_rate"] >= 0.05)
                & (math_data["between_test_absence_rate"] < 0.10)
            ).sum(),
            (
                (math_data["between_test_absence_rate"] >= 0.10)
                & (math_data["between_test_absence_rate"] < 0.15)
            ).sum(),
            (
                (math_data["between_test_absence_rate"] >= 0.15)
                & (math_data["between_test_absence_rate"] < 0.20)
            ).sum(),
            (math_data["between_test_absence_rate"] >= 0.20).sum(),
        ],
    }
)

display(absence_range_counts)


In [ ]:
# Descriptive display bands; not official attendance categories.
# Create descriptive absence bands

attendance_learning["absence_band"] = pd.cut(
    attendance_learning["between_test_absence_rate"],
    bins=[-0.001, 0.05, 0.10, 0.15, 0.20, float("inf")],
    labels=["<5%", "5% to <10%", "10% to <15%", "15% to <20%", "20%+"],
    right=False,
)


In [ ]:
# Summarize learning within grade and absence band.
# Counts are student-year records pooled across three years.

absence_band_summary = (
    attendance_learning.groupby(
        ["subject", "grade_level", "absence_band"], observed=True
    )
    .agg(
        students=("student_id", "size"),
        median_eligible_days=("between_test_eligible_days", "median"),
        median_fall_score=("fall_score", "median"),
        median_spring_score=("spring_score", "median"),
        median_score_change=("score_change", "median"),
        p25_score_change=("score_change", lambda x: x.quantile(0.25)),
        p75_score_change=("score_change", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

absence_band_summary["grade_display"] = absence_band_summary["grade_level"].replace(
    {0: "K", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
)

absence_band_summary["small_cell"] = absence_band_summary["students"] < 20

display(absence_band_summary)


#### Findings from EDA 6

- These plots and summaries combine three years. Reading has 3,653 student-year pairs (1,780 unique students); Mathematics has 3,606 (1,779 unique students). Models must account for students appearing in several years.
- Within grades, higher between-test absence tends to occur with lower Fall and Spring scores. Spearman correlations are about -0.23 to -0.15 in both cases, indicating modest negative relationships. These do not prove that absence caused the score differences.
- The Fall relationship matters because those scores were measured before the attendance interval. Students with higher later absence were already starting lower. A Spring comparison therefore needs to account for Fall achievement.
- Absence has little raw relationship with score change: correlations range from -0.06 to +0.05, and median changes are often similar across absence bands. This does not establish that absence is harmless or answer the adjusted Spring question.
- Two of 60 grade-subject-band groups have fewer than 20 records: Mathematics grade 1 and Reading grade 5 in the 20%+ band, each with 18. I use 20 as a caution for these displays, not an official privacy or statistical rule.
- The next question is whether absence remains related to Spring scores after accounting for Fall score and recorded context, and how large and uncertain that relationship is.


### 3.8 Check other possible briefing priorities

I reviewed three additional topics before choosing the main story:

- State tests: ELA/Math in grades 3-5 and Science in grade 5, with participation and tested counts. These stay separate from benchmarks.
- Discipline: incidents, students involved, and removal days, using stated denominators and showing small counts.
- Staffing: recorded yearly assignment FTE and roles. FTE per enrolled student is not class size. The 14 missing teacher-year assignments limit historical comparisons, and a missing role label does not establish missing services.

I consider the size and consistency of a finding, who the data cover, and what the LD could usefully ask or verify. A large percentage difference alone is not enough to identify a management problem.


In [ ]:
# Keep elementary State Summative tests

state_tests = assessment.loc[
    assessment["assessment_name"].eq("State Summative Assessment")
    & assessment["school_id"].isin(ELEMENTARY_SCHOOL_IDS)
    & assessment["school_year"].isin(ANALYSIS_YEARS)
    & (
        (
            assessment["grade_level_tested"].isin([3, 4, 5])
            & assessment["subject"].isin(["English Language Arts", "Mathematics"])
        )
        | (assessment["grade_level_tested"].eq(5) & assessment["subject"].eq("Science"))
    )
].copy()

display(
    state_tests[["school_year", "school_id", "grade_level_tested", "subject"]]
    .value_counts()
    .sort_index()
)


In [ ]:
# Build observed State-test windows

state_window_proxy = (
    state_tests.groupby(["school_year", "school_id", "grade_level_tested", "subject"])
    .agg(state_window_start=("test_date", "min"), state_window_end=("test_date", "max"))
    .reset_index()
    .rename(columns={"grade_level_tested": "grade_level"})
)


In [ ]:
# Define tested grade-subject combinations

state_requirements = pd.DataFrame(
    [
        [3, "English Language Arts"],
        [3, "Mathematics"],
        [4, "English Language Arts"],
        [4, "Mathematics"],
        [5, "English Language Arts"],
        [5, "Mathematics"],
        [5, "Science"],
    ],
    columns=["grade_level", "subject"],
)


In [ ]:
# Build State-test opportunity population

state_opportunity = (
    elem_enrollments.loc[elem_enrollments["grade_level"].isin([3, 4, 5])]
    .merge(state_requirements, on="grade_level", how="inner", validate="many_to_many")
    .merge(
        schools[["school_id", "school_name"]],
        on="school_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        state_window_proxy,
        on=["school_year", "school_id", "grade_level", "subject"],
        how="left",
        validate="many_to_one",
    )
)

state_opportunity["state_test_opportunity"] = (
    state_opportunity["entry_date"] <= state_opportunity["state_window_end"]
) & (state_opportunity["exit_date"] >= state_opportunity["state_window_start"])


In [ ]:
# Attach observed State-test results

state_results = state_tests[
    [
        "student_id",
        "school_id",
        "school_year",
        "grade_level_tested",
        "subject",
        "test_date",
        "met_standard",
    ]
].rename(columns={"grade_level_tested": "grade_level", "test_date": "state_test_date"})

assert (
    state_results.duplicated(
        ["student_id", "school_id", "school_year", "grade_level", "subject"]
    ).sum()
    == 0
)

state_participation = state_opportunity.merge(
    state_results,
    on=["student_id", "school_id", "school_year", "grade_level", "subject"],
    how="left",
    validate="one_to_one",
)

state_participation["tested"] = state_participation["state_test_date"].notna()


In [ ]:
# Summarize State-test participation and outcomes

state_screen = (
    state_participation.loc[state_participation["state_test_opportunity"]]
    .groupby(["school_year", "school_name", "grade_level", "subject"])
    .agg(
        opportunity_students=("student_id", "size"),
        tested_students=("tested", "sum"),
        met_standard_students=("met_standard", "sum"),
    )
    .reset_index()
)

state_screen["participation_rate"] = (
    state_screen["tested_students"] / state_screen["opportunity_students"]
)

state_screen["met_standard_rate"] = (
    state_screen["met_standard_students"] / state_screen["tested_students"]
)

display(state_screen)


In [ ]:
# Keep elementary discipline incidents

elem_discipline = discipline.loc[
    discipline["school_id"].isin(ELEMENTARY_SCHOOL_IDS)
    & discipline["school_year"].isin(ANALYSIS_YEARS)
].copy()


In [ ]:
# Build discipline denominators

discipline_denominator = (
    elem_context.groupby(
        ["school_year", "school_year_label", "school_id", "school_name"]
    )
    .agg(
        enrolled_students=("student_id", "size"),
        enrolled_instructional_days=("enrolled_instructional_days", "sum"),
    )
    .reset_index()
)


In [ ]:
# Summarize discipline indicators

discipline_summary = (
    elem_discipline.groupby(["school_year", "school_id"])
    .agg(
        incidents=("incident_id", "size"),
        students_involved=("student_id", "nunique"),
        removal_days=("days_removed", "sum"),
    )
    .reset_index()
)

discipline_summary = discipline_denominator.merge(
    discipline_summary,
    on=["school_year", "school_id"],
    how="left",
    validate="one_to_one",
)

for col in ["incidents", "students_involved", "removal_days"]:
    discipline_summary[col] = discipline_summary[col].fillna(0)


In [ ]:
# Add stated discipline denominators

discipline_summary["incidents_per_100_students"] = (
    discipline_summary["incidents"] / discipline_summary["enrolled_students"] * 100
)

discipline_summary["pct_students_with_incident"] = (
    discipline_summary["students_involved"] / discipline_summary["enrolled_students"]
)

discipline_summary["removal_days_per_1000_enrolled_days"] = (
    discipline_summary["removal_days"]
    / discipline_summary["enrolled_instructional_days"]
    * 1000
)

display(discipline_summary)


In [ ]:
# Check repeat incidents

discipline_student_counts = (
    elem_discipline.groupby(["school_year", "school_id", "student_id"])
    .agg(incidents=("incident_id", "size"), removal_days=("days_removed", "sum"))
    .reset_index()
)

repeat_incident_summary = (
    discipline_student_counts.groupby(["school_year", "school_id"])
    .agg(
        students_with_incident=("student_id", "size"),
        students_with_2plus_incidents=("incidents", lambda x: (x >= 2).sum()),
    )
    .reset_index()
)

repeat_incident_summary["repeat_share"] = (
    repeat_incident_summary["students_with_2plus_incidents"]
    / repeat_incident_summary["students_with_incident"]
)

display(repeat_incident_summary)


In [ ]:
# Keep elementary teacher assignments

elem_teacher_assignments = teacher_assignments.loc[
    teacher_assignments["school_id"].isin(ELEMENTARY_SCHOOL_IDS)
    & teacher_assignments["school_year"].isin(ANALYSIS_YEARS)
].merge(
    schools[["school_id", "school_name"]],
    on="school_id",
    how="left",
    validate="many_to_one",
)


In [ ]:
# Summarize recorded assignment FTE

staffing_summary = (
    elem_teacher_assignments.groupby(["school_year", "school_id", "school_name"])
    .agg(
        recorded_teachers=("teacher_id", "nunique"),
        recorded_assignment_fte=("fte", "sum"),
    )
    .reset_index()
)


In [ ]:
student_denominator = (
    elem_context.groupby(["school_year", "school_id", "school_name"])
    .agg(enrolled_students=("student_id", "size"))
    .reset_index()
)

staffing_summary = staffing_summary.merge(
    student_denominator,
    on=["school_year", "school_id", "school_name"],
    how="left",
    validate="one_to_one",
)

staffing_summary["recorded_fte_per_100_enrolled_students"] = (
    staffing_summary["recorded_assignment_fte"]
    / staffing_summary["enrolled_students"]
    * 100
)

display(staffing_summary)


In [ ]:
# Review recorded assignment role mix

staffing_role_mix = (
    elem_teacher_assignments.groupby(["school_year", "school_name", "assignment"])
    .agg(recorded_teachers=("teacher_id", "nunique"), recorded_fte=("fte", "sum"))
    .reset_index()
)

display(staffing_role_mix)


In [ ]:
# Latest-year recorded role mix

latest_staffing_role_mix = (
    staffing_role_mix.loc[staffing_role_mix["school_year"].eq(2026)]
    .pivot(index="school_name", columns="assignment", values="recorded_fte")
    .fillna(0)
)

display(latest_staffing_role_mix)


In [ ]:
# Compact latest-year State-test view

latest_state_screen = state_screen.loc[state_screen["school_year"].eq(2026)][
    [
        "school_name",
        "grade_level",
        "subject",
        "opportunity_students",
        "tested_students",
        "participation_rate",
        "met_standard_rate",
    ]
].sort_values(["subject", "grade_level", "school_name"])

display(latest_state_screen)


In [ ]:
# Check recurrence of State-test outcomes

state_recurrence = (
    state_screen.groupby(["school_name", "grade_level", "subject"])
    .agg(
        years=("school_year", "nunique"),
        min_met_standard_rate=("met_standard_rate", "min"),
        max_met_standard_rate=("met_standard_rate", "max"),
        mean_met_standard_rate=("met_standard_rate", "mean"),
        min_participation_rate=("participation_rate", "min"),
    )
    .reset_index()
)

display(state_recurrence)


#### Which findings belong in the briefing?

| Topic | Finding | What limits the conclusion | Follow-up for the LD | Briefing role |
|---|---|---|---|---|
| School context | Enrollment grew 9.0% at Cedar Ridge and 9.9% at Lincoln Park; it fell 2.5% at Harper Creek and 5.0% at Meadowbrook. Lincoln Park has the largest recorded EL share (15.7%). SPED shares are 15.7% at Meadowbrook and 15.2% at Lincoln Park. | Counts include anyone enrolled during the year. Labels are fixed, and the two largest SPED shares differ by only about 0.5 percentage points. No incoming roster is supplied. | Verify incoming enrollment and support arrangements, particularly at the growing schools. | Main context |
| Benchmark learning | Spring status varies across schools. Median paired change is positive in every latest-year school-grade-subject group. Some Fall differences are already visible. | Spring status and change use different samples. Overall rates reflect student mix. There is no expected-growth target. | Discuss starting needs and supports; review adjusted differences before judging performance. | Main analysis |
| Attendance | All school rates improved. Latest rates are 5.20% to 5.73%, with different grades higher within each school. Absence relates negatively to Fall and Spring score levels, but weakly to change. | Cutoffs are exploratory. Enrollment lengths vary. Correlations pool three years and cannot establish cause. | Check current concerns and whether attendance and academic needs are followed up together. | Follow-up alongside completed adjusted analysis |
| State tests | Grade 5 ELA proficiency among tested students ranges from 41.0% to 53.2% at Lincoln Park and 49.1% to 50.0% at Meadowbrook over three years. Latest Science ranges from 50.0% to 71.0%. | Elementary state tests cover grades 3-5 only. Science school ordering differs in earlier years. Participation opportunities are estimated. Former grade 5 cohorts have moved on. | Ask about upper-elementary literacy and preparation for middle school. Verify Science context if relevant. | Literacy supports the main story; Science is a backup finding |
| Discipline | Latest incident rates are 9.85 to 12.89 per 100 ever-enrolled students; 7.2% to 8.9% have an incident. Meadowbrook's removal rate is highest at 0.49 days per 1,000 enrolled days. | Counts are modest. Incidents differ from students involved. Recording practices and incident context are unknown. | Ask for context if principals raise a concern; rates alone cannot describe safety or climate. | Supporting context |
| Staffing | Latest recorded FTE per 100 ever-enrolled students is 5.05 at Lincoln Park and 7.59 at Harper Creek. Lincoln Park and Meadowbrook have no recorded SPED assignment label. | Assignments do not capture class size or necessarily all services. Fourteen employed teacher-year periods lack assignments across the extract. | Check shared staff, services, role coding, and missing records before drawing resource conclusions. | Verification question, especially alongside enrollment growth |

School context, benchmark learning, and attendance cover all elementary grades and answer the three linked questions. Recurring grade 5 literacy results add a focused academic point. Discipline and staffing remain available as context. Section 3.10 supplies the focused learning-path findings for the main narrative. Final slide choices follow review of these findings and the completed models.


### 3.9 What the EDA answers, and what models should add

| Question | What we know | What we need to allow for | Remaining question |
|---|---|---|---|
| Q1: Who do the schools serve? | Enrollment changes, student needs, and movement during the year differ. Latest enrollment ranges from 325 to 376. All absence rates improved; latest school rates are 5.20% to 5.73%. | Annual enrollment counts and fixed labels do not establish next-fall demand. School averages can hide grade patterns. | Which current enrollment and support arrangements should the LD verify? Which context belongs in the learning comparison? |
| Q2: What do starting points and Spring results show? | Latest Spring status is 53.6% to 61.3% in Reading and 52.8% to 62.2% in Mathematics. Median paired changes are positive in every latest-year school-grade-subject group. Some differences start at Fall. | Grade mix, Fall achievement, and who has tests affect comparisons. Positive change does not establish that growth met a target. | Do school differences remain after accounting for Fall score, grade, and year? Are the conclusions similar after adding selected background labels in a sensitivity analysis? |
| Q3: What does attendance add? | Higher-absence grades vary by school. Within-grade absence correlates modestly with Fall and Spring levels (about -0.23 to -0.15), but weakly with change (-0.06 to +0.05). | Fall differences already exist. Correlation cannot establish cause; high-absence groups are smaller. | Is absence still related to Spring scores after adjustment? How large and uncertain is the relationship? |

Decisions carried into the models:

- Use the checked context and paired tables, with Reading and Mathematics separate. Complete pairs have valid attendance intervals.
- Choose the sample and explanatory variables based on the question, before looking for a stronger result. Fixed labels do not provide yearly eligibility histories.
- Account for repeated students across years. Comparisons describe these four schools under the chosen model; they do not establish a general causal school effect.
- Compare raw and adjusted results with uncertainty. Small or uncertain adjusted differences are valid findings too.
- Keep the populations clear: all enrolled students for attendance, Spring-tested students for status, and both-tested students for change.

Section 4 reports the fitted primary models, diagnostic review, and sensitivity checks. Section 3.10 then adds a focused descriptive question about student status paths and overlapping support needs. The analysis sequence follows the questions; a small model association does not erase the learning needs in the student records.


### 3.10 Follow the same students: change and continuing needs

QC confirmed unique student-year-subject testing windows and consistent enrollment joins. That lets me follow the same student across windows. This analysis deepens Q2 and Q3. The EDA showed positive median score change in every latest-year school-grade-subject group. The models found small, uncertain overall school contrasts and a small additional absence association. I now ask who crossed the reported Benchmark boundary, who was below in both windows, and where those patterns overlap across subjects and with attendance.

Four paths use the supplied performance labels: At/Above in both windows; Below in both windows; Below to At/Above; At/Above to Below. "Below" includes Below and Well Below Benchmark. "Persistent below" is shorthand for below at the two observed endpoints. It does not establish that a student remained below throughout the year, received an intervention, or failed to improve.

The headline view uses 2025-26. Tables also show all three years. Unknown tests and missing levels stay visible. Subject transitions use paired students; the combined view returns to one student-year per enrollment and keeps students with incomplete information.

The data dictionary describes a vertical growth scale and four Benchmark labels, but provides no numerical cut scores by grade or window and no expected-growth target. I use recorded labels directly and distinguish status movement from score growth. The [methodology](../docs/methodology.md) summarizes the definitions used here.


#### 3.10A Who changed status, and who remained below?

First show the denominators, then the paths. The percentage still below among Fall-below students uses a different denominator from the percentage below in both windows among all pairs. Neither is a percentage of all enrolled students.


In [ ]:
learning_paths = make_learning_paths(benchmark_pairs)
learning_latest_year = int(elem_context.school_year.max())
learning_latest_label = elem_context.loc[
    elem_context.school_year.eq(learning_latest_year), "school_year_label"
].iloc[0]
latest_learning_paths = learning_paths.loc[
    learning_paths.school_year.eq(learning_latest_year)
].copy()
learning_annual_summary = summarize_learning_paths(
    learning_paths, ["school_year", "subject"]
)
learning_latest_summary = summarize_learning_paths(latest_learning_paths, ["subject"])
learning_school_summary = summarize_learning_paths(
    latest_learning_paths, ["subject", "school_name"]
)
learning_grade_summary = summarize_learning_paths(
    latest_learning_paths, ["subject", "school_name", "grade_level"]
)
assert (
    learning_latest_summary.classified_pairs
    + learning_latest_summary.incomplete_tests
    + learning_latest_summary.missing_levels
).equals(learning_latest_summary.enrolled_subject_rows)
display(learning_latest_summary.style.format(precision=1))
display(learning_annual_summary.style.format(precision=1))
display(learning_school_summary.style.format(precision=1))

classified = latest_learning_paths.loc[latest_learning_paths.levels_complete].copy()
learning_level_transitions = (
    classified.groupby(
        ["subject", "fall_performance_level", "spring_performance_level"]
    )
    .size()
    .rename("students")
    .reset_index()
)
persistent = classified.loc[classified.persistent_below.fillna(False)].copy()
persistent_score_summary = (
    persistent.groupby("subject")
    .agg(
        students=("student_id", "size"),
        median_score_change=("score_change", "median"),
        positive_score_change=("score_change", lambda scores: int(scores.gt(0).sum())),
        improved_level=("level_improved", "sum"),
    )
    .reset_index()
)
persistent_score_summary["positive_score_change_pct"] = (
    100
    * persistent_score_summary.positive_score_change
    / persistent_score_summary.students
)
learning_starting_level_summary = (
    classified.loc[
        classified.fall_performance_level.isin(
            ["Well Below Benchmark", "Below Benchmark"]
        )
    ]
    .groupby(["subject", "fall_performance_level"])
    .agg(
        students=("student_id", "size"),
        still_below=("persistent_below", "sum"),
    )
    .reset_index()
)
learning_starting_level_summary["still_below_pct"] = (
    100
    * learning_starting_level_summary.still_below
    / learning_starting_level_summary.students
)
display(persistent_score_summary.style.format(precision=1))
display(learning_starting_level_summary.style.format(precision=1))


In [ ]:
# Read the two panels in order; their denominators are deliberately different.
learning_progress_figure, axes = plt.subplots(1, 2, figsize=(13, 5.4))
subjects = ["Reading", "Mathematics"]
path_rows = learning_latest_summary.set_index("subject").loc[subjects]
score_rows = persistent_score_summary.set_index("subject").loc[subjects]
for ax, left_counts, right_counts, left_label, right_label, title in [
    (
        axes[0],
        path_rows.persistent_below,
        path_rows.moved_to_at_above,
        "Still below in Spring",
        "Reached At/Above",
        "1. Among students below in Fall",
    ),
    (
        axes[1],
        score_rows.positive_score_change,
        score_rows.students - score_rows.positive_score_change,
        "Positive score change",
        "Flat or negative change",
        "2. Among students below at both windows",
    ),
]:
    total = left_counts + right_counts
    rate = left_counts / total
    ax.barh(subjects, rate, color="#405781", label=left_label)
    ax.barh(subjects, 1 - rate, left=rate, color="#91c8bd", label=right_label)
    for i, (first, second, n_students, share) in enumerate(
        zip(left_counts, right_counts, total, rate)
    ):
        ax.text(
            share / 2,
            i,
            f"{int(first)}/{int(n_students)}\n{share:.1%}",
            ha="center",
            va="center",
            color="white",
            fontsize=10,
        )
        ax.text(
            share + (1 - share) / 2,
            i,
            f"{int(second)}\n{1-share:.1%}",
            ha="center",
            va="center",
            color="#183338",
            fontsize=9,
        )
    ax.set(xlim=(0, 1), xlabel="Share of the students in this panel", title=title)
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.invert_yaxis()
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.21), frameon=False, fontsize=9)
learning_progress_figure.suptitle(
    f"Scores often rose while below-Benchmark status continued, {learning_latest_label}",
    fontsize=15,
)
learning_progress_figure.text(
    0.08,
    0.025,
    "Observed paired students. Positive score change does not establish expected growth; no growth target is supplied.",
    fontsize=9,
)
learning_progress_figure.subplots_adjust(
    top=0.81, bottom=0.3, left=0.08, right=0.97, wspace=0.28
)
plt.show()


In [ ]:
benchmark_levels = [
    "Well Below Benchmark",
    "Below Benchmark",
    "At Benchmark",
    "Above Benchmark",
]
level_tick_labels = ["Well Below", "Below", "At", "Above"]
transition_figure, axes = plt.subplots(1, 2, figsize=(13, 6.4))
for ax, subject in zip(axes, ["Reading", "Mathematics"]):
    data = classified.loc[classified.subject.eq(subject)]
    counts = pd.crosstab(
        data.fall_performance_level, data.spring_performance_level
    ).reindex(index=benchmark_levels, columns=benchmark_levels, fill_value=0)
    row_rates = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0)
    image = ax.imshow(row_rates, vmin=0, vmax=1, cmap="YlGnBu")
    for row in range(4):
        for column in range(4):
            rate = row_rates.iloc[row, column]
            text = (
                f"{counts.iloc[row, column]}\n({rate:.0%})" if pd.notna(rate) else "-"
            )
            ax.text(
                column,
                row,
                text,
                ha="center",
                va="center",
                fontsize=10,
                color="white" if rate > 0.50 else "#183338",
            )
    ax.set(
        xticks=range(4),
        xticklabels=level_tick_labels,
        yticks=range(4),
        yticklabels=level_tick_labels,
        xlabel="Spring Benchmark level",
        ylabel="Fall Benchmark level",
        title=f"{subject}: {len(data):,} paired students",
    )
    bar = transition_figure.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    bar.ax.yaxis.set_major_formatter(PercentFormatter(1))
transition_figure.suptitle(
    f"Where students moved between reported levels, {learning_latest_label}",
    fontsize=15,
)
transition_figure.text(
    0.5,
    0.035,
    "Each cell shows students and % within its Fall row. Paired students only; missing tests are reported separately.",
    ha="center",
    fontsize=9,
)
transition_figure.subplots_adjust(top=0.84, bottom=0.16, wspace=0.34)
plt.show()


In [ ]:
school_order = (
    elem_context[["school_id", "school_name"]]
    .drop_duplicates()
    .sort_values("school_id")
    .school_name.tolist()
)
short_school_names = [name.replace(" Elementary", "") for name in school_order]
persistent_grade_figure, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, subject in zip(axes, ["Reading", "Mathematics"]):
    data = learning_grade_summary.loc[learning_grade_summary.subject.eq(subject)]
    rate = data.pivot(
        index="school_name",
        columns="grade_level",
        values="persistent_pct_of_fall_below",
    ).reindex(index=school_order, columns=range(6))
    still = data.pivot(
        index="school_name", columns="grade_level", values="persistent_below"
    ).reindex(index=school_order, columns=range(6))
    start = data.pivot(
        index="school_name", columns="grade_level", values="fall_below"
    ).reindex(index=school_order, columns=range(6))
    image = ax.imshow(rate / 100, vmin=0, vmax=1, cmap="YlGnBu", aspect="auto")
    for row in range(4):
        for column in range(6):
            value = rate.iloc[row, column]
            label = (
                f"{int(still.iloc[row,column])}/{int(start.iloc[row,column])}\n{value:.0f}%"
                if pd.notna(value)
                else "-"
            )
            ax.text(
                column,
                row,
                label,
                ha="center",
                va="center",
                fontsize=9,
                color="white" if value > 50 else "#183338",
            )
    ax.set(
        xticks=range(6),
        xticklabels=["K", "1", "2", "3", "4", "5"],
        yticks=range(4),
        yticklabels=short_school_names,
        xlabel="Historical grade in 2025-26",
        title=subject,
    )
    bar = persistent_grade_figure.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    bar.ax.yaxis.set_major_formatter(PercentFormatter(1))
persistent_grade_figure.suptitle(
    "Among Fall-below students, many remained below in Spring", fontsize=15
)
persistent_grade_figure.text(
    0.5,
    0.025,
    "Each cell: still below / Fall below, then %. Paired students only. Starting severity differs; this is not an effectiveness ranking.",
    ha="center",
    fontsize=9,
)
persistent_grade_figure.subplots_adjust(
    top=0.83, bottom=0.17, left=0.13, right=0.95, wspace=0.5
)
plt.show()


#### What the learning paths show

| 2025-26, paired students | Reading | Mathematics |
|---|---:|---:|
| Fall below Benchmark | 539 | 516 |
| Below at both Fall and Spring | 443 / 539 (82.2%) | 429 / 516 (83.1%) |
| Moved from below to At/Above | 96 / 539 (17.8%) | 87 / 516 (16.9%) |
| Positive score change among those below in both windows | 401 / 443 (90.5%) | 399 / 429 (93.0%) |
| Median score change among those below in both windows | +10 points | +11 points |

Many students improved their scores and still ended below the reported Spring Benchmark. This makes continuing academic support a clearer briefing topic than average growth alone. It does not establish that growth was insufficient: the file supplies no expected-growth target or window-specific cut scores.

Starting level matters. Of Fall Well Below students, 228/235 (97.0%) in Reading and 218/222 (98.2%) in Mathematics were still below in Spring. Among Fall Below students, the corresponding rates were 215/304 (70.7%) and 211/294 (71.8%). These are descriptive comparisons of different starting groups. They support asking how schools monitor students with the lowest starting levels, rather than treating every below-Benchmark student as having the same needs.

Even within the below-in-both group, 49 Reading and 61 Mathematics students moved from Well Below to Below. The four-level matrix preserves that improvement. Avoid describing the group as having made no progress.

The broad persistence pattern occurs in all three years: 82.2%-83.5% for Reading and 80.3%-83.1% for Mathematics among paired Fall-below students. These yearly figures involve overlapping students and changing cohorts. They describe recurring needs, not three independent replications or an intervention effect.

The school-grade heatmap shows needs across the four schools. Cell denominators are only 11-32 Fall-below students, so a few students can change a percentage noticeably. Use counts, starting severity, and current records together; do not select the highest percentage as a school-performance verdict. For example, Lincoln Park's historical grade 3 has 30/32 Reading and 27/31 Mathematics Fall-below students still below in Spring. This is a specific group to verify, not proof that grade 3 instruction caused the pattern.

Coverage remains visible: 170 Reading and 179 Mathematics student records lack a complete pair in 2025-26. None of the paired score records has a missing performance label. The transition rates describe the 1,231 Reading and 1,222 Mathematics pairs, not all 1,401 enrolled students.

Section 4.11 now evaluates the starting-group contrast on all three years. The adjusted gap is about 26 percentage points in both subjects, with intervals excluding zero and both Holm-adjusted focal p values below .001. Keep those pooled model probabilities distinct from the latest-year descriptive percentages above.


#### 3.10B How do learning and attendance needs overlap?

Join Reading and Mathematics back to one row per student in 2025-26. A student can count once as having a persistent-below pattern in both subjects. A missing subject prevents a complete two-subject classification, while any known persistent-below pattern remains visible for the attendance summary.

Use annual absence here to describe enrolled-day support needs, with the same exploratory 10% cutoff used in the EDA. This differs from the between-test predictor in Model 2. A student below 10% absence may still have attendance needs; the cutoff is not a diagnosis or a rule for services.


In [ ]:
joint_learning_needs = make_joint_needs(
    learning_paths, elem_context, learning_latest_year
)
assert (
    len(joint_learning_needs) == elem_context.school_year.eq(learning_latest_year).sum()
)
assert not joint_learning_needs.duplicated(["student_id", "school_year"]).any()
need_order = [
    "Persistent below in both subjects",
    "Persistent below in Reading only",
    "Persistent below in Mathematics only",
    "No persistent-below pattern in either subject",
    "Incomplete paired information",
]
joint_academic_summary = (
    joint_learning_needs.academic_need.value_counts()
    .reindex(need_order, fill_value=0)
    .rename("students")
    .rename_axis("academic_need")
    .reset_index()
)
joint_academic_summary["pct_of_enrolled"] = (
    100 * joint_academic_summary.students / len(joint_learning_needs)
)
joint_school_summary = pd.crosstab(
    joint_learning_needs.school_name, joint_learning_needs.academic_need
).reindex(index=school_order, columns=need_order, fill_value=0)
joint_school_summary["enrolled_students"] = joint_school_summary.sum(axis=1)
joint_academic_attendance = (
    joint_learning_needs.groupby(["academic_need", "attendance_band"])
    .size()
    .rename("students")
    .reset_index()
)
identified_persistent_students = joint_learning_needs.loc[
    joint_learning_needs.any_persistent_identified
].copy()
attendance_order = [
    "Below 10% absence",
    "10% or more absence",
    "Attendance unavailable",
]
persistent_attendance_summary = (
    identified_persistent_students.groupby("attendance_band")
    .agg(
        students=("student_id", "size"),
        median_enrolled_days=("enrolled_instructional_days", "median"),
    )
    .reindex(attendance_order)
    .reset_index()
)
persistent_attendance_summary["students"] = (
    persistent_attendance_summary.students.fillna(0).astype(int)
)
persistent_attendance_summary["pct_of_identified_students"] = (
    100 * persistent_attendance_summary.students / len(identified_persistent_students)
    if len(identified_persistent_students)
    else np.nan
)
display(joint_academic_summary.style.format(precision=1))
display(joint_school_summary)
display(persistent_attendance_summary.style.format(precision=1))
print(
    "Known persistent-below pattern with incomplete two-subject information:",
    int(
        (
            joint_learning_needs.any_persistent_identified
            & ~joint_learning_needs.two_subject_pairs
        ).sum()
    ),
)


In [ ]:
# All enrolled students remain represented, including incomplete paired information.
support_overlap_figure, ax = plt.subplots(figsize=(12, 5.7))
colors = ["#405781", "#237f78", "#66a6ba", "#dfe6e5", "#b5b9bf"]
legend_labels = [
    "Below in both subjects at both windows",
    "Reading pattern only",
    "Mathematics pattern only",
    "No pattern in either subject",
    "Incomplete paired information",
]
left = np.zeros(len(school_order))
for column, color, label in zip(need_order, colors, legend_labels):
    counts = joint_school_summary[column].to_numpy()
    bars = ax.barh(short_school_names, counts, left=left, color=color, label=label)
    for i, (bar, count) in enumerate(zip(bars, counts)):
        if count >= 16:
            ax.text(
                left[i] + count / 2,
                bar.get_y() + bar.get_height() / 2,
                str(count),
                ha="center",
                va="center",
                fontsize=10,
                color="white" if color in colors[:2] else "#183338",
            )
    left += counts
for i, total in enumerate(left):
    ax.text(total + 4, i, f"n={int(total)}", va="center", fontsize=9)
ax.set(
    xlabel="Students, counted once per enrollment",
    title=f"Where subject needs overlap, {learning_latest_label}",
)
ax.invert_yaxis()
ax.set_xlim(0, left.max() * 1.15)
ax.legend(
    loc="upper left", bbox_to_anchor=(0, -0.15), frameon=False, ncol=2, fontsize=9
)
support_overlap_figure.text(
    0.12,
    0.015,
    'A pattern means below at both Fall and Spring. "No pattern" does not mean no learning needs. Counts are historical.',
    fontsize=9,
)
support_overlap_figure.subplots_adjust(left=0.18, right=0.95, top=0.87, bottom=0.29)
plt.show()

persistent_attendance_figure, ax = plt.subplots(figsize=(10, 4.6))
rows = persistent_attendance_summary
bars = ax.barh(
    rows.attendance_band, rows.students, color=["#237f78", "#c38752", "#b5b9bf"]
)
for bar, row in zip(bars, rows.itertuples()):
    pct = (
        f" ({row.pct_of_identified_students:.1f}%)"
        if pd.notna(row.pct_of_identified_students)
        else ""
    )
    ax.text(
        bar.get_width() + 4,
        bar.get_y() + bar.get_height() / 2,
        f"{row.students}{pct}",
        va="center",
    )
ax.set(
    xlabel="Unique students with a known persistent-below pattern in at least one subject",
    title="Continuing learning needs occur across attendance groups",
)
ax.set_xlim(0, max(rows.students.max(), 1) * 1.27)
ax.invert_yaxis()
persistent_attendance_figure.text(
    0.08,
    0.025,
    "Annual enrolled-day absence; exploratory 10% cutoff. This describes overlap, not an attendance effect.",
    fontsize=9,
)
persistent_attendance_figure.subplots_adjust(
    left=0.23, right=0.95, top=0.83, bottom=0.18
)
plt.show()


#### What the overlap adds

The combined table counts each of the 1,401 enrolled students once. It identifies 254 students below at both windows in both subjects. Another 153 have the pattern in Reading only and 135 in Mathematics only, with complete two-subject information. These counts sum to 542 students with at least one persistent-below pattern among the 1,120 students whose two-subject paths are complete.

There are 281 students (20.1%) with incomplete two-subject paths. Among them, 76 still have an observed persistent-below pattern in the subject with a complete pair. Adding those students gives 618 students with a known persistent-below pattern in at least one subject. The incomplete group remains separate in the subject chart; it is not assigned to "no needs." Incomplete paths do not necessarily mean missing Spring tests, and partial-year enrollment can limit testing opportunity.

Among those 618 students, 475 (76.9%) had annual absence below 10%, and 143 (23.1%) had absence of at least 10%. Attendance is available for all 618. Each band has a median of 185 enrolled instructional days. Most of these identified learning needs therefore fall outside the exploratory high-absence band.

For the LD, an academic follow-up based only on high absence would miss many students in this observed group. The 143 students with both patterns are a reason to ask about coordinated attendance and academic support. Below 10% absence does not guarantee good attendance, and this overlap does not show that absence is irrelevant or estimate its causal effect.

Students with no persistent-below pattern can still need support. For example, some moved below Benchmark in Spring. The four paths and all-Spring-tested results should accompany the combined view.


#### 3.10C What should be checked before the new school year?

Use historical K-4 records as a starting point for verifying learning needs with the new roster. "Potential next grade" adds one to the recorded grade as a planning label. The data do not show promotion, return to the school, or the coming fall enrollment. Current Kindergarten entrants are absent from this planning view. Former grade 5 students have a separate transition-to-middle-school summary.

Known persistent-below counts may include students with only one subject fully paired. The table also reports incomplete two-subject information. These are counts to verify, not a forecast, eligibility decision, or resource-allocation rule.


In [ ]:
planning_source = joint_learning_needs.loc[
    joint_learning_needs.grade_level.le(4)
].copy()
planning_source["persistent_both_subjects"] = planning_source.academic_need.eq(
    "Persistent below in both subjects"
)
planning_source["incomplete_information"] = ~planning_source.two_subject_pairs
planning_cohort_summary = (
    planning_source.groupby(["school_name", "grade_level", "potential_next_grade"])
    .agg(
        historical_students=("student_id", "size"),
        complete_two_subject_information=("two_subject_pairs", "sum"),
        identified_persistent_any=("any_persistent_identified", "sum"),
        persistent_both_subjects=("persistent_both_subjects", "sum"),
        persistent_with_absence_ge_10pct=("overlapping_identified_needs", "sum"),
        incomplete_two_subject_information=("incomplete_information", "sum"),
    )
    .reset_index()
)
planning_school_summary = (
    planning_source.groupby("school_name")
    .agg(
        historical_k4_students=("student_id", "size"),
        identified_persistent_any=("any_persistent_identified", "sum"),
        persistent_both_subjects=("persistent_both_subjects", "sum"),
        persistent_with_absence_ge_10pct=("overlapping_identified_needs", "sum"),
        incomplete_two_subject_information=("incomplete_information", "sum"),
    )
    .reset_index()
)
grade5_transition_summary = (
    joint_learning_needs.loc[joint_learning_needs.grade_level.eq(5)]
    .groupby("school_name")
    .agg(
        historical_grade5_students=("student_id", "size"),
        identified_persistent_any=("any_persistent_identified", "sum"),
        persistent_with_absence_ge_10pct=("overlapping_identified_needs", "sum"),
    )
    .reset_index()
)
display(planning_cohort_summary)
display(planning_school_summary)
display(grade5_transition_summary)


#### What belongs in the briefing

The historical K-4 group contains 1,159 students. At least one persistent-below pattern is identified in 513; 206 have the pattern in both subjects, and 117 also have annual absence of at least 10%. Another 237 students have incomplete two-subject paths; some are included in the 513 because their other subject has a known pattern. These categories overlap and should not be added.

| School | Historical K-4 students | Identified pattern in at least one subject | Pattern in both subjects | Identified pattern + absence at least 10% | Incomplete two-subject paths |
|---|---:|---:|---:|---:|---:|
| Cedar Ridge | 301 | 115 | 43 | 26 | 76 |
| Harper Creek | 283 | 131 | 51 | 36 | 52 |
| Lincoln Park | 309 | 148 | 58 | 35 | 57 |
| Meadowbrook | 266 | 119 | 54 | 20 | 52 |

Use these as counts to verify with the incoming roster. They are not projected enrollment or prescribed caseloads. Lincoln Park's historical grade 3 is one useful example: 42/75 students have an identified pattern in at least one subject, including 15 with both and 10 with overlapping high absence; 16 have incomplete paths. If those students return and advance, this informs a conversation about grade 4. It does not describe the actual incoming grade 4 roster.

The 105 identified students in historical grade 5 belong in a separate middle-school transition conversation. New Kindergarten students, transfers, promotion decisions, and return to the same school are not observed in the coming year.

**Briefing focus:** students often gained score points while below-Benchmark status continued. Identify subject needs first, distinguish the lowest starting levels, and then check which students also have attendance barriers. The main figures are the progress/status overview and the overlap charts. The detailed transition matrix and school-grade heatmap support follow-up or appendix discussion.

The original models remain useful context: they do not support a clear pooled school ranking, and the additional absence association is small. Section 4.11 evaluates the stated follow-up question about starting groups, with school, grade, and year adjustment. It is exploratory and the original OLS models remain unchanged. This focused analysis describes support needs; deciding whether a different intervention helps would require additional evidence.


In [ ]:
# Aggregate exports for briefing review. Student IDs are not exported.
learning_results_dir = DATA_DIR.parent.parent / "learning_results"
learning_results_dir.mkdir(exist_ok=True)
for filename, table in {
    "latest_paths.csv": learning_latest_summary,
    "annual_paths.csv": learning_annual_summary,
    "school_paths.csv": learning_school_summary,
    "grade_paths.csv": learning_grade_summary,
    "level_transitions.csv": learning_level_transitions,
    "persistent_score_change.csv": persistent_score_summary,
    "starting_level_summary.csv": learning_starting_level_summary,
    "joint_subject_summary.csv": joint_academic_summary,
    "joint_academic_attendance.csv": joint_academic_attendance,
    "persistent_attendance.csv": persistent_attendance_summary,
    "planning_cohorts.csv": planning_cohort_summary,
    "planning_schools.csv": planning_school_summary,
    "grade5_transition.csv": grade5_transition_summary,
}.items():
    table.to_csv(learning_results_dir / filename, index=False)
joint_school_summary.to_csv(learning_results_dir / "joint_school_summary.csv")
for filename, figure in {
    "learning_progress_and_status": learning_progress_figure,
    "fall_spring_status_transitions": transition_figure,
    "persistent_below_by_grade": persistent_grade_figure,
    "subject_needs_overlap": support_overlap_figure,
    "learning_needs_and_attendance": persistent_attendance_figure,
}.items():
    figure.savefig(
        learning_results_dir / f"{filename}.png", dpi=220, bbox_inches="tight"
    )
    figure.savefig(learning_results_dir / f"{filename}.svg", bbox_inches="tight")
print(f"Aggregate tables and five figures saved to {learning_results_dir.resolve()}")


## 4. Statistical models and assumption checks

**Status: complete.** Four primary OLS models were fitted, their diagnostics reviewed, and two focused sensitivity checks completed. Section 4.11 adds two exploratory logistic models for the starting-group question, with their own binary-outcome checks and stability review. The prespecified OLS specifications were retained after the plots and targeted specification checks. Section 4.7A checks Fall slopes and curvature; Section 4.11 also checks starting-group interactions. The reported interpretations remain consistent with these checks. This is evidence for using the models within the stated historical scope, not a claim that every assumption has been proved.

The sequence is: define comparable samples, describe unadjusted relationships, allow for Fall starting points, add between-test absence, then inspect fit and stability. These are adjusted associations in historical paired students, not estimates of school effectiveness or intervention effects.

| Step | Work completed | Output |
|---|---|---|
| 4.1-4.2 | Questions and prespecified formulas | Comparison rules and limits |
| 4.3 | Common samples for each subject | Counts and exclusions |
| 4.3A | Univariable preliminary analysis | Clustered intervals and whole-predictor tests; no variable selection |
| 4.4 | Model 1 | Adjusted school contrasts |
| 4.5 | Model 2 | Additional association with between-test absence |
| 4.6-4.7 | Diagnostic review | Plots, decisions, and remaining limits |
| 4.8 | Influence and background-label sensitivity | Stability of the main findings |
| 4.9-4.10 | Briefing summary | Two exported figures, result tables, and LD follow-up |


### 4.1 Why these models follow the QC and EDA

QC established which records can be joined, who has both tests, and how attendance should be counted. It also showed that students can appear in several years and that program labels are fixed in the student file. These findings determine the sample and how I handle uncertainty.

The EDA showed some school differences at Fall, alongside differences at Spring. It also showed that absence relates to both Fall and Spring scores, while its raw relationship with score change is weak. A Spring comparison therefore needs to allow for the Fall starting point.

| Remaining question | Model | Why it matters for the LD |
|---|---|---|
| Q2: Do Spring school differences remain after allowing for Fall score, grade, and year? | Model 1 | Helps the LD interpret school comparisons alongside the students' starting points |
| Q3: Is between-test absence associated with Spring score after the same adjustment? | Model 2 | Helps the LD judge whether attendance and academic needs warrant coordinated follow-up |

These models describe adjusted associations among students with both tests. They cannot identify the causal effect of a school or estimate what an attendance intervention would achieve.

### 4.2 Model plan before fitting

Start with ordinary least squares (OLS), fitted separately for Reading and Mathematics. Use standard errors clustered by `student_id` because the same student can appear in several years. Treat school, grade, and school year as categories.

| Model | Outcome | Explanatory variables |
|---|---|---|
| Model 1 | `spring_score` | `fall_score` + school + grade + school year |
| Model 2 | `spring_score` | The same variables + between-test absence |

Use the same rows for Models 1 and 2 within each subject so a change in the results is not caused by a change in the sample. Confirm all required fields before fitting. Choose and record the school reference category explicitly.

For Model 2, express absence in units of **5 percentage points**. For a rate stored from 0 to 1, divide it by 0.05. The coefficient then describes the adjusted score difference associated with, for example, 10% rather than 5% absence, where the observed data support that comparison.

Selected EL, economic-disadvantage, and SPED labels can be added in a sensitivity analysis if they answer a stated concern. They are fixed labels, and adding them changes what the school comparison means. Record that decision before reviewing the alternative results.

Student clustering allows dependence within a student's records. It still assumes independence across students; shared classroom or school conditions may challenge that assumption. Four schools provide too few school clusters for a routine school-clustered analysis. Keep school comparisons specific to these four schools and state this limitation.

**Recorded decisions:**

- Reference school: Cedar Ridge (101). Reference grade: Kindergarten. Reference year: 2023-24 (coded 2024).
- Primary sample: all checked complete pairs with valid between-test attendance; no additional exclusions (4.3).
- Formulas: shown below. Fall score is uncentered; absence is divided by 0.05. The intercept is not a substantive finding.
- Background sensitivity: add all three recorded EL, economic-disadvantage, and SPED labels to assess whether the main results depend on the student mix shown in Q1. These are fixed labels, not annual program histories.

The intervals use the student-clustered covariance with its finite-sample correction and Student t reference based on the number of student clusters. Whole-factor tests use clustered Wald F tests. Ordinary independent-observation likelihood-ratio tests are not used. See [statsmodels robust covariance documentation](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.get_robustcov_results.html).

Univariable models in 4.3A show unadjusted associations. The primary variables are prespecified from Q2 and Q3, rather than selected using a univariable p-value cutoff. The prepared absence term is `absence_5pp`.


In [ ]:
# Prespecified formulas. School, grade, and year are categorical.
model_1_formula = (
    "spring_score ~ fall_score"
    " + C(school_id, Treatment(reference=101))"
    " + C(grade_level, Treatment(reference=0))"
    " + C(school_year, Treatment(reference=2024))"
)
model_2_formula = model_1_formula + " + absence_5pp"


### 4.3 Prepare the model samples

Use `benchmark_pairs`, which already contains the checked scores, school context, and between-test attendance. Keep complete Fall-Spring pairs and the fields required for both models. Do not replace a missing score or attendance rate with zero.

Before fitting, report rows and unique students separately, check the student-year-subject key, and show any exclusions by subject and school. Check the range of Fall scores and absence within school and grade. Adjusted comparisons need enough overlap between schools to avoid relying on extrapolation.

The current EDA has **3,653 Reading pairs and 3,606 Mathematics pairs**. These are reference counts before any additional model exclusions. Explain any change.

In [ ]:
# One common sample per subject, with absence measured in 5-percentage-point units.
regression_samples, regression_sample_audit = prepare_regression_samples(benchmark_pairs)
reading_model_data = regression_samples["Reading"]
math_model_data = regression_samples["Mathematics"]
display(regression_sample_audit)


In [ ]:
# Review category coverage and the number of records contributed by each student.
model_category_counts = (
    pd.concat(regression_samples.values())
    .groupby(["subject", "school_year", "school_name", "grade_level"])
    .size()
    .rename("model_rows")
    .reset_index()
)
display(model_category_counts)

for subject, data in regression_samples.items():
    records_per_student = data.groupby("student_id").size().value_counts().sort_index()
    print(f"{subject}: number of students with 1, 2, or 3 model records")
    display(records_per_student.rename("Students").to_frame())


#### Model sample notes

| Subject | Paired rows before selection | Common model rows | Unique students | Extra exclusions |
|---|---:|---:|---:|---:|
| Reading | 3,653 | 3,653 | 1,780 | 0 |
| Mathematics | 3,606 | 3,606 | 1,779 | 0 |

All paired records have the required fields and positive attendance intervals. Missing tests remain outside these models, as documented in the participation analysis. The common samples include every checked pair, and the source table is unchanged.

The sample covers all four schools, K-5, and three years. The category-count table shows 37-72 Reading records and 32-70 Mathematics records per school-year-grade cell. This is a coverage check, not evidence that starting-score distributions overlap adequately. Section 4.6A provides the predictor ranges and further overlap checks after fitting.

Use these samples for both primary models within a subject. Adjusted comparisons describe paired students rather than all enrolled or all Spring-tested students.


### 4.3A Preliminary analysis: univariable regression

I fit one predictor at a time to describe the unadjusted relationship with Spring score. Reading and Mathematics use separate samples. All five fits within a subject use the same rows as the planned primary models.

| Predictor | Interpretation | Reference or unit |
|---|---|---|
| Fall score | Spring score difference associated with a higher Fall starting point | Per 1 Fall-score point |
| School | Raw Spring score differences between school groups | Cedar Ridge (school 101) |
| Grade | Raw Spring score differences between grades | Kindergarten |
| School year | Raw Spring score differences between years | 2023-24 (`school_year=2024`) |
| Between-test absence | Raw Spring score difference associated with more absence | Per 5 percentage points |

School, grade, and year are categorical. The overall p value tests the whole predictor: for example, the three school contrasts together. All intervals and tests use student-clustered covariance, with a small-sample correction. Coefficient intervals use a t reference and joint tests use a Wald F reference based on the student clusters.

These are preliminary associations. Grade comparisons reflect different score levels and do not compare grade effectiveness. School contrasts also reflect grade mix, starting points, and year. Student clustering does not address dependence between different students who share classrooms or schools.

**No p-value screening is used.** Fall, school, grade, and year remain in Model 1. Model 2 adds the attendance variable that answers Q3. Background labels are considered separately if there is a substantive reason; no stepwise selection is planned. A p value above 0.2 does not remove a prespecified variable.

Method reference: [statsmodels term-level Wald tests](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.wald_test_terms.html).


In [ ]:
# Keep fitted objects for later comparisons; these are the univariable fits only.
univariable_models = {}
overall_tables = []
coefficient_tables = []

for subject, data in regression_samples.items():
    fits, overall, coefficients = fit_univariable_models(data, subject)
    univariable_models[subject] = fits
    overall_tables.append(overall)
    coefficient_tables.append(coefficients)

univariable_overall = pd.concat(overall_tables, ignore_index=True)
univariable_coefficients = pd.concat(coefficient_tables, ignore_index=True)

print("Overall predictor tests (student-clustered Wald F):")
display(univariable_overall.style.format({"wald_F": "{:.3f}", "p_value": lambda value: "<0.001" if value < 0.001 else f"{value:.3f}"}))
print("Coefficients and 95% student-clustered intervals; intercepts omitted:")
display(
    univariable_coefficients.style.format(
        {
            "estimate": "{:.3f}",
            "clustered_se": "{:.3f}",
            "ci_low": "{:.3f}",
            "ci_high": "{:.3f}",
            "p_value": lambda value: "<0.001" if value < 0.001 else f"{value:.3f}",
        }
    )
)


#### What the preliminary results show

The ten univariable fits use the same checked pairs within each subject. Their confidence intervals and overall tests account for repeated records from the same student.

| Preliminary relationship | Reading | Mathematics | Interpretation |
|---|---|---|---|
| 1 point higher Fall score | +0.789 Spring points; 95% CI 0.781 to 0.796 | +0.801; 95% CI 0.794 to 0.808 | Starting score is strongly associated with Spring score. This pooled slope also includes grade differences. |
| 5 percentage points more between-test absence | -2.724 Spring points; 95% CI -3.447 to -2.001 | -2.745; 95% CI -3.536 to -1.954 | Higher absence accompanies lower Spring scores before adjustment. This does not estimate an intervention benefit. |
| Overall school comparison | p = 0.778 | p = 0.658 | Raw pooled Spring scores do not show a clear school difference. This does not establish equivalence or remove school from Q2. |
| Overall grade comparison | p < 0.001 | p < 0.001 | Grade strongly relates to score level. Grade effectiveness cannot be inferred from these contrasts. |
| Overall year comparison | p = 0.012 | p = 0.015 | Spring scores differ modestly across the pooled yearly samples; this is not an adjusted time trend. |

Compared with 2023-24, the raw 2025-26 Spring difference is +2.346 points in Reading (95% CI 0.796 to 3.895) and +2.467 in Mathematics (0.794 to 4.139). Different student cohorts and mixes may contribute to these differences.

The school results are particularly useful for the modeling decision: school remains the comparison of interest even though its preliminary p values exceed 0.2. No prespecified variable is removed. These tests pool three years and all grades, use paired students, and analyze numeric Spring scores. The latest-year benchmark percentages and grade-specific patterns in the EDA describe other measures or scopes.

**Why the primary models are still needed:** Fall score, grade, and attendance are related to the students' starting context. The unadjusted absence estimate may change after accounting for that context. Model 1 examines adjusted school differences; Model 2 examines the adjusted absence association. The fitted results are reported below.

Small p values are reported as p < 0.001. A displayed computational zero is numerical underflow, not an exact probability of zero. Review effect sizes and intervals alongside p values. The primary-model diagnostics and substantive interpretation are reported in 4.6-4.9.


#### What to carry into the primary models

The univariable results describe raw associations. The next step answers different questions:

- **Model 1:** allow for Fall starting point, grade, and year when comparing schools.
- **Model 2:** add between-test absence to ask whether its association remains after that adjustment.

Use `reading_model_data` and `math_model_data` below. `absence_5pp` is already prepared, and the reference school for these preliminary results is Cedar Ridge. Keep the same sample when comparing raw and adjusted results. If you add background labels that introduce missing values, document the exclusions and refit the relevant comparison on that common sample.

Primary model fitting, assumption review, and interpretation are reported separately below. The preliminary models are not the final adjusted results.


### 4.4 Model 1: Spring differences after allowing for starting points

Fit Spring score on Fall score, school, grade, and year for each subject. Save the fitted model and its student-clustered covariance results for later tables and plots.

Review school contrasts with 95% confidence intervals. If displaying adjusted school means, predict each school over the **same supported mix** of Fall scores, grades, and years, then average. Give the averaging rule and use the clustered covariance for uncertainty. Avoid presenting each school's own student mix as a common adjusted comparison.

With several school contrasts, focus on the preplanned comparisons. If reporting all pairwise significance tests, account for multiple comparisons. A ranking alone is not a useful conclusion.

In [ ]:
reading_m1 = fit_primary_model(reading_model_data, model_1_formula)
reading_m1_coefficients, reading_m1_tests = primary_result_tables(
    reading_m1, "Reading", "Model 1"
)
display(reading_m1_coefficients.style.format({"p_value": lambda p: "< .001" if p < .001 else f"{p:.3f}"}, precision=3))
display(reading_m1_tests.style.format({"p_value": lambda p: "< .001" if p < .001 else f"{p:.3f}"}, precision=3))


In [ ]:
math_m1 = fit_primary_model(math_model_data, model_1_formula)
math_m1_coefficients, math_m1_tests = primary_result_tables(
    math_m1, "Mathematics", "Model 1"
)
display(math_m1_coefficients.style.format({"p_value": lambda p: "< .001" if p < .001 else f"{p:.3f}"}, precision=3))
display(math_m1_tests.style.format({"p_value": lambda p: "< .001" if p < .001 else f"{p:.3f}"}, precision=3))


In [ ]:
# These are score contrasts against Cedar Ridge, not school effectiveness rankings.

model_1_coefficients = pd.concat(
    [reading_m1_coefficients, math_m1_coefficients], ignore_index=True
)
model_1_school_contrasts = model_1_coefficients.loc[
    model_1_coefficients.term.str.startswith("C(school_id")
].copy()
model_1_school_contrasts["school"] = (
    model_1_school_contrasts.term.str.extract(r"\[T\.(\d+)\]", expand=False)
    .astype(int)
    .map(schools.set_index("school_id").school_name)
)
display(
    model_1_school_contrasts[
        ["subject", "school", "estimate", "ci_low", "ci_high", "p_value"]
    ]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, constrained_layout=True)
for ax, (subject, rows) in zip(
    axes, model_1_school_contrasts.groupby("subject", sort=False)
):
    ax.errorbar(
        rows.estimate,
        rows.school,
        xerr=[rows.estimate - rows.ci_low, rows.ci_high - rows.estimate],
        fmt="o",
        color="#227c77",
        capsize=4,
    )
    ax.axvline(0, color="gray", linestyle="--")
    ax.set(title=subject, xlabel="Adjusted Spring difference vs Cedar Ridge (points)")
fig.suptitle("Model 1: school contrasts with 95% student-clustered intervals")
school_contrast_figure = fig
plt.show()


#### Model 1 interpretation

Model 1 compares schools after allowing for Fall score, grade and school year. Its school contrasts against Cedar Ridge are small in score units:

| School vs Cedar Ridge | Reading difference (95% CI), points | Mathematics difference (95% CI), points |
|---|---:|---:|
| Harper Creek | +0.20 (-0.32 to +0.73) | +0.16 (-0.40 to +0.72) |
| Lincoln Park | -0.32 (-0.86 to +0.21) | -0.10 (-0.66 to +0.45) |
| Meadowbrook | +0.15 (-0.41 to +0.70) | +0.16 (-0.41 to +0.73) |

The whole-school-factor p values are .200 for Reading and .748 for Mathematics. Each pointwise interval includes zero. Under this specification and its student-cluster assumptions, these pooled paired data do not establish a clear adjusted difference against Cedar Ridge. They also do not establish that the schools are equivalent. The intervals are reported separately; the analysis does not test every possible school pair or produce an effectiveness ranking.

The assumption review retained this formula and its clustered uncertainty. Adding absence in Model 2 also leaves the school interpretation unchanged: all school contrast intervals still include zero, and the whole-school-factor p values are .166 and .706. These Model 2 p values belong to a different fit and should not be substituted for the Model 1 tests above.

The EDA's latest-year Benchmark percentages describe students with Spring results and use a different measure. Their gaps are useful for discussing grade and literacy needs, but they are not adjusted school effects. Principals should use current rosters and starting achievement to understand those needs.


### 4.5 Model 2: Attendance alongside Spring performance

Add between-test absence to Model 1, using the same sample and student-clustered standard errors. Annual absence covers a different period and population; use the between-test measure here.

Report the adjusted score difference associated with **5 percentage points more absence**, its 95% interval, and the observed range that supports the comparison. Review whether the school contrasts change after adding absence. A small or uncertain absence association is a valid result.

In [ ]:
reading_m2 = fit_primary_model(reading_model_data, model_2_formula)
reading_m2_coefficients, reading_m2_tests = primary_result_tables(
    reading_m2, "Reading", "Model 2"
)
display(reading_m2_coefficients.style.format({"p_value": lambda p: "< .001" if p < .001 else f"{p:.3f}"}, precision=3))
display(reading_m2_tests.style.format({"p_value": lambda p: "< .001" if p < .001 else f"{p:.3f}"}, precision=3))


In [ ]:
math_m2 = fit_primary_model(math_model_data, model_2_formula)
math_m2_coefficients, math_m2_tests = primary_result_tables(
    math_m2, "Mathematics", "Model 2"
)
display(math_m2_coefficients.style.format({"p_value": lambda p: "< .001" if p < .001 else f"{p:.3f}"}, precision=3))
display(math_m2_tests.style.format({"p_value": lambda p: "< .001" if p < .001 else f"{p:.3f}"}, precision=3))


In [ ]:
primary_models = {
    "Reading M1": (reading_m1, reading_model_data),
    "Reading M2": (reading_m2, reading_model_data),
    "Mathematics M1": (math_m1, math_model_data),
    "Mathematics M2": (math_m2, math_model_data),
}
primary_coefficients = pd.concat(
    [
        reading_m1_coefficients,
        reading_m2_coefficients,
        math_m1_coefficients,
        math_m2_coefficients,
    ],
    ignore_index=True,
)
primary_tests = pd.concat(
    [
        reading_m1_tests,
        reading_m2_tests,
        math_m1_tests,
        math_m2_tests,
    ],
    ignore_index=True,
)
primary_model_summary = pd.DataFrame(
    [
        {
            "model": name,
            "rows": int(result.nobs),
            "students": data.student_id.nunique(),
            "parameters": len(result.params),
            "R_squared": result.rsquared,
            "residual_SD": np.sqrt(result.mse_resid),
        }
        for name, (result, data) in primary_models.items()
    ]
)
display(primary_model_summary)

raw_absence = univariable_coefficients.loc[
    univariable_coefficients.term.eq("absence_5pp")
].copy()
raw_absence["model"] = "Unadjusted"
adjusted_absence = primary_coefficients.loc[
    primary_coefficients.term.eq("absence_5pp")
].copy()
attendance_comparison = pd.concat(
    [
        raw_absence[["subject", "model", "estimate", "ci_low", "ci_high", "p_value"]],
        adjusted_absence[
            ["subject", "model", "estimate", "ci_low", "ci_high", "p_value"]
        ],
    ],
    ignore_index=True,
)
display(attendance_comparison.style.format({"p_value": lambda p: "< .001" if p < .001 else f"{p:.3f}"}, precision=3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, constrained_layout=True)
for ax, (subject, rows) in zip(
    axes, attendance_comparison.groupby("subject", sort=False)
):
    ax.errorbar(
        rows.estimate,
        rows.model,
        xerr=[rows.estimate - rows.ci_low, rows.ci_high - rows.estimate],
        fmt="o",
        color="#227c77",
        capsize=4,
    )
    ax.axvline(0, color="gray", linestyle="--")
    ax.set(
        title=subject, xlabel="Spring difference per 5 percentage points more absence"
    )
fig.suptitle(
    "Raw and adjusted attendance associations: 95% student-clustered intervals"
)
attendance_association_figure = fig
plt.show()


#### Model 2 interpretation

Model 2 holds Fall score, school, grade and school year constant and adds absence between the two tests. The absence coefficient describes the Spring score difference associated with a 5 percentage-point higher absence rate:

| Subject | Adjusted Spring score difference | 95% interval with student clustering | p value |
|---|---:|---:|---:|
| Reading | -0.20 points | -0.36 to -0.04 | .016 |
| Mathematics | -0.32 points | -0.48 to -0.15 | < .001 |

The unadjusted differences were about -2.72 and -2.74 points. The associations become much smaller after Fall score, school, grade and year are considered together. The EDA also found a relationship between absence and Fall starting scores, but the reduction cannot be assigned entirely to Fall score or any other single adjustment.

Both adjusted intervals are below zero, so the associations are statistically significant under the stated model and cluster assumptions. Their magnitudes are small compared with residual standard deviations of about 5.67 and 5.76 points. The task supplies no threshold for an educationally meaningful score difference. Statistical significance alone therefore does not establish that this is a major influence on learning. R-squared changes from .93137 to .93149 in Reading and from .93631 to .93660 in Mathematics; these are fits to the observed data, not validated prediction performance.

School estimates change by at most about 0.04 points when absence is added, and their intervals still include zero. The review retained the primary formula and clustered uncertainty. In Section 4.8, the influence and background-label sensitivity fits preserve a small negative absence association and uncertain school comparisons. EL, economic-disadvantage and SPED labels are included in those sensitivity fits, not in the primary Model 2 above.

Observed absence ranges from 0% to 44.6% in Reading and from 0% to 45.4% in Mathematics. Most observations lie below the extremes. A 5 percentage-point comparison may be less well supported in some schools or grades; these coefficients should not be extrapolated beyond comparable observed students.

For the LD, the results support asking about students with overlapping learning and attendance needs and the barriers affecting them. They do not establish absence as the largest influence, or show how many score points an attendance intervention would produce. Shared classroom conditions, unmeasured needs and selection into paired testing remain limits.


### 4.6 Assumption checks and model diagnostics

Check both subjects. Review Models 1 and 2 where their specifications differ. The goal is to find problems that could change the interpretation or its uncertainty.

| Check | Priority | What I will inspect | Decision if there is a concern |
|---|---|---|---|
| Valid sample and predictor overlap | Essential | Missing fields, repeated keys, school-grade counts, comparable Fall and absence ranges | Correct sample errors or narrow the supported comparison |
| Repeated student records | Essential | Records per student and alignment of cluster IDs with the fitted rows | Use student-clustered uncertainty; document dependence that remains |
| Mean relationship and grade differences | Essential | Residuals versus fitted values and Fall score, separated by grade; residuals versus absence for Model 2 | Consider a Fall-by-grade interaction or a curved term if the plots justify it |
| Residual spread | Core visual review | Residual spread across fitted scores, grades, and schools | Retain clustered uncertainty; investigate severe patterns and model fit |
| Influential observations | Core review | Leverage and Cook's distance; examine flagged records | Verify records and check whether conclusions depend on them |
| Identifiable specification | Essential | Design-matrix rank, empty categories, redundant predictors | Remove redundancy or revise unsupported comparisons |
| Residual distribution | Supporting visual review | Q-Q plot and unusually heavy tails | Check extreme residuals and whether uncertainty or conclusions are sensitive |
| Formal normality or heteroscedasticity tests | Optional | Only if a test resolves a specific remaining concern | Interpret alongside the plots and repeated-record structure |
| Full VIF table or additional tests | Optional | Only if the specification suggests unstable or redundant predictors | Investigate the particular predictor problem |

Predictors do not need to be normally distributed. Residual normality is not a simple pass/fail gate for this large-sample analysis with clustered uncertainty. Robust standard errors also do not fix a wrong mean relationship, poor overlap, or omitted explanations.

A Durbin-Watson statistic on an arbitrarily sorted table does not establish independence for these repeated student records. Document the student structure directly.

Methods reference: [statsmodels regression diagnostics](https://www.statsmodels.org/stable/diagnostic.html).

#### 4.6A Connect fitted models and check the sample

`check_ols_model`, defined in **Custom functions**, reads an existing **statsmodels formula OLS result**, including a result fitted with student-clustered standard errors. It does not fit, modify, or select a model. GEE and WLS require separate diagnostics.

Fit with a unique DataFrame index and keep that index unchanged. Supply the same data used for fitting, with `student_id`, `school_year`, `school_id`, `grade_level`, `fall_score`, and `spring_score`. Keep `subject` and `between_test_absence_rate` for the subject and attendance checks.

The code aligns records to the fitted-row labels, rebuilds the formula's design matrix, and checks the observed outcome. It also verifies that cluster labels represent the same student groups in the correct order. A mismatch stops the check rather than attaching residuals to the wrong records.

Run **Custom functions** before this section. The next cell connects the fitted Models 1 and 2 to the diagnostic helper through `models_to_check`.


In [ ]:
models_to_check = globals().get("primary_models", {})
assumption_results = {}
if not models_to_check:
    print("No fitted models connected yet. Run Sections 4.4-4.5 when ready.")
else:
    for name, (result, model_data) in models_to_check.items():
        report = check_ols_model(result, model_data, label=name)
        assumption_results[name] = report
        print(f"\n{name}: sample and specification")
        display(report["summary"].to_frame("Value"))
        for note in report["notes"]:
            print(f"- {note}")
        print("Predictor ranges by grade, year, and school:")
        display(report["predictor_ranges"])
        print("Fall range overlap within grade and year:")
        display(report["fall_overlap"])


#### 4.6B Residual patterns and variance

The plots review the conditional mean, residual spread, Fall patterns within grades, and the absence relationship. An orange curve is a descriptive smoother. Look for sustained curvature, grade-specific patterns, or a widening residual band.

Model 1's absence plot can show what remains before attendance is included. For Model 2, use it to check whether a straight-line absence term describes the relationship adequately. The plots help choose a justified revision; they do not establish a causal relationship.

Record the patterns you see in the review log. Clustered standard errors address some uncertainty, but do not repair a wrong mean specification.


In [ ]:
if not assumption_results:
    print("Connect and check a fitted model in 4.6A first.")
else:
    for report in assumption_results.values():
        plot_residual_checks(report)
        plt.show()


#### 4.6C Influence and residual distribution

Use the Q-Q plot to inspect the residual tails. The Cook's-distance and leverage lines are rough screening rules. Inspect flagged rows and students before deciding whether a sensitivity refit is warranted. The code keeps every fitted record and never deletes observations.

The student table sums record-level Cook's distances to help locate students worth reviewing. It does not measure the effect of deleting a student's entire record set. Any such refit remains a separate step for me to perform.

These diagnostics use OLS influence quantities from the supplied fit. Cook's-distance p values, formal normality tests, and Breusch-Pagan p values are not used as automatic gates; ordinary reference tests do not account for the repeated-student structure.

Reference: [statsmodels OLS influence](https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.OLSInfluence.html).


In [ ]:
if not assumption_results:
    print("Connect and check a fitted model in 4.6A first.")
else:
    for name, report in assumption_results.items():
        plot_influence_checks(report)
        plt.show()
        rows = report["rows"]
        review_columns = [column for column in [
            "student_id", "school_year", "school_id", "grade_level", "subject",
            "fall_score", "spring_score", "between_test_absence_rate",
            "fitted", "residual", "leverage", "studentized_residual", "cooks_distance",
        ] if column in rows]
        print(f"{name}: flagged rows, largest Cook's distances first (up to 20)")
        display(rows.loc[rows.influence_flag, review_columns].sort_values("cooks_distance", ascending=False).head(20))
        print("Student-level review list (screening only; up to 10 students):")
        display(report["student_influence"].head(10))


#### How to read the output

| Output | Question to ask | Where to record the decision |
|---|---|---|
| Sample summary and notes | Are the fitted rows, student clusters, and specification appropriate? | Sample, repeated students, and rank rows below |
| Predictor ranges and Fall overlap | Are the school comparisons supported by comparable students? | Sample and overlap row |
| Residual curves by fitted score, Fall, and absence | Is there a systematic shape the model misses? | Mean relationship and grade patterns row |
| Residual spread and group boxes | Is variation much larger in some groups? | Residual spread row |
| Q-Q plot | Are a few extreme residuals driving the tail pattern? | Residual tails row |
| Flagged records and student review list | Do particular records warrant verification or a sensitivity refit? | Influence row |

These outputs do not produce an overall "assumptions passed" verdict. Describe what you see, whether it could change the substantive conclusion, and the action you take. Formal tests and a full VIF table remain optional if a specific concern justifies them.

If a check stops with a data or cluster mismatch, correct that issue before interpreting the diagnostic plots. A warning about covariance means the fitted model's uncertainty needs review; the diagnostic code does not change it for you.


#### Diagnostic review log

| Check | Reading | Mathematics | Decision |
|---|---|---|---|
| Sample and overlap | 3,653 rows; 1,780 students; no dropped rows | 3,606 rows; 1,779 students; no dropped rows | Retain common samples. All 18 grade-year groups include four schools with overlapping Fall ranges. The EDA describes score distributions; overlapping endpoints are only an initial screen and do not establish full multivariable common support. |
| Repeated students and covariance | 1,211 students contribute more than one year | 1,204 students contribute more than one year | Student groups match the fitted rows. Retain student-clustered intervals; cross-student dependence remains a limitation. |
| Mean relationship and grade patterns | Residual curves stay near zero; grade-specific Fall patterns are small. Added mean terms have joint p = .908 for M1 and .821 for M2 (4.7A). | No pronounced shape pattern. Added terms have joint p = .788 for M1 and .764 for M2. | Retain the linear Fall and absence terms. These targeted checks support the specification; they do not prove exact linearity. |
| Residual spread | Some narrowing at higher fitted scores, without a severe pattern | Some variation, without a severe pattern | Keep clustered uncertainty; do not switch to independent homoskedastic standard errors. |
| Influence | 157 M1 / 187 M2 records flagged by rough screening rules; maximum Cook's distance < .007 | 162 M1 / 193 M2 flagged; maximum Cook's distance < .006 | Flags do not establish errors. Keep primary rows and refit M2 without the five students highest on the review list as a sensitivity (4.8). |
| Rank | M1 rank 12/12; M2 13/13 | Same | No redundant design columns. No p-value variable selection. |
| Residual tails | Q-Q plot broadly follows the line, with modest upper-tail departures | Broadly follows the line | No severe tail problem requiring a revision. Check the influence sensitivity rather than delete valid extremes. |

The plots support retaining the initial mean specification. They cannot prove that the model is correct or resolve omitted variables, synthetic-data limits, selective test participation, or shared classroom/school conditions. Four schools are insufficient for routine school-clustered uncertainty. Conclusions stay specific to these historical paired samples; no causal or predictive-performance claims are made.


### 4.7 Model problems, actions and rechecks

A diagnostic review must end with a decision. If a problem could change an estimate, its interval, or the interpretation, correct it and repeat the affected checks before using the result. The action must match the problem: clustered uncertainty handles repeated student records and unequal residual variance for inference; it cannot repair a wrong mean relationship.

For a revised fit, keep the question and sample comparable where possible. Record the reason for the revision, check the revised fit, and compare the school contrasts or absence association with the original result. Choose the specification from the data structure and diagnostic evidence.

The record below distinguishes problems already handled by the fitting method, concerns examined through sensitivity refits, and limits the available data cannot resolve. Section 4.11 records the separate checks for the binary outcome model.


In [ ]:
diagnostic_summary = pd.concat(
    [report["summary"] for report in assumption_results.values()], axis=1
).T
display(diagnostic_summary)


#### Actions taken and current status

| Issue | Evidence | Action already taken | Recheck and decision |
|---|---|---|---|
| Repeated records from the same student | 1,211 Reading and 1,204 Mathematics students contribute more than one year | Fit both OLS models with student-clustered covariance, a finite-sample correction, and a t reference based on student clusters | Cluster labels match the fitted rows. Use the clustered intervals and tests. This allows dependence within students; it does not make the records independent. |
| Unequal residual spread | The spread plots show some variation across fitted scores and groups | Use clustered covariance, which also allows heteroscedasticity under the stated cluster assumptions | No severe remaining spread pattern is apparent in the plots. Retain this inference method; equal variance is not imposed for the reported intervals. |
| Possible curvature or different Fall slopes by grade | Overall residual curves stay near zero; some small grade-specific patterns remain, especially at sparse score extremes | Review residuals against fitted values, Fall score within grades, and absence for each subject and model | No clear sustained pattern requiring a curved term or interaction was identified. Retain the linear mean specification. This is a visual assessment, not proof of exact linearity. |
| Influential records and residual tails | Rough influence screens flag records; Q-Q plots show modest tail departures | Keep valid records in the primary fits. Refit M2 after omitting all records from the five students highest on the influence review list | Reading absence changes from -0.20 to -0.23 points per 5 percentage points; Mathematics from -0.32 to -0.37. School contrast intervals still include zero. These checks support retaining the primary fits (4.8). |
| Sample errors or redundant terms | No fitted-row or cluster mismatch; M1 rank 12/12 and M2 rank 13/13 | Check sample alignment, required fields and design rank | No correction is needed for these checks. |
| Shared conditions between different students; paired-test selection | Student clustering does not cover dependence between different students. Students without both tests are outside these models | Restrict the conclusions to the historical paired samples in these four schools and carry these limits into the briefing | These limits remain unresolved. Four schools do not support routine school-clustered inference. The results do not establish causal effects or apply automatically to all enrolled students. |

**Current decision:** the uncertainty method already addresses repeated student records and does not require equal residual variance. The saved plots and sensitivity refits do not identify a substantial remaining mean or influence problem requiring a new primary formula. The primary models retain Fall score, categorical school, grade and year; Model 2 also includes between-test absence.

This review does not give an overall “all assumptions passed” verdict. If a later diagnostic shows a material problem, the next step is an actual correction and a recheck:

- **Data, join or cluster mismatch:** correct the underlying records or fitting inputs, then rebuild and refit the affected models.
- **Sustained curvature or grade-specific slope differences:** fit an appropriate curved term or Fall-by-grade interaction on the comparable sample; repeat residual and influence checks and update the reported contrasts.
- **Verified data error or strong dependence on a few students:** correct errors at their source. For valid extreme records, use a justified alternative estimator or specification and compare it with the full-sample fit; do not remove records only because they change a p value.
- **An unsupported comparison or dependence that cannot be handled with these data:** narrow or withhold the inference and explain what additional data are needed.

The targeted mean-specification alternatives in 4.7A were fitted to examine the small within-grade patterns. Their checks did not identify a need to replace the primary formulas. The primary estimates and briefing conclusions remain unchanged. Methods reference: [statsmodels robust covariance options](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.get_robustcov_results.html).


#### 4.7A Targeted check of the mean specification

The residual plots showed small patterns within grades. I checked whether a more flexible relationship was warranted, using the same observations and student clustering:

- Allow the Fall-score slope to differ by grade.
- Add a quadratic Fall term, centered and expressed in tens of points for numerical scaling.
- For Model 2, also add a quadratic absence term.

For each subject and model, a clustered Wald F test assesses all added terms together. The full table also records the individual diagnostic blocks. These are specification checks, not a procedure for screening the substantive adjustment variables. They do not test every possible nonlinear relationship, and a large p value does not prove the original mean function correct.


In [ ]:
ols_mean_rows = []
ols_shape_fits = {}
for label, (primary, original_data) in primary_models.items():
    data = original_data.copy()
    data['fall_centered_10'] = (data.fall_score - data.fall_score.mean()) / 10
    formula = primary.model.formula + ' + fall_score:C(grade_level, Treatment(reference=0)) + I(fall_centered_10 ** 2)'
    if label.endswith('M2'):
        formula += ' + I(absence_5pp ** 2)'
    alternative = fit_primary_model(data, formula)
    ols_shape_fits[label] = (alternative, data)
    tests = alternative.wald_test_terms(skip_single=False, scalar=True).table
    terms = ['fall_score:C(grade_level, Treatment(reference=0))', 'I(fall_centered_10 ** 2)']
    if label.endswith('M2'):
        terms.append('I(absence_5pp ** 2)')
    # Patsy can place the interaction's factors in the opposite order.
    interaction = next(t for t in tests.index if 'fall_score' in t and ':' in t)
    terms[0] = interaction
    names = alternative.params.index
    columns = [i for i, name in enumerate(names) if (':' in name and 'fall_score' in name) or name.startswith('I(')]
    restriction = np.eye(len(names))[columns]
    joint = alternative.wald_test(restriction, use_f=True, scalar=True)
    ols_mean_rows.append({'model':label,'check':'All added mean terms','wald_F':float(joint.statistic),'p_value':float(joint.pvalue),'df_added':len(columns),'rows':int(alternative.nobs),'rank':np.linalg.matrix_rank(alternative.model.exog),'columns':alternative.model.exog.shape[1],'residual_sd':np.sqrt(alternative.mse_resid),'primary_residual_sd':np.sqrt(primary.mse_resid)})
    for term in terms:
        test = tests.loc[term]
        ols_mean_rows.append({'model':label,'check':term,'wald_F':float(test.statistic),'p_value':float(test.pvalue),'df_added':float(test.df_constraint),'rows':int(alternative.nobs),'rank':np.linalg.matrix_rank(alternative.model.exog),'columns':alternative.model.exog.shape[1],'residual_sd':np.sqrt(alternative.mse_resid),'primary_residual_sd':np.sqrt(primary.mse_resid)})
ols_mean_checks = pd.DataFrame(ols_mean_rows)
mean_check_results_dir = DATA_DIR.parent.parent / 'model_results'
mean_check_results_dir.mkdir(exist_ok=True)
ols_mean_checks.to_csv(mean_check_results_dir / 'mean_specification_checks.csv', index=False)
display(ols_mean_checks.loc[ols_mean_checks['check'].eq('All added mean terms')].style.format({'p_value': '{:.3f}', 'wald_F': '{:.3f}', 'residual_sd': '{:.3f}', 'primary_residual_sd': '{:.3f}'}))



#### Decision after the targeted mean checks

| Primary model | Joint p value for the added terms | Decision |
|---|---:|---|
| Reading M1 | .908 | No clear evidence requiring this added flexibility |
| Reading M2 | .821 | Same |
| Mathematics M1 | .788 | Same |
| Mathematics M2 | .764 | Same |

All four alternatives retain every primary row and have full-rank designs. Their residual standard deviations are essentially unchanged, about 5.67 points for Reading and 5.76 to 5.78 for Mathematics. The joint tests and earlier plots support retaining the original linear mean specification for the stated questions. They do not establish an overall assumption pass.

The primary school contrasts and absence estimates remain those in 4.4-4.5. The diagnostic alternatives are not replacement results. `model_results/mean_specification_checks.csv` records the checks.


### 4.8 Focused sensitivity checks

Two checks address specific concerns:

1. **Influence:** rough diagnostic flags identified records to review. For each subject's Model 2, temporarily omit all records from the five students with the highest sum of record-level Cook's distances. This is a stability check, not a data-cleaning rule or a formal measure of cluster influence. All valid records remain in the primary models.
2. **Recorded student needs:** Q1 showed differences in EL, economic-disadvantage, and SPED labels. Add all three labels to Models 1 and 2 on the same paired samples, using the same student clustering. This checks whether the main findings depend on representing that recorded mix. Fixed labels cannot reconstruct annual eligibility.

Section 4.7A fits diagnostic alternatives with grade-specific Fall slopes and quadratic terms. The joint tests and plots do not identify a need for that added flexibility, so the primary formulas remain unchanged. No GEE or latest-year-only model was added. Student clustering addresses the stated uncertainty within students; the participation and dependence-between-students limits remain.


In [ ]:
# Two checks motivated by the influence screens and recorded student needs.
background_terms = [
    "english_learner_flag",
    "economically_disadvantaged_flag",
    "special_education_flag",
]
sensitivity_models = {}
sensitivity_audit_rows = []
for subject, data in regression_samples.items():
    label = f"{subject} M2"
    review = assumption_results[label]["student_influence"]
    reviewed_students = review.head(5).index
    reduced = data.loc[~data.student_id.isin(reviewed_students)].copy()
    assert not reduced.student_id.isin(reviewed_students).any()
    name = f"{subject}: omit five reviewed students"
    sensitivity_models[name] = (
        fit_primary_model(reduced, model_2_formula),
        reduced,
        subject,
        "Influence M2",
    )
    sensitivity_audit_rows.append(
        {
            "subject": subject,
            "check": "Influence M2",
            "rows": len(reduced),
            "students": reduced.student_id.nunique(),
            "rows_removed": len(data) - len(reduced),
        }
    )
    enriched = data.copy()
    if enriched[background_terms].isna().any().any():
        raise ValueError(
            "Review missing background labels before fitting a sensitivity."
        )
    for term in background_terms:
        if not enriched[term].isin([0, 1, False, True]).all():
            raise ValueError(f"{term} must contain binary labels.")
        enriched[term] = enriched[term].astype(int)
    for model_name, formula in [("M1", model_1_formula), ("M2", model_2_formula)]:
        name = f"{subject}: background {model_name}"
        sensitivity_models[name] = (
            fit_primary_model(enriched, formula + " + " + " + ".join(background_terms)),
            enriched,
            subject,
            f"Background {model_name}",
        )
        sensitivity_audit_rows.append(
            {
                "subject": subject,
                "check": f"Background {model_name}",
                "rows": len(enriched),
                "students": enriched.student_id.nunique(),
                "rows_removed": 0,
            }
        )
sensitivity_audit = pd.DataFrame(sensitivity_audit_rows)
sensitivity_tables = []
sensitivity_diagnostics = {}
for name, (result, data, subject, check) in sensitivity_models.items():
    coefficients, _ = primary_result_tables(result, subject, check)
    sensitivity_tables.append(coefficients)
    sensitivity_diagnostics[name] = check_ols_model(result, data, name)
sensitivity_coefficients = pd.concat(sensitivity_tables, ignore_index=True)
sensitivity_key_results = sensitivity_coefficients.loc[
    sensitivity_coefficients.term.eq("absence_5pp")
    | sensitivity_coefficients.term.str.startswith("C(school_id")
].copy()
display(sensitivity_audit)
display(sensitivity_key_results.style.format({"p_value": lambda p: "< .001" if p < .001 else f"{p:.3f}"}, precision=3))
# These alternative fits do not replace the prespecified primary fits or delete source rows.


#### Sensitivity results

| Model 2 check | Reading absence estimate (95% CI) | Mathematics absence estimate (95% CI) |
|---|---:|---:|
| Primary | -0.20 (-0.36 to -0.04) | -0.32 (-0.48 to -0.15) |
| Omit five reviewed students | -0.23 (-0.39 to -0.07) | -0.37 (-0.53 to -0.21) |
| Add all three background labels | -0.25 (-0.42 to -0.09) | -0.36 (-0.53 to -0.19) |

All estimates are points per 5 percentage points more absence. The influence refits omit 13 Reading rows and 14 Mathematics rows, each from five students. The background fits keep every primary row. Their samples, cluster alignment, finite design values, and full rank were checked using the same diagnostic helper.

School contrasts remain small; every pointwise interval against Cedar Ridge includes zero in both sensitivity types. The absence estimates remain negative, small in score units, and have intervals below zero. These alternatives support the primary interpretation across the two concerns examined. They remain sensitivity fits; they do not replace the primary estimates or establish robustness to every specification or omitted variable.

Background-label coefficients are recorded in the full table for transparency. They are not causal estimates of program effects and are not used to rank groups or choose which variables to retain.


### 4.9 What the models add to the briefing

The visual review and targeted mean-specification checks retained the primary formulas and their student-clustered uncertainty. A fresh run after organizing the functions reproduced all 31 exported analytical tables. The estimates and intervals below therefore still match the checked results; there is no replacement model with a different numerical conclusion.

| Question | EDA finding | Checked primary result | Message for the LD | Limit / follow-up |
|---|---|---|---|---|
| Q2: school differences | Some Spring gaps already appear at Fall | M1 school contrasts against Cedar Ridge range from about -0.32 to +0.20 Reading points and -0.10 to +0.16 Mathematics points; all intervals include zero | Interpret raw school percentages alongside starting points. These fits do not establish school equivalence or an effectiveness ranking. | Historical paired students pooled across years and grades; verify current rosters and supports. |
| Q3: attendance and learning | Absence relates to score levels more than raw change | Per 5 percentage points more between-test absence: Reading -0.20 points (95% CI -0.36 to -0.04); Mathematics -0.32 (-0.48 to -0.15) | The adjusted association is statistically significant but small. Review overlapping needs and attendance barriers without promising a score gain. | The primary fit adjusts for Fall, school, grade and year. Background labels enter sensitivity fits. Shared conditions and unmeasured needs remain. |

The influence and background-label sensitivity fits retain these conclusions. Their estimates are reported in 4.8 and should be labelled as alternatives when shown.

Suggested briefing wording:

"School populations and observed outcomes differ. In students with both tests, the adjusted school comparisons are small and uncertain. Higher absence between tests is associated with slightly lower Spring scores after accounting for Fall score, school, grade and year. The association is statistically significant, but much smaller than the raw comparison. I would use the findings to review current learning needs and attendance barriers with principals. They do not show the impact of a school or an attendance intervention."

Keep these score estimates separate from the percentages meeting Benchmark among all Spring-tested students. The exploratory starting-group model in 4.11 answers a further question about continuing below-Benchmark status; it does not replace these OLS results. The two figures below support the completed 10-slide briefing.


In [ ]:
# Export aggregate results and two figures for the later PPT.
model_results_dir = DATA_DIR.parent.parent / "model_results"
model_results_dir.mkdir(exist_ok=True)
for filename, table in {
    "primary_coefficients.csv": primary_coefficients,
    "primary_term_tests.csv": primary_tests,
    "primary_model_summary.csv": primary_model_summary,
    "model1_school_contrasts.csv": model_1_school_contrasts,
    "attendance_comparison.csv": attendance_comparison,
    "sensitivity_audit.csv": sensitivity_audit,
    "sensitivity_coefficients.csv": sensitivity_coefficients,
}.items():
    table.to_csv(model_results_dir / filename, index=False)
diagnostic_summary.to_csv(model_results_dir / "diagnostic_summary.csv")
for filename, figure in {
    "model1_school_contrasts": school_contrast_figure,
    "attendance_raw_vs_adjusted": attendance_association_figure,
}.items():
    figure.savefig(model_results_dir / f"{filename}.png", dpi=220, bbox_inches="tight")
    figure.savefig(model_results_dir / f"{filename}.svg", bbox_inches="tight")
print(
    f"Aggregate tables and two briefing figures saved to {model_results_dir.resolve()}"
)


### 4.10 Model completion checklist

- [x] Reading and Mathematics samples and exclusions are documented.
- [x] Models 1 and 2 use the same rows within each subject.
- [x] School, grade, and year are treated as categories; the school reference is recorded.
- [x] Student-clustered intervals use the correct fitted rows.
- [x] Absence units are clear and comparisons stay within supported ranges.
- [x] Diagnostics are reviewed, with any revisions and remaining concerns recorded.
- [x] Final school contrasts and absence results include 95% intervals.
- [x] Interpretation distinguishes adjusted association from causal effect.
- [x] A short conclusion and LD follow-up are ready for Sections 5-6.

The model work, interpretation checks, briefing, and dashboard are complete and have been reviewed for submission.


### 4.11 Focused follow-up: starting severity and Spring status

Section 3.10 found a large descriptive difference between the two Fall-below groups. I now test one question per subject: among students below in Fall, is a Well Below starting level associated with a higher probability of remaining below in Spring after allowing for school, grade, and year?

This is an exploratory question identified using the same data, not an independent confirmation or a question prespecified before the EDA. The original OLS models stay unchanged. No variables are selected by p value and no school-performance ranking is added.

**Plan recorded before these fits:**

- Use all three years of the checked paired Reading and Mathematics samples. Restrict to Fall Below or Well Below. Missing results are not coded as meeting Benchmark.
- Outcome: `spring_below = 1` for Spring Below or Well Below; `0` for At or Above.
- Focal predictor: `fall_well_below = 1` versus Fall Below as reference. Adjust for categorical school, grade, and year. Numeric Fall score is not added: this question compares recorded starting-level groups rather than groups at an identical numeric starting score.
- Fit binary logistic regression with student-clustered covariance, finite-sample correction, and t-based inference using student clusters. The predictor of interest has one coefficient test per subject. Apply Holm correction across these two tests.
- Present probabilities standardized over the same observed school-grade-year mix within each subject. Restrict that averaging target to strata with both starting groups. Each student-year gets equal weight. Confidence intervals use the delta method and clustered covariance, conditional on this observed target mix.
- Check sample/cluster alignment, both groups' coverage, convergence, separation warnings, weighted rank, finite estimates, probability calibration by bins, and influential observations. Binary models do not require normally distributed residuals.
- Two focused stability checks: add all three fixed background labels; temporarily omit all records of the five students highest on the model's Cook-distance review list. Keep the primary model and averaging target fixed for comparison.

Starting level is a measured condition, not an intervention. These models do not estimate the causal effect of being below, a service benefit, individual future risk, or model performance on new students. Student clustering still leaves shared conditions across different students unresolved.

Method references: [statsmodels GLM fitting](https://www.statsmodels.org/stable/generated/statsmodels.genmod.generalized_linear_model.GLM.fit.html) and [robust covariance](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.get_robustcov_results.html).


In [ ]:
persistence_samples, persistence_sample_audit = prepare_persistence_samples(
    regression_samples
)
persistence_formula = (
    "spring_below ~ fall_well_below"
    " + C(school_id, Treatment(reference=101))"
    " + C(grade_level, Treatment(reference=0))"
    " + C(school_year, Treatment(reference=2024))"
)
display(persistence_sample_audit)


In [ ]:
persistence_models = {}
persistence_targets = {}
persistence_support = {}
persistence_risk_tables = []
persistence_contrast_rows = []
persistence_coefficient_tables = []
persistence_diagnostic_rows = []
persistence_review_students = {}
persistence_calibration_tables = []
for subject, data in persistence_samples.items():
    result = fit_persistence_model(data, persistence_formula)
    persistence_models[subject] = result
    # Compare supported school-grade-year strata, not each group's own mix.
    strata = ["school_id", "grade_level", "school_year"]
    support = pd.crosstab([data[c] for c in strata], data.fall_well_below).reindex(
        columns=[0, 1], fill_value=0
    )
    support.columns = ["fall_below", "fall_well_below"]
    support["both_groups"] = (
        support[["fall_below", "fall_well_below"]].gt(0).all(axis=1)
    )
    support = support.reset_index()
    persistence_support[subject] = support
    target = data.merge(
        support.loc[support.both_groups, strata],
        on=strata,
        how="inner",
        validate="many_to_one",
    )
    persistence_targets[subject] = target
    risks, contrast = standardized_persistence(result, target)
    risks.insert(0, "subject", subject)
    persistence_risk_tables.append(risks)
    contrast = contrast.to_dict()
    contrast["subject"] = subject
    interval = result.conf_int().loc["fall_well_below"]
    contrast.update(
        {
            "log_odds_coefficient": result.params["fall_well_below"],
            "odds_ratio": np.exp(result.params["fall_well_below"]),
            "odds_ratio_ci_low": np.exp(interval.iloc[0]),
            "odds_ratio_ci_high": np.exp(interval.iloc[1]),
            "p_value": result.pvalues["fall_well_below"],
        }
    )
    persistence_contrast_rows.append(contrast)
    coefficients, _ = primary_result_tables(result, subject, "Exploratory logistic")
    persistence_coefficient_tables.append(coefficients)
    influence = result.get_influence()
    review = data[
        ["student_id", "school_year", "fall_well_below", "spring_below"]
    ].copy()
    review["cooks_distance"] = influence.cooks_distance[0]
    review["fitted_probability"] = result.fittedvalues.to_numpy()
    student_review = (
        review.groupby("student_id").cooks_distance.sum().sort_values(ascending=False)
    )
    persistence_review_students[subject] = student_review
    weighted_design = (
        result.model.exog
        * np.sqrt(
            result.fittedvalues.to_numpy() * (1 - result.fittedvalues.to_numpy())
        )[:, None]
    )
    persistence_diagnostic_rows.append(
        {
            "subject": subject,
            "rows": int(result.nobs),
            "students": data.student_id.nunique(),
            "parameters": len(result.params),
            "converged": result.converged,
            "design_rank": np.linalg.matrix_rank(result.model.exog),
            "weighted_rank": np.linalg.matrix_rank(weighted_design),
            "max_absolute_score": np.abs(result.model.score(result.params)).max(),
            "minimum_fitted_probability": result.fittedvalues.min(),
            "maximum_fitted_probability": result.fittedvalues.max(),
            "maximum_cooks_distance": review.cooks_distance.max(),
            "supported_strata": int(support.both_groups.sum()),
            "observed_strata": len(support),
            "standardization_rows": len(target),
            "well_below_reached_at_above": int(
                ((data.fall_well_below == 1) & (data.spring_below == 0)).sum()
            ),
        }
    )
    review["probability_bin"] = pd.qcut(
        review.fitted_probability, q=6, duplicates="drop"
    )
    calibration = (
        review.groupby("probability_bin", observed=True)
        .agg(
            rows=("student_id", "size"),
            observed=("spring_below", "mean"),
            fitted=("fitted_probability", "mean"),
        )
        .reset_index()
    )
    calibration["probability_bin"] = calibration.probability_bin.astype(str)
    calibration.insert(0, "subject", subject)
    persistence_calibration_tables.append(calibration)
persistence_probabilities = pd.concat(persistence_risk_tables, ignore_index=True)
persistence_contrasts = pd.DataFrame(persistence_contrast_rows)
persistence_contrasts["p_holm_two_subjects"] = multipletests(
    persistence_contrasts.p_value, method="holm"
)[1]
persistence_coefficients = pd.concat(persistence_coefficient_tables, ignore_index=True)
persistence_diagnostics = pd.DataFrame(persistence_diagnostic_rows)
persistence_calibration = pd.concat(persistence_calibration_tables, ignore_index=True)
display(persistence_probabilities.style.format(precision=3))
display(
    persistence_contrasts.style.format(
        {
            "p_value": lambda p: "< .001" if p < 0.001 else f"{p:.3f}",
            "p_holm_two_subjects": lambda p: "< .001" if p < 0.001 else f"{p:.3f}",
        },
        precision=3,
    )
)
display(persistence_diagnostics)


In [ ]:
persistence_sensitivity_rows = []
for subject, data in persistence_samples.items():
    # A few Well Below students reach At/Above, so review their influence on the contrast.
    reviewed_ids = persistence_review_students[subject].head(5).index
    reduced = data.loc[~data.student_id.isin(reviewed_ids)].copy()
    enriched = data.copy()
    labels = [
        "english_learner_flag",
        "economically_disadvantaged_flag",
        "special_education_flag",
    ]
    for label in labels:
        if (
            enriched[label].isna().any()
            or not enriched[label].isin([0, 1, False, True]).all()
        ):
            raise ValueError(
                "Background labels must be complete and binary for the sensitivity."
            )
        enriched[label] = enriched[label].astype(int)
    for check, check_data, formula in [
        ("Omit five reviewed students", reduced, persistence_formula),
        (
            "Add fixed background labels",
            enriched,
            persistence_formula + " + " + " + ".join(labels),
        ),
    ]:
        alternative = fit_persistence_model(check_data, formula)
        risks, contrast = standardized_persistence(
            alternative, persistence_targets[subject]
        )
        persistence_sensitivity_rows.append(
            {
                "subject": subject,
                "check": check,
                "rows": len(check_data),
                "students": check_data.student_id.nunique(),
                "rows_removed": len(data) - len(check_data),
                "below_probability": risks.iloc[0].probability,
                "well_below_probability": risks.iloc[1].probability,
                "difference_pp": contrast.difference_pp,
                "ci_low_pp": contrast.ci_low_pp,
                "ci_high_pp": contrast.ci_high_pp,
                "p_value": alternative.pvalues["fall_well_below"],
                "converged": alternative.converged,
            }
        )
persistence_sensitivity = pd.DataFrame(persistence_sensitivity_rows)
display(
    persistence_sensitivity.style.format(
        {"p_value": lambda p: "< .001" if p < 0.001 else f"{p:.3f}"}, precision=3
    )
)


#### Check the common starting-group association

The primary logistic model assumes the Fall starting-group association is common on the log-odds scale across grades and years. I checked that concern by adding starting-group-by-grade and starting-group-by-year interactions together, keeping the same samples, student clustering and averaging targets.

A joint clustered Wald F test evaluates the seven added interaction coefficients. I also recompute the average probability gap over the original supported mix. This checks one specific extension of the mean structure; it is not external prediction validation or proof that every logistic assumption holds.


In [ ]:
logit_mean_rows=[]
logit_mean_fits={}
for subject, primary in persistence_models.items():
    data=persistence_samples[subject]
    formula=persistence_formula + ' + fall_well_below:C(grade_level, Treatment(reference=0)) + fall_well_below:C(school_year, Treatment(reference=2024))'
    alternative=fit_persistence_model(data,formula)
    logit_mean_fits[subject]=(alternative,data)
    names=alternative.params.index
    columns=[i for i,name in enumerate(names) if ':' in name and 'fall_well_below' in name]
    joint=alternative.wald_test(np.eye(len(names))[columns],use_f=True,scalar=True)
    risks,contrast=standardized_persistence(alternative,persistence_targets[subject])
    logit_mean_rows.append({'subject':subject,'check':'Starting group by grade and year','wald_F':float(joint.statistic),'p_value':float(joint.pvalue),'df_added':len(columns),'rows':int(alternative.nobs),'rank':np.linalg.matrix_rank(alternative.model.exog),'columns':alternative.model.exog.shape[1],'converged':alternative.converged,'difference_pp':contrast.difference_pp,'ci_low_pp':contrast.ci_low_pp,'ci_high_pp':contrast.ci_high_pp})
logit_mean_checks = pd.DataFrame(logit_mean_rows)
logit_mean_checks.to_csv(learning_results_dir / 'persistence_mean_specification_checks.csv', index=False)
display(logit_mean_checks.style.format({'p_value': '{:.3f}', 'wald_F': '{:.3f}', 'difference_pp': '{:.1f}', 'ci_low_pp': '{:.1f}', 'ci_high_pp': '{:.1f}'}))


#### Decision after the logistic mean check

The added interactions have joint p values of .845 for Reading and .910 for Mathematics. Both alternative fits converge and have full rank. Their average probability differences remain about +26.2 percentage points, with 95% intervals of 22.9 to 29.5 in Reading and 22.8 to 29.6 in Mathematics.

Together with the existing support, calibration and influence checks, these results support retaining the common starting-group model. The primary estimates and their Holm-adjusted focal tests remain unchanged. The interaction-model tests are diagnostic checks; they are not additional evidence to present as independent confirmation of the original finding. `learning_results/persistence_mean_specification_checks.csv` records this comparison.


#### Adjusted results and diagnostic review

The fits use 1,690 Reading student-year records from 967 students and 1,637 Mathematics records from 938 students. All records are Fall Below or Well Below with both tests. There are no additional missing-field exclusions. Each subject has both starting groups in all 72 school-grade-year strata, so every model row contributes to the shared standardization target.

| Subject | Fall Below: adjusted probability of Spring below | Fall Well Below: adjusted probability | Difference, percentage points (95% CI) | Holm-adjusted focal p value |
|---|---:|---:|---:|---:|
| Reading | 70.8% (67.6%-73.8%) | 97.0% (95.5%-98.0%) | +26.2 (22.9 to 29.5) | < .001 |
| Mathematics | 70.5% (67.3%-73.4%) | 96.7% (95.1%-97.8%) | +26.2 (22.8 to 29.6) | < .001 |

These are model-based probabilities averaged over the same historical school-grade-year mix within each subject. They are not the latest-year raw rates or predictions for the coming fall. Intervals use student-clustered covariance. The p values test the focal starting-level coefficient, with Holm correction across Reading and Mathematics; probability differences have separate delta-method intervals.

The adjusted odds ratios are 13.26 for Reading (95% CI 8.58-20.48) and 12.50 for Mathematics (7.94-19.67). Since the outcome is common, an odds ratio is much larger than the probability ratio. Use the probabilities and percentage-point differences for the LD, rather than saying the probability is thirteen times higher.

The estimated group difference is about 26 percentage points in each subject, with intervals above zero and Holm-adjusted focal p values below .001. This is a substantial historical difference under the model and its cluster assumptions. The primary logistic fits adjust for school, grade and year; the student clustering concerns uncertainty, and the three background labels enter a separate sensitivity fit. The results support reviewing progress and learning support by starting group. They do not explain why students started low or establish which service would help.

| Diagnostic | Finding | Decision |
|---|---|---|
| Sample and groups | Unique student-year keys; all rows retained; student-cluster labels match fit order | Keep the stated samples. Four-school cross-student dependence remains a limitation. |
| Starting-group support | Both groups in all 72 school-grade-year strata in each subject | Use the full observed mix for standardization. |
| Convergence and separation | Both fits converge without separation warnings; finite coefficients and covariance; largest score-equation remainder < 2e-10 | No fitting revision needed. |
| Rank | Ordinary and probability-weighted designs both rank 12/12 | No redundant model terms. |
| Near-ceiling group | Only 24 Reading and 23 Mathematics Well Below records reach At/Above across all three years | Keep the outcome and inspect stability; do not treat the estimated probability as certainty. |
| Calibration within fitted data | Six probability bins have observed-versus-fitted differences up to 4.3 percentage points in Reading and 1.5 in Mathematics | No gross calibration problem seen. This is a fitting diagnostic, not validation on new students. |
| Influence | Maximum record Cook's distance about .0078 and .0080; five student record sets reviewed by sensitivity refit | Keep all primary observations; inspect the separate refits below. |

**Action after review:** student-clustered covariance already handles repeated records for the reported logistic-model uncertainty under the stated cluster assumptions. Both primary fits and the influence/background sensitivity fits converge. The current convergence, rank, support and calibration checks do not identify a fitting problem requiring replacement of the logistic model. If a refit develops separation, nonconvergence or a clear mean-pattern problem, its results must be withheld until the specification or fitting method is corrected and checked again. Shared conditions between different students remain an unresolved limit, as in the OLS models.

The targeted interaction check above found no clear evidence requiring the starting-group association to differ by grade or year; the average probability gap remains about 26.2 percentage points. The OLS residual-normality checklist is not applied to these binary outcomes. The model retains a common starting-group association on the log-odds scale and additive category adjustments. These checks cannot prove the mean specification correct or remove omitted explanations, selection into paired testing, fixed-label limitations, or shared classroom conditions.

**Stability:** omitting the five reviewed students removes 14 Reading rows and 11 Mathematics rows. The probability gaps are then +26.7 (95% CI 23.4-30.0) and +26.5 (23.2-29.8) points. Adding all three fixed background labels on unchanged samples gives +23.9 (20.6-27.3) and +25.2 (21.8-28.7) points. All alternative fits converge and retain a positive, substantial gap. They support stability across these two checks, not every possible specification. They do not replace the primary fit because of their p values.

**Briefing wording:**

"Students often gained score points and still ended below Benchmark. Among students who started below, those recorded as Well Below had a much higher fitted probability of remaining below in Spring: about 97%, compared with about 71% for the Below group, after school, grade, and year adjustment. The difference was about 26 percentage points in both subjects. I would ask how schools monitor progress and review support for different starting groups, then verify which historical needs remain in the new roster."

This question followed the EDA on the same synthetic data. The diagnostic and sensitivity review supports retaining the reported historical association. The two-test Holm correction does not account for choosing this question after seeing the EDA. Treat the finding as exploratory; it needs confirmation with another cohort before use as a prediction or a rule for assigning services.


In [ ]:
persistence_probability_figure, axes = plt.subplots(
    1, 2, figsize=(12.5, 5.4), sharey=True
)
for ax, subject in zip(axes, ["Reading", "Mathematics"]):
    rows = persistence_probabilities.loc[persistence_probabilities.subject.eq(subject)]
    difference = persistence_contrasts.loc[
        persistence_contrasts.subject.eq(subject)
    ].iloc[0]
    ax.errorbar(
        [0, 1],
        rows.probability,
        yerr=[rows.probability - rows.ci_low, rows.ci_high - rows.probability],
        fmt="o",
        color="#237f78",
        capsize=5,
        markersize=8,
    )
    for position, row in enumerate(rows.itertuples()):
        ax.text(
            position,
            row.ci_high + 0.025,
            f"{row.probability:.1%}",
            ha="center",
            fontsize=11,
        )
    ax.set(
        xticks=[0, 1],
        xticklabels=["Fall Below", "Fall Well Below"],
        xlim=(-0.45, 1.45),
        ylim=(0, 1.05),
        yticks=np.arange(0, 1.01, 0.2),
        title=subject,
    )
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis="y", alpha=0.18)
    ax.text(
        0.5,
        -0.21,
        f"Difference: +{difference.difference_pp:.1f} percentage points\n"
        f"95% CI {difference.ci_low_pp:.1f} to {difference.ci_high_pp:.1f}; Holm p < .001",
        transform=ax.transAxes,
        ha="center",
        fontsize=10,
    )
axes[0].set_ylabel("Fitted probability of Spring below Benchmark")
persistence_probability_figure.suptitle(
    "Lowest starting group has greater continuing needs, 2023-24 to 2025-26",
    fontsize=14,
)
persistence_probability_figure.text(
    0.08,
    0.025,
    "Common school-grade-year mix within each subject; student-clustered 95% intervals. Historical exploratory association.",
    fontsize=9,
)
persistence_probability_figure.subplots_adjust(
    left=0.08, right=0.97, top=0.81, bottom=0.29, wspace=0.22
)
plt.show()


In [ ]:
# Aggregate exploratory model outputs, with no student identifiers.
for filename, table in {
    "persistence_model_sample.csv": persistence_sample_audit,
    "persistence_adjusted_probabilities.csv": persistence_probabilities,
    "persistence_adjusted_contrasts.csv": persistence_contrasts,
    "persistence_model_coefficients.csv": persistence_coefficients,
    "persistence_model_diagnostics.csv": persistence_diagnostics,
    "persistence_model_calibration.csv": persistence_calibration,
    "persistence_model_sensitivity.csv": persistence_sensitivity,
}.items():
    table.to_csv(learning_results_dir / filename, index=False)
for subject, support in persistence_support.items():
    support.to_csv(
        learning_results_dir / f"{subject.lower()}_starting_group_support.csv",
        index=False,
    )
persistence_probability_figure.savefig(
    learning_results_dir / "adjusted_starting_group_probabilities.png",
    dpi=220,
    bbox_inches="tight",
)
persistence_probability_figure.savefig(
    learning_results_dir / "adjusted_starting_group_probabilities.svg",
    bbox_inches="tight",
)
print(
    "Exploratory logistic summaries and the probability figure saved to learning_results."
)


## 5. Candidate briefing visuals

The 10-slide briefing in [PowerPoint briefing](../briefing/KaWaiSit_Level_Director_Briefing.pptx) uses the school profile, progress and continuing-status comparison, adjusted starting-group probabilities, subject overlap, attendance findings and proposed principal conversations. The other candidates below remain available for discussion. The deck includes editable charts and tables and detailed speaker notes.

| Visual | Source | Message | Population and limits |
|---|---|---|---|
| School profile and enrollment comparison | EDA 1: `latest_school_profile` and enrollment tables | School populations and needs differ | Latest annual enrollment profile; three-year comparison; fixed student labels |
| Absence over time, with a school-grade view | EDA 2 and 5 | School rates improved, but some grade groups still have higher absence | Enrolled instructional days; historical grades |
| Fall and Spring learning by subject and grade | EDA 3-4 | Interpret Spring results alongside starting points and change | Separate subjects; paired counts; no expected-growth target |
| Score progress and continuing status | 3.10A: `learning_progress_and_status.png` | About 91%-93% of those below in both windows gained points; continuing needs remain visible | Paired students; separate denominators; no expected-growth target |
| Adjusted starting-group probabilities | Section 4.11: `adjusted_starting_group_probabilities.png` | About 97% versus 71% probability of remaining below, adjusted gap about 26 percentage points | Paired Fall-below students across three years; common mix; exploratory association |
| Subject and attendance overlap | 3.10B: overlap charts | At least 618 students have an observed pattern; most are below the 10% absence cutoff | Unique historical students; incomplete paths retained; verify current enrollment |
| Attendance and learning summary | EDA 6 | Fall differences already exist, so Spring comparisons need adjustment | Three years pooled; student-year records; descriptive relationships |
| Adjusted school contrasts | Section 4.4 and `model_results/model1_school_contrasts.png` | Small, uncertain school differences after allowing for starting points | Historical paired students; pointwise intervals against Cedar Ridge; no effectiveness ranking |
| Raw versus adjusted absence association | Section 4.5 and `model_results/attendance_raw_vs_adjusted.png` | Raw association becomes much smaller after adjustment | Per 5 percentage points of between-test absence; not an intervention effect |
| Optional grade 5 literacy table | State-test review | Lincoln Park and Meadowbrook have recurring results worth discussing | Tested proficiency; estimated opportunity; earlier cohorts have moved on |

The selected briefing visuals state years, counts and relevant denominators, and distinguish descriptive results from adjusted probabilities. The Fall Reading boxplot and school-contrast figure remain outside the main slides. The briefing retains the stated populations, messages, and limitations.


In [ ]:
# Final briefing visuals will follow the model review.


## 6. Briefing conclusions and process notes

### 6.1 What the new LD should know

#### Scores often rose while learning needs continued

In 2025-26, 443/539 Fall-below Reading students (82.2%) and 429/516 Fall-below Mathematics students (83.1%) were still below in Spring. Among those below in both windows, 90.5% in Reading and 93.0% in Mathematics gained score points, with median changes of +10 and +11. The data show improvement alongside continuing below-Benchmark status. They do not supply a target for adequate growth.

Students with the lowest starting levels deserve a specific monitoring question: 97.0% of Fall Well Below Reading students and 98.2% of the Mathematics group remained below in Spring, compared with about 71%-72% of Fall Below students. Some moved from Well Below to Below, so below-in-both does not mean no improvement. Ask how principals review needs and progress across these starting groups. The exploratory logistic follow-up across all three years finds adjusted probabilities of 97.0% versus 70.8% in Reading and 96.7% versus 70.5% in Mathematics. The differences are +26.2 percentage points (95% CI 22.9-29.5 and 22.8-29.6), with both focal tests below .001 after Holm correction. These probabilities use a common school, grade and year mix within each subject; their intervals allow repeated records within students. The primary fits do not include the background labels, which enter separate sensitivity fits. The diagnostic review retained these results, but the question arose from the same data and remains exploratory. It does not establish a cause or service effect.

Combining subjects identifies 618 unique students with a known below-in-both pattern in at least one subject, including 254 in both subjects. Of the 618, 475 have annual absence below 10% and 143 have absence at least 10%. Academic follow-up should therefore begin with learning needs, while coordinated attendance support is a question for the overlapping group. These are historical descriptive counts and do not estimate an intervention effect.

The latest historical K-4 group contains 513 students with an identified pattern. Use the school-grade table to verify incoming enrollment and support arrangements; do not turn these counts into a forecast. Grade 5 transition needs are listed separately. There are 281 students across K-5 with incomplete two-subject paths, which limits a complete picture and warrants record review alongside estimated testing opportunity.

#### School populations are changing in different directions

Cedar Ridge and Lincoln Park grew by about 9% to 10% over three years; Harper Creek and Meadowbrook declined. Lincoln Park has the largest latest enrollment (376) and recorded EL share (15.7%). Meadowbrook and Lincoln Park both have SPED shares around 15% to 16%. The LD should check incoming enrollment and support arrangements. These historical figures alone do not establish next-fall demand or staff shortages.

#### Attendance improved, but school averages hide grade differences

Latest school absence rates are close, at 5.20% to 5.73%. Each school's highest grade-level rate occurs in Cedar Ridge grade 1 (7.08%, 56 students), Harper Creek grade 1 (6.49%, 53), Lincoln Park Kindergarten (7.26%, 59), and Meadowbrook grade 5 (6.37%, 59). Between 15.7% and 18.1% of students miss at least 10% of enrolled days, using an exploratory cutoff. These patterns can guide principal conversations, with current students and barriers checked before action.

#### Spring differences need to be understood alongside Fall starting points

Spring At/Above Benchmark is 53.6% to 61.3% in Reading and 52.8% to 62.2% in Mathematics among tested students. Median paired change is positive in every latest-year school-grade-subject group, but no expected-growth target is supplied. Grade 5 state ELA also suggests literacy follow-up at Lincoln Park and Meadowbrook. Model 1 finds small, uncertain school contrasts after allowing for Fall score, grade, and year. These pooled contrasts do not explain every grade-specific pattern or establish school equivalence.

#### Attendance and academic support may need to be discussed together

Within grades, higher absence tends to accompany lower Fall and Spring scores across the three years. Its raw relationship with score change is weak. After accounting for Fall score, school, grade and year, 5 percentage points more between-test absence is associated with about 0.20 points lower Reading and 0.32 points lower Mathematics Spring score. The 95% intervals are -0.36 to -0.04 and -0.48 to -0.15. Both associations are statistically significant under the stated model and cluster assumptions, but small in score units and much weaker than the raw differences of about 2.7 points. The influence and background-label sensitivity fits preserve this interpretation. The LD can ask how principals identify overlapping needs and understand attendance barriers. These results do not establish absence as the largest influence or estimate the benefit of an intervention.

### 6.2 Questions to take to principals this fall

| Follow-up | Evidence | What to check |
|---|---|---|
| Continuing academic needs | 82.2%-83.1% of paired Fall-below students remain below; most of those students gain score points | Current roster, starting severity, progress monitoring, and available support. Positive change and reaching Benchmark answer different questions. |
| Overlapping subject needs | 254 historical students have the pattern in both subjects; 513 K-4 students have an identified pattern in at least one | Verify which students return and how Reading and Math support are coordinated. Missing subject information stays unknown. |
| Enrollment and support at growing schools | Cedar Ridge +9.0%; Lincoln Park +9.9% over three years | Incoming roster, grade demand, newcomer support, and current services |
| Attendance within schools | Higher-absence grades differ; a group with high absence remains | Current students affected, time enrolled, barriers, and existing follow-up. Former grade 5 students may now be in middle school. |
| Upper-elementary literacy | Grade 5 ELA proficiency among tested students is 41.0% to 53.2% at Lincoln Park and 49.1% to 50.0% at Meadowbrook over three years | Literacy support and preparation between grades. These results describe earlier cohorts, not incoming grade 5 students. |
| Coordination of attendance and academic support | 143 of 618 students with an identified learning pattern also have annual absence at least 10%; 475 are below that cutoff | Coordinate follow-up for overlapping needs, while keeping academic support available beyond the high-absence band |
| Staffing and service records | Recorded FTE and role labels differ; assignment coverage has an unresolved gap | Shared services, role coding, year definitions, and missing assignments before drawing resource conclusions |

### 6.3 How I approached the task

- QC identified partial-year enrollment, several testing windows, different calendar lengths, fixed student labels, and staffing gaps. These shaped the sequence: school context, learning, then attendance.
- I retained all enrolled elementary students for school profiles, summarized attendance before joining, and paired tests by student, year, and subject. Between-test attendance uses actual test dates. Join checks protect the intended row structure.
- Official test windows were unavailable. I estimated opportunity from recorded testing dates and disclosed that choice. Paired records have longer average enrollment than excluded records, which limits how widely their findings apply.
- Grade and Fall achievement affect learning comparisons. Schoolwide rates and raw correlations identify follow-up questions; Models 1 and 2 examine adjusted differences and associations, using student-clustered intervals because students recur across years.
- After the primary models showed small, uncertain school contrasts and a small absence association, I deepened the learning question by following the same students across reported levels and counting overlaps once per student. This focuses the briefing on identified needs without changing the original models to search for stronger results.
- Context, benchmarks, and attendance cover all elementary grades. Recurring grade 5 literacy provides a supporting point. Discipline and staffing stay as context or verification questions because their counts and coverage limit stronger conclusions.
- With more time, I would obtain official test eligibility and windows, the coming fall roster, historical program labels, complete staffing and service records, and information on attendance barriers and existing support. Those would improve both the comparisons and their usefulness for decisions.

### 6.4 AI use and analytical responsibility

I set the project direction, refined the research questions and analytical choices, and decided which findings and proposed follow-up actions belonged in the briefing. I take responsibility for reviewing the code, calculations, and final interpretation.

I used OpenAI Codex mainly to help write and troubleshoot Python code for preparation, quality checks, models, diagnostics and charts, and to build the HTML dashboard. It also helped organize questions, edit explanations and prepare the presentation. The humanizer skill supported clearer wording, and the presentations and frontend-design skills supported slide layout.

QC, EDA, the four primary OLS models, the exploratory logistic follow-up, diagnostic review, sensitivity checks, and the 10-slide briefing are complete and have been reviewed for submission. The current deck is [PowerPoint briefing](../briefing/KaWaiSit_Level_Director_Briefing.pptx); the dashboard setup is documented [here](../dashboard/README.md).


## 7. Interactive briefing dashboard

The five views follow the briefing: school context, score progress and continuing status, overlapping learning needs, attendance alongside learning, and completed adjusted models. The dashboard uses the checked tables above and does not fit models. The dashboard runs locally with the supplied case-study data.

Run the next cell to refresh `Fairmont_LD_Dashboard.html`. It contains the data and chart code, so it can open offline. The year, school, grade, and subject controls update both figures and sample counts. Counts across years are student-year records. The source student IDs are replaced with anonymous keys in this export.

The notebook remains the analysis record. The HTML is the separate presentation aid.

The first four dashboard views are descriptive. The fifth reads the completed Section 4 model exports, including confidence intervals and sample definitions. Its results have a fixed three-year scope; school/year/grade filters do not refit them. Raw percentages and scatterplots are not adjusted effects.

The HTML now includes the focused student-path and overlap views in 3.10, and the adjusted starting-group and absence results in Section 4. The descriptive views preserve incomplete records and distinguish annual from between-test attendance.


See the [dashboard instructions](../dashboard/README.md) for setup and controls. The current briefing and its notes explain the adjusted results.


In [ ]:
dashboard_path = build_dashboard(
    elem_context,
    benchmark_pairs,
    participation,
    project_dir / "Fairmont_LD_Dashboard.html",
)
print(f"Dashboard saved: {dashboard_path.name}")
